# Análise Exploratória dos Dados

Este notebook apresenta uma **Análise Exploratória de Dados (Exploratory Data Analysis — EDA)** sobre o conjunto de dados utilizado para modelar o valor mediano dos imóveis (`MEDV`).

O objetivo desta etapa é compreender a **estrutura, qualidade e distribuição dos dados**, além de investigar as relações entre as variáveis antes da construção de modelos estatísticos ou de aprendizado de máquina. A análise também busca identificar possíveis problemas, como valores faltantes, duplicatas, assimetria, observações discrepantes, relações não lineares e multicolinearidade.

A análise será dividida em quatro partes:

## 1. Qualidade e estrutura dos dados

Inicialmente, será realizada uma inspeção geral do conjunto de dados, verificando:

- dimensões do dataset;
- tipos das variáveis;
- valores faltantes;
- observações duplicadas;
- estatísticas descritivas.

Também será analisada a **natureza de cada variável**. Algumas variáveis possuem características particulares que devem ser consideradas posteriormente na modelagem, como:

- `CHAS`, uma variável binária;
- `RAD`, um índice discreto com poucos valores distintos;
- `ZN`, que apresenta uma grande quantidade de valores iguais a zero.

Além disso, será investigado o comportamento da variável resposta `MEDV`. Essa variável apresenta **censura no valor 50**, com aproximadamente 16 observações exatamente iguais a `50.0`. Essa concentração no limite superior pode influenciar a análise dos resíduos e, portanto, deve ser considerada durante a modelagem.

## 2. Análise univariada

Em seguida, será analisada individualmente a distribuição de cada variável.

Serão utilizados:

- histogramas e estimativas de densidade;
- medidas de assimetria;
- medidas de curtose.

A variável `MEDV` receberá atenção especial. Caso seja observada uma assimetria à direita significativa, será avaliada a transformação


log(MEDV)


como alternativa para obter uma distribuição mais adequada e potencialmente melhorar propriedades dos resíduos, como normalidade e homocedasticidade.

Algumas variáveis preditoras, como `CRIM`, `LSTAT`, `DIS` e `NOX`, também podem apresentar assimetria acentuada. Para essas variáveis, serão avaliadas possíveis transformações, como **logaritmo**, buscando obter distribuições mais adequadas para a modelagem.

## 3. Relação de cada preditor com `MEDV`

Após compreender individualmente cada variável, será investigada a relação entre os preditores e a variável resposta `MEDV`.

Serão utilizados gráficos de dispersão de `MEDV` em função de cada variável preditora, acompanhados de uma curva suavizada **LOWESS**, permitindo verificar se as relações observadas são aproximadamente lineares ou apresentam algum padrão de não linearidade.

Também serão analisados:

- boxplots de `MEDV` para os diferentes valores de `CHAS`;
- boxplots de `MEDV` para possíveis faixas de `RAD`;
- padrões de agrupamento ou concentração de observações.

Espera-se, por exemplo, observar uma relação aproximadamente linear e positiva entre `RM` e `MEDV`, enquanto `LSTAT` pode apresentar um comportamento mais curvo. Esses padrões podem indicar a necessidade de transformações ou da inclusão de termos não lineares, como termos quadráticos.

Também será dada atenção especial às variáveis `TAX` e `RAD`, nas quais podem existir agrupamentos de observações com valores idênticos, capazes de influenciar a interpretação das relações e dos modelos ajustados.

## 4. Multicolinearidade

Por fim, será investigada a dependência entre as variáveis preditoras, com o objetivo de identificar possíveis problemas de **multicolinearidade**.

Serão utilizados:

- matriz de correlação de **Pearson**;
- heatmaps das correlações;
- **Variance Inflation Factor (VIF)**;
- número de condição da matriz de design.

O VIF será calculado para cada variável, dando atenção especial a valores elevados, particularmente aqueles acima de **5 a 10**, que podem indicar problemas relevantes de multicolinearidade.

Os resultados dessa etapa servirão para orientar decisões posteriores sobre a seleção e transformação das variáveis, incluindo a possibilidade de **remover ou combinar variáveis** ou utilizar métodos de regularização, como **Ridge** e **Lasso**.

---

> **Objetivo final da EDA:** compreender as características dos dados e identificar padrões ou problemas que devem ser considerados antes da construção e avaliação dos modelos de regressão.

In [1]:
import os
import pandas as pd
import numpy as np
import math
from statsmodels.tools.tools import add_constant
from scipy.stats import mannwhitneyu, kruskal
from scipy.stats import gaussian_kde
from statsmodels.nonparametric.smoothers_lowess import lowess
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

In [2]:
output_fig_dir = '../results/figures'
os.makedirs(output_fig_dir, exist_ok=True)

In [3]:
df = pd.read_csv("../data/housing.csv")

# CHAS é uma variável indicadora (0/1): tratada como categórica, não contínua
df["CHAS"] = df["CHAS"].astype("category")

# RAD é um índice com 9 níveis (1 a 8 e 24): tratado como categórico ordinal, não contínuo
df["RAD"] = df["RAD"].astype(pd.CategoricalDtype(ordered=True))

In [4]:
# Gráficos interativos (Plotly): passe o mouse sobre um ponto para ver a linha do arquivo, a cidade
# e os valores da observação; clique na legenda para esconder ou mostrar uma série.
COR_PONTO = "#2a78d6"
COR_CURVA = "#eb6834"
COR_DESTAQUE = "#4a3aa7"
COR_REFERENCIA = "#898781"
COR_ALERTA = "#fab219"
COR_CRITICO = "#d03b3b"
COR_ENVELOPE = "rgba(137, 135, 129, 0.25)"
ESCALA_DIVERGENTE = [[0, "#184f95"], [0.25, "#6da7ec"], [0.5, "#f0efec"], [0.75, "#ec835a"], [1, "#a32a2a"]]

pio.templates["boston"] = go.layout.Template(layout=dict(
    colorway=["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"],
    font=dict(family="system-ui, -apple-system, Segoe UI, sans-serif", color="#0b0b0b"),
    paper_bgcolor="#fcfcfb",
    plot_bgcolor="#fcfcfb",
    hovermode="closest",
    hoverlabel=dict(bgcolor="#ffffff", font_size=12),
))
pio.templates.default = "plotly_white+boston"

# Cidade de cada linha (versão corrigida do Boston Housing, mesma ordem de linhas de housing.csv)
cidades = pd.read_csv("../data/boston_corrected.csv", usecols=["TOWN"])["TOWN"]
CONTEXTO_HOVER = {"MEDV": "{:.1f}", "CRIM": "{:.3f}", "RM": "{:.2f}", "LSTAT": "{:.2f}", "RAD": "{}"}


def texto_hover(indices, extras=None):
    # Um texto por observação: linha, cidade, os valores do gráfico (extras) e as principais variáveis
    indices = list(indices)
    extras = {nome: np.asarray(valores) for nome, valores in ({} if extras is None else extras).items()}
    contexto = df.loc[indices, list(CONTEXTO_HOVER)]
    textos = []
    for k, indice in enumerate(indices):
        partes = [f"<b>Linha {indice}</b> ({cidades.get(indice, 'sem cidade')})"]
        partes += [f"{nome}: {valores[k]:.3f}" for nome, valores in extras.items()]
        partes += [f"{coluna}: {formato.format(contexto.iloc[k][coluna])}" for coluna, formato in CONTEXTO_HOVER.items()]
        textos.append("<br>".join(partes))
    return textos


def pontos(x, y, indices, extras=None, nome="Observações", cor=COR_PONTO, tamanho=7, **kwargs):
    return go.Scatter(
        x=np.asarray(x), y=np.asarray(y), mode="markers", name=nome,
        marker=dict(color=cor, size=tamanho, opacity=0.65, line=dict(width=0.5, color="#fcfcfb")),
        hovertext=texto_hover(indices, extras), hovertemplate="%{hovertext}<extra></extra>", **kwargs,
    )


def curva_lowess(x, y, nome="LOWESS"):
    # Mesmo ajuste do sns.regplot(lowess=True): statsmodels lowess com frac = 2/3
    suave = lowess(np.asarray(y, dtype=float), np.asarray(x, dtype=float), frac=2 / 3)
    return go.Scatter(x=suave[:, 0], y=suave[:, 1], mode="lines", name=nome, line=dict(color=COR_CURVA, width=2.5),
                      hovertemplate=f"{nome}<br>x = %{{x:.3f}}<br>y = %{{y:.3f}}<extra></extra>")


def linha_ref(fig, valor, inicio, fim, nome, cor=COR_REFERENCIA, vertical=False, traco="dash", row=None, col=None):
    # Linha de referência como série (aparece na legenda e mostra o valor ao passar o mouse)
    ao_longo = np.linspace(inicio, fim, 60)
    constante = np.full_like(ao_longo, valor)
    x, y = (constante, ao_longo) if vertical else (ao_longo, constante)
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=nome, line=dict(color=cor, width=1.5, dash=traco),
                             hovertemplate=f"{nome}: {valor:.4g}<extra></extra>"), row=row, col=col)


def faixa(valores, margem=0.03):
    valores = np.asarray(valores, dtype=float)
    minimo, maximo = np.nanmin(valores), np.nanmax(valores)
    folga = (maximo - minimo) * margem
    return minimo - folga, maximo + folga


def envelope_qq(n_m, rng):
    # Quantis teóricos e envelope 95% de 100 amostras N(0, 1) ordenadas
    q = stats.norm.ppf((np.arange(1, n_m + 1) - 0.375) / (n_m + 0.25))
    sim = np.sort(rng.standard_normal((100, n_m)), axis=1)
    return q, np.percentile(sim, 2.5, axis=0), np.percentile(sim, 97.5, axis=0)


def tracos_qq(residuos, indices, q, inf, sup, nome_residuo="Res. studentizado"):
    # Gráfico QQ: envelope, pontos dentro e fora dele e reta y = x. Devolve as séries e o nº fora do envelope.
    ordem = np.argsort(np.asarray(residuos), kind="stable")
    rs = np.asarray(residuos)[ordem]
    idx = np.asarray(indices)[ordem]
    fora = (rs < inf) | (rs > sup)
    tracos = [
        go.Scatter(x=q, y=sup, mode="lines", name="Envelope 95%", line=dict(width=0, color=COR_REFERENCIA), showlegend=False,
                   hovertemplate="Envelope 95% (limite superior): %{y:.3f}<extra></extra>"),
        go.Scatter(x=q, y=inf, mode="lines", name="Envelope 95%", line=dict(width=0, color=COR_REFERENCIA),
                   fill="tonexty", fillcolor=COR_ENVELOPE,
                   hovertemplate="Envelope 95% (limite inferior): %{y:.3f}<extra></extra>"),
        go.Scatter(x=q, y=q, mode="lines", name="Reta y = x", line=dict(color=COR_CRITICO, width=1.5),
                   hovertemplate="Reta y = x: %{y:.3f}<extra></extra>"),
    ]
    for nome, mascara, cor in [("Dentro do envelope", ~fora, COR_PONTO), ("Fora do envelope", fora, COR_CRITICO)]:
        if mascara.any():
            tracos.append(pontos(q[mascara], rs[mascara], idx[mascara], nome=nome, cor=cor, tamanho=6,
                                 extras={"Quantil teórico": q[mascara], nome_residuo: rs[mascara]}))
    return tracos, int(fora.sum())


def legenda_unica(fig):
    # Séries de mesmo nome em painéis diferentes viram um grupo: um item na legenda liga e desliga todas
    vistos = set()
    for traco in fig.data:
        if traco.name is None:
            continue
        traco.legendgroup = traco.name
        traco.showlegend = traco.name not in vistos and traco.showlegend is not False
        if traco.showlegend:
            vistos.add(traco.name)


def mostrar_e_salvar(fig, nome=None, titulo=None, altura=500, largura=None):
    legenda_unica(fig)
    fig.update_layout(title=titulo, height=altura, width=largura, margin=dict(t=90 if titulo else 60))
    if nome:
        fig.write_html(f"{output_fig_dir}/{nome}.html", include_plotlyjs="cdn")
    fig.show()


def gerar_subplot_hist(variaveis: list[str], size: tuple[int, int], name: str):

    n_cols = 4
    n_rows = math.ceil(len(variaveis)/n_cols)

    titulos = [
        f"{v}<br><sup>Assimetria: {df[v].skew():.2f} | Curtose: {df[v].kurt():.2f}</sup>"
        for v in variaveis
    ]
    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=titulos, vertical_spacing=0.12)

    for k, variavel in enumerate(variaveis):
        linha, coluna = k // n_cols + 1, k % n_cols + 1
        valores = df[variavel].dropna().to_numpy()
        # Mesmas classes do sns.histplot (regra "auto" do numpy) e a densidade (KDE) na escala da frequência
        bordas = np.histogram_bin_edges(valores, bins="auto")
        largura_classe = bordas[1] - bordas[0]
        fig.add_trace(go.Histogram(
            x=valores, name="Frequência", marker=dict(color=COR_PONTO, line=dict(width=1, color="#fcfcfb")),
            xbins=dict(start=bordas[0], end=bordas[-1], size=largura_classe),
            hovertemplate=f"{variavel}: %{{x}}<br>Frequência: %{{y}}<extra></extra>",
        ), row=linha, col=coluna)
        grade = np.linspace(valores.min(), valores.max(), 200)
        fig.add_trace(go.Scatter(
            x=grade, y=gaussian_kde(valores)(grade) * len(valores) * largura_classe, mode="lines", name="KDE",
            line=dict(color=COR_CURVA, width=2),
            hovertemplate=f"KDE<br>{variavel} = %{{x:.3f}}<br>frequência esperada = %{{y:.1f}}<extra></extra>",
        ), row=linha, col=coluna)

    fig.update_yaxes(title_text="Frequência", col=1)
    mostrar_e_salvar(fig, name, altura=size[1] * 75)

def gerar_subplot_regplot(variaveis: list[str], y: str, size: tuple[int, int], name: str):

    n_cols = 4
    n_rows = math.ceil(len(variaveis)/n_cols)

    fig = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=[f"{y} x {v}" for v in variaveis],
                        vertical_spacing=0.08)

    for k, variavel in enumerate(variaveis):
        linha, coluna = k // n_cols + 1, k % n_cols + 1
        dados = df[[variavel, y]].dropna()
        fig.add_trace(pontos(dados[variavel], dados[y], dados.index, tamanho=5,
                             extras={variavel: dados[variavel], y: dados[y]}), row=linha, col=coluna)
        fig.add_trace(curva_lowess(dados[variavel], dados[y]), row=linha, col=coluna)

    mostrar_e_salvar(fig, name, altura=size[1] * 85)

def grafico_categoria(variavel: str, name: str):
    # Boxplot de MEDV por nível (com todos os pontos) e contagem de observações por nível
    niveis = [str(c) for c in df[variavel].cat.categories]
    fig = make_subplots(rows=1, cols=2, subplot_titles=[f"MEDV por {variavel}", f"Frequência de {variavel}"])
    fig.add_trace(go.Box(
        x=df[variavel].astype(str), y=df["MEDV"], name="MEDV", boxpoints="all", jitter=0.4, pointpos=0,
        marker=dict(color=COR_PONTO, size=4, opacity=0.5), line=dict(color=COR_PONTO), fillcolor="rgba(42, 120, 214, 0.15)",
        hovertext=texto_hover(df.index), hoveron="boxes+points",
    ), row=1, col=1)
    contagem = df[variavel].astype(str).value_counts().reindex(niveis)
    fig.add_trace(go.Bar(x=niveis, y=contagem.values, name="Frequência", marker=dict(color=COR_PONTO),
                         hovertemplate=f"{variavel} = %{{x}}<br>Frequência: %{{y}}<extra></extra>"), row=1, col=2)
    fig.update_xaxes(title_text=variavel, type="category", categoryorder="array", categoryarray=niveis)
    fig.update_yaxes(title_text="MEDV", row=1, col=1)
    fig.update_yaxes(title_text="Frequência", row=1, col=2)
    fig.update_layout(showlegend=False)
    mostrar_e_salvar(fig, name, altura=450)

def heatmap_correlacao(matriz: pd.DataFrame, titulo: str, name: str):
    fig = go.Figure(go.Heatmap(
        z=matriz.values, x=list(matriz.columns), y=list(matriz.index), zmin=-1, zmax=1, zmid=0,
        colorscale=ESCALA_DIVERGENTE, text=matriz.round(2).values, texttemplate="%{text:.2f}",
        xgap=1, ygap=1, colorbar=dict(title="ρ"),
        hovertemplate="%{y} x %{x}<br>correlação = %{z:.3f}<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed", scaleanchor="x")
    mostrar_e_salvar(fig, name, titulo=titulo, altura=750, largura=850)

def medidas_diagnostico(m):
    # Valores mostrados ao passar o mouse sobre cada ponto dos gráficos de diagnóstico
    infl = m.get_influence()
    return {"Ajustado": m.fittedvalues, "Resíduo": m.resid, "Res. studentizado": infl.resid_studentized_external,
            "Alavanca": infl.hat_matrix_diag, "Cook": infl.cooks_distance[0]}

def graficos_diagnostico(m, dados, titulo, nome_arquivo):
    # Gráfico QQ com envelope, resíduos x ajustados, escala-locação e distância de Cook de um ajuste MQO
    infl = m.get_influence()
    n_m = int(m.nobs)
    indices = m.fittedvalues.index
    medidas = medidas_diagnostico(m)
    q, inf, sup = envelope_qq(n_m, np.random.default_rng(42))
    tracos, fora = tracos_qq(infl.resid_studentized_external, indices, q, inf, sup)
    rotulo_x = f"Valor ajustado ({m.model.endog_names})"
    raiz = np.sqrt(np.abs(infl.resid_studentized_internal))
    cook = infl.cooks_distance[0]

    fig = make_subplots(rows=2, cols=2, vertical_spacing=0.12, subplot_titles=[
        f"Gráfico QQ | fora do envelope: {fora} de {n_m}", "Resíduos x valores ajustados",
        "Escala-locação", "Distância de Cook"])
    for traco in tracos:
        fig.add_trace(traco, row=1, col=1)

    fig.add_trace(pontos(m.fittedvalues, m.resid, indices, medidas), row=1, col=2)
    fig.add_trace(curva_lowess(m.fittedvalues, m.resid), row=1, col=2)
    linha_ref(fig, 0, *faixa(m.fittedvalues), "Resíduo zero", cor="#0b0b0b", traco="solid", row=1, col=2)

    fig.add_trace(pontos(m.fittedvalues, raiz, indices, {**medidas, "√|res. padronizado|": raiz}), row=2, col=1)
    fig.add_trace(curva_lowess(m.fittedvalues, raiz), row=2, col=1)

    acima = cook > 4 / n_m
    fig.add_trace(go.Bar(x=np.asarray(indices), y=cook, name="Distância de Cook",
                         marker=dict(color=np.where(acima, COR_CRITICO, COR_PONTO)),
                         hovertext=texto_hover(indices, medidas), hovertemplate="%{hovertext}<extra></extra>"),
                  row=2, col=2)
    linha_ref(fig, 4 / n_m, *faixa(indices), "Cook = 4/n", cor=COR_ALERTA, row=2, col=2)

    fig.update_xaxes(title_text="Quantil teórico N(0, 1)", row=1, col=1)
    fig.update_yaxes(title_text="Resíduo studentizado", row=1, col=1)
    fig.update_xaxes(title_text=rotulo_x, row=1, col=2)
    fig.update_yaxes(title_text="Resíduo", row=1, col=2)
    fig.update_xaxes(title_text=rotulo_x, row=2, col=1)
    fig.update_yaxes(title_text="√|resíduo padronizado|", row=2, col=1)
    fig.update_xaxes(title_text="Índice da observação", row=2, col=2)
    fig.update_yaxes(title_text="Distância de Cook", row=2, col=2)
    mostrar_e_salvar(fig, nome_arquivo, titulo=titulo, altura=900)

def calcular_vif(df: pd.DataFrame, colunas: list[str]):
    # VIF generalizado (GVIF, Fox & Monette, 1992): uma variável categórica com k níveis
    # entra como k-1 indicadoras, e o GVIF mede a colinearidade do bloco inteiro.
    # Para variáveis numéricas e binárias (GL = 1), o GVIF é igual ao VIF.
    X = df[colunas].dropna().copy()

    blocos = {}
    partes = []
    for coluna in colunas:
        if isinstance(X[coluna].dtype, pd.CategoricalDtype):
            parte = pd.get_dummies(X[coluna], prefix=coluna, drop_first=True, dtype=float)
        else:
            parte = X[[coluna]].astype(float)
        blocos[coluna] = list(parte.columns)
        partes.append(parte)

    X_design = pd.concat(partes, axis=1)
    R = X_design.corr()
    det_R = np.linalg.det(R)

    linhas = []
    for coluna, colunas_bloco in blocos.items():
        outras = [c for c in X_design.columns if c not in colunas_bloco]
        gvif = (
            np.linalg.det(R.loc[colunas_bloco, colunas_bloco])
            * np.linalg.det(R.loc[outras, outras])
            / det_R
        )
        gl = len(colunas_bloco)
        linhas.append({
            "Variável": coluna,
            "GL": gl,
            "GVIF": gvif,
            # Comparável a sqrt(VIF): 2,24 equivale a VIF 5 e 3,16 a VIF 10
            "GVIF^(1/(2·GL))": gvif ** (1 / (2 * gl)),
        })

    return (
        pd.DataFrame(linhas)
        .sort_values(by="GVIF^(1/(2·GL))", ascending=False)
        .reset_index(drop=True)
    )

def calcular_numero_condicao(df: pd.DataFrame, colunas: list[str]):
    X = df[colunas].dropna().copy()
    X = pd.get_dummies(X, drop_first=True, dtype=float)
    # Padroniza para que o número de condição não dependa das unidades
    X = (X - X.mean()) / X.std()
    X_com_const = add_constant(X)
    return np.linalg.cond(X_com_const.values)

## **Parte 1: Qualidade e estrutura dos dados**

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 511 entries, 0 to 510
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   CRIM     511 non-null    float64 
 1   ZN       511 non-null    float64 
 2   INDUS    511 non-null    float64 
 3   CHAS     511 non-null    category
 4   NOX      511 non-null    float64 
 5   RM       506 non-null    float64 
 6   AGE      511 non-null    float64 
 7   DIS      511 non-null    float64 
 8   RAD      511 non-null    category
 9   TAX      511 non-null    int64   
 10  PTRATIO  511 non-null    float64 
 11  B        511 non-null    float64 
 12  LSTAT    511 non-null    float64 
 13  MEDV     511 non-null    float64 
dtypes: category(2), float64(11), int64(1)
memory usage: 49.1 KB


In [6]:
df.describe()

,CRIM,ZN,INDUS,NOX,RM,AGE,DIS,TAX,PTRATIO,B,LSTAT,MEDV
count,511.000000,511.000000,511.000000,511.000000,506.000000,511.000000,511.000000,511.000000,511.000000,511.000000,511.000000,511.000000
mean,3.584139,11.252446,11.151096,0.554757,6.287589,68.616243,3.783876,407.440313,18.500000,356.600900,12.879550,22.682192
std,8.564433,23.234838,6.828175,0.115310,0.703802,28.099130,2.098631,167.903532,2.200348,90.882679,7.797416,9.484262
min,0.006320,0.000000,0.460000,0.385000,3.561000,2.900000,1.129600,187.000000,12.600000,0.320000,1.730000,5.000000
25%,0.082325,0.000000,5.190000,0.449000,5.885500,45.050000,2.100350,279.500000,17.400000,374.710000,7.065000,17.050000
50%,0.261690,0.000000,9.690000,0.538000,6.209000,77.300000,3.152300,330.000000,19.100000,391.340000,11.450000,21.200000
75%,3.621175,12.500000,18.100000,0.624000,6.629750,94.050000,5.118000,666.000000,20.200000,396.210000,17.105000,25.000000
max,88.976200,100.000000,27.740000,0.871000,8.780000,100.000000,12.126500,711.000000,23.000000,396.900000,76.000000,67.000000


In [7]:
pd.DataFrame({
    'Frequência': df['CHAS'].value_counts().sort_index(),
    'Proporção': df['CHAS'].value_counts(normalize=True).sort_index().round(3)
})

,Frequência,Proporção
CHAS,,
0,476,0.932
1,35,0.068


In [8]:
pd.DataFrame({
    'Frequência': df['RAD'].value_counts().sort_index(),
    'Proporção': df['RAD'].value_counts(normalize=True).sort_index().round(3)
})

,Frequência,Proporção
RAD,,
1,20,0.039
2,24,0.047
3,43,0.084
4,110,0.215
5,115,0.225
6,26,0.051
7,17,0.033
8,24,0.047
24,132,0.258


In [9]:
for i in df.columns:
    print(f"Valores únicos de {i}: {df[i].nunique()}")

Valores únicos de CRIM: 509
Valores únicos de ZN: 26
Valores únicos de INDUS: 79
Valores únicos de CHAS: 2
Valores únicos de NOX: 82
Valores únicos de RM: 444
Valores únicos de AGE: 357
Valores únicos de DIS: 416
Valores únicos de RAD: 9
Valores únicos de TAX: 67
Valores únicos de PTRATIO: 47
Valores únicos de B: 360
Valores únicos de LSTAT: 460
Valores únicos de MEDV: 231


In [10]:
print((df["MEDV"] == 50.0).sum())
df["MEDV"].value_counts().sort_index(ascending=False).head(10)

16


MEDV
67.0     1
54.0     1
50.0    16
48.8     1
48.5     1
48.3     1
46.7     1
46.0     1
45.4     1
44.8     1
Name: count, dtype: int64

In [11]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")

Linhas duplicadas: 0


Observações com `RM` ausente:

In [12]:
df[df["RM"].isna()]

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
10,0.22489,12.5,7.87,0,0.524,NaN,94.3,6.3467,5,311,15.2,392.52,20.45,15.0
35,0.06417,0.0,5.96,0,0.499,NaN,68.2,3.3603,5,279,19.2,396.90,9.68,18.9
63,0.12650,25.0,5.13,0,0.453,NaN,43.4,7.9809,8,284,19.7,395.58,9.50,25.0
96,0.11504,0.0,2.89,0,0.445,NaN,69.6,3.4952,2,276,18.0,391.83,11.34,21.4
135,0.55778,0.0,21.89,0,0.624,NaN,98.2,2.1107,4,437,21.2,394.67,16.96,18.1


Verificação de consistência com o domínio das variáveis. No Boston Housing original, `MEDV` é limitado a 50 e `LSTAT` (percentual) não passa de ~38. Valores fora disso indicam observações suspeitas:

In [13]:
suspeitas = df[(df["MEDV"] > 50) | (df["LSTAT"] > 40)]
suspeitas

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
508,0.44433,0.0,12.5,0,0.561,6.123,98.0,2.987,3,320,23.0,343.0,21.0,54.0
509,0.77763,0.0,12.7,0,0.561,6.222,34.0,2.543,3,329,23.0,343.0,76.0,67.0
510,0.65432,0.0,12.8,0,0.561,6.760,67.0,2.987,3,345,23.0,321.0,45.0,24.0


O dataset original possui 506 observações (índices 0 a 505). As 5 linhas extras deste arquivo (índices 506 a 510) são exibidas abaixo:

In [14]:
df.loc[506:]

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
506,0.98765,0.0,12.5,0,0.561,6.980,89.0,2.098,3,320,23.0,396.0,12.0,12.0
507,0.23456,0.0,12.5,0,0.561,6.980,76.0,2.654,3,320,23.0,343.0,25.0,32.0
508,0.44433,0.0,12.5,0,0.561,6.123,98.0,2.987,3,320,23.0,343.0,21.0,54.0
509,0.77763,0.0,12.7,0,0.561,6.222,34.0,2.543,3,329,23.0,343.0,76.0,67.0
510,0.65432,0.0,12.8,0,0.561,6.760,67.0,2.987,3,345,23.0,321.0,45.0,24.0


Contagem de valores fora das cercas de Tukey (Q1 − 1,5·IQR, Q3 + 1,5·IQR) para cada variável numérica:

In [15]:
numericas = df.select_dtypes(include="number").columns
q1 = df[numericas].quantile(0.25)
q3 = df[numericas].quantile(0.75)
iqr = q3 - q1
fora = (df[numericas] < q1 - 1.5 * iqr) | (df[numericas] > q3 + 1.5 * iqr)

pd.DataFrame({
    "Outliers (IQR)": fora.sum(),
    "Proporção": fora.mean().round(3)
}).sort_values("Outliers (IQR)", ascending=False)

,Outliers (IQR),Proporção
B,76,0.149
ZN,68,0.133
CRIM,67,0.131
MEDV,42,0.082
RM,30,0.059
PTRATIO,15,0.029
LSTAT,8,0.016
DIS,5,0.010
NOX,0,0.000
INDUS,0,0.000


In [16]:

fig = go.Figure(go.Histogram(
    x=df["MEDV"], marker=dict(color=COR_PONTO, line=dict(width=0.5, color="#fcfcfb")),
    xbins=dict(start=df["MEDV"].min(), end=df["MEDV"].max(), size=(df["MEDV"].max() - df["MEDV"].min()) / 100),
    hovertemplate="MEDV: %{x}<br>Frequência: %{y}<extra></extra>",
))
fig.update_xaxes(title_text="MEDV")
fig.update_yaxes(title_text="Frequência")
mostrar_e_salvar(fig, altura=400)

### Conclusão - Qualidade e estrutura dos dados

A análise inicial mostrou que o conjunto de dados possui **511 observações e 14 variáveis**, sendo 11 variáveis do tipo `float64`, 1 do tipo `int64` (`TAX`) e 2 convertidas para o tipo `category` (`CHAS` e `RAD`). De modo geral, os dados apresentam uma estrutura adequada para a análise, sem problemas generalizados de valores ausentes.

Entretanto, foi identificado um problema específico na variável `RM`, que possui **5 valores ausentes**, enquanto as demais variáveis apresentam 511 observações não nulas. Essa ausência deverá ser tratada antes da etapa de modelagem, avaliando-se posteriormente uma estratégia adequada, como imputação ou remoção das observações.

A análise do número de valores únicos também permitiu identificar diferentes naturezas entre as variáveis. `CHAS` possui apenas **2 valores distintos**, caracterizando uma variável binária. Por isso, ela é tratada como **categórica**: sua média e desvio padrão não têm interpretação de variável contínua, e a descrição adequada é a frequência de cada categoria — 476 observações (93,2%) com `CHAS = 0` e apenas 35 (6,8%) com `CHAS = 1`. `RAD` apresenta apenas **9 valores distintos** (1 a 8 e 24). Por ser um índice de acessibilidade, e não uma medida, seus valores funcionam como rótulos ordenados: o salto de 8 para 24 não representa uma acessibilidade três vezes maior. Por isso, `RAD` é tratada como **categórica ordinal**. Suas categorias são desbalanceadas: `RAD = 24` concentra 132 observações (25,8%), seguida de `RAD = 5` (115; 22,5%) e `RAD = 4` (110; 21,5%), enquanto `RAD = 7` tem apenas 17 (3,3%). `ZN`, por sua vez, possui apenas **26 valores distintos**, indicando uma concentração significativa de observações em determinados valores.

As demais variáveis apresentam maior quantidade de valores distintos, sendo tratadas predominantemente como variáveis contínuas. Destacam-se `CRIM`, com 509 valores únicos, `LSTAT`, com 460, e `RM`, com 444 valores únicos entre as 506 observações disponíveis.

A variável resposta `MEDV` apresentou uma característica particularmente importante: existem **16 observações exatamente iguais a 50,0**, enquanto os valores próximos apresentam uma quantidade consideravelmente menor de observações. Essa concentração no limite superior indica a presença de **censura no valor 50**, característica que pode afetar a distribuição da variável resposta e, consequentemente, os resíduos e os pressupostos dos modelos de regressão. Entretanto, existem também dois valores **acima** do limite de censura (`MEDV` = 54 e 67), o que é incompatível com a censura em 50 e motivou a verificação de consistência abaixo.

Não foram encontradas observações duplicadas. A verificação de consistência com o domínio das variáveis, porém, revelou que as **5 últimas linhas do arquivo (índices 506 a 510)** são suspeitas. O Boston Housing original possui exatamente 506 observações, e essas linhas extras apresentam:

- valores de `CRIM` com padrão sequencial de dígitos (0,98765; 0,23456; 0,44433; 0,77763; 0,65432);
- `AGE`, `B`, `LSTAT` e `MEDV` todos inteiros, diferentemente do restante do conjunto;
- `MEDV` = 54 e 67, acima do limite de censura em 50;
- `LSTAT` = 76 e 45, muito acima do máximo de ~38 do conjunto original (e, no caso de 76, implausível para um percentual dessa variável).

Essas observações **foram mantidas** nesta etapa, mas ficam sinalizadas: elas afetam as medidas de assimetria e curtose, as correlações, o VIF e o teste de Mann-Whitney, e sua remoção deve ser avaliada antes do ajuste do modelo. Da mesma forma, as 5 observações com `RM` ausente (índices 10, 35, 63, 96 e 135) também foram mantidas; até que sejam tratadas, o cálculo do VIF as descarta (`dropna`) e as matrizes de correlação usam exclusão par a par, de modo que os dois cálculos não usam exatamente as mesmas linhas.

A contagem de valores fora das cercas de Tukey (1,5·IQR) mostrou maior concentração de valores extremos em `B` (76 observações), `ZN` (68), `CRIM` (67) e `MEDV` (42). Em `ZN` e `B`, isso reflete principalmente a forte concentração da distribuição (zeros em `ZN` e valores próximos de 396,9 em `B`), e não necessariamente erros; em `CRIM` e `MEDV`, reflete a assimetria à direita.

Portanto, embora o conjunto de dados apresente uma estrutura geral adequada, a análise revelou quatro pontos que deverão receber atenção nas próximas etapas: **os valores ausentes em `RM`, as 5 observações suspeitas (índices 506 a 510), a natureza discreta ou concentrada de algumas variáveis e a censura de `MEDV` em 50**. Esses aspectos serão considerados durante a análise univariada e na investigação das relações entre as variáveis.

## **Parte 2: Análise univariada**

In [17]:
variaveis_hist = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX',
       'PTRATIO', 'B', 'LSTAT', 'MEDV']

gerar_subplot_hist(
    variaveis=variaveis_hist,
    size=(15, 10),
    name='hist_variaveis_sem_tranformacoes'
)

In [18]:
df["logCRIM"] = np.log1p(df["CRIM"])
df["logZN"] = np.log1p(df["ZN"])
df["logDIS"] = np.log(df["DIS"])
df["logLSTAT"] = np.log(df["LSTAT"])
df["logMEDV"] = np.log(df["MEDV"])

In [19]:
novas_variaveis_hist = ["logCRIM", "logZN", "logDIS", "logLSTAT", "logMEDV"]

gerar_subplot_hist(
    variaveis=novas_variaveis_hist,
    size=(15, 8),
    name='hist_variaveis_transformadas'
)

### Conclusão — Análise Univariada

A análise univariada, apoiada pela inspeção dos histogramas, das distribuições e das medidas de assimetria, revelou que algumas variáveis apresentam desvios acentuados em relação à simetria, tornando-se candidatas à aplicação de transformações antes da modelagem.

A variável `CRIM` apresentou **assimetria positiva acentuada**, com valor de 5,25. Após a aplicação da transformação logarítmica, a assimetria de `logCRIM` foi reduzida para 1,28, indicando uma melhora expressiva na simetria da distribuição. Em contrapartida, `B` apresentou assimetria negativa acentuada, de -2,90, evidenciando uma concentração das observações em uma região da distribuição e uma cauda prolongada à esquerda.

A variável resposta `MEDV` também apresentou assimetria positiva, com valor de 1,23. Após a transformação logarítmica, a assimetria de `logMEDV` passou para -0,28, indicando uma distribuição consideravelmente mais simétrica. Esse resultado reforça a possibilidade de utilizar **`log(MEDV)` como variável resposta** nos modelos posteriores, embora a escolha definitiva também deva considerar os diagnósticos dos resíduos e a censura observada no valor 50.

Outro resultado relevante foi observado em `LSTAT`, cuja assimetria passou de 1,81 para -0,25 após a transformação logarítmica. Essa redução indica que o logaritmo também é uma alternativa promissora para essa variável preditora.

Por outro lado, `ZN` manteve um comportamento bimodal mesmo após a transformação logarítmica. Isso ocorre porque a variável apresenta uma grande concentração de valores iguais a zero, característica estrutural que não é resolvida simplesmente pela transformação. Portanto, seu tratamento deverá considerar a distribuição dos valores e a possibilidade de distinguir as observações iguais a zero daquelas com valores positivos.

Em síntese, os resultados indicam que **`CRIM`, `MEDV` e `LSTAT` são candidatos relevantes à transformação logarítmica**, enquanto `B` merece uma investigação mais aprofundada devido à sua assimetria negativa. Já `ZN` exige uma abordagem que considere sua concentração de zeros e sua distribuição bimodal. Essas transformações deverão ser avaliadas também em relação às relações entre as variáveis e ao desempenho dos modelos, pois uma distribuição mais simétrica, isoladamente, não garante o atendimento aos pressupostos da regressão.


## **Parte 3: Relação de cada preditor com MEDV**

In [20]:
variaveis_regplot = ['CRIM', 'ZN', 'INDUS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX',
       'PTRATIO', 'B', 'LSTAT', "logCRIM", "logZN", "logDIS", "logLSTAT"]

gerar_subplot_regplot(
    variaveis=variaveis_regplot,
    y='MEDV',
    size=(15, 10),
    name='MEDV_vs_variaveis'
)

/home/ribamarneto/Documentos/Trabalho Ronald/.venv/lib/python3.12/site-packages/statsmodels/nonparametric/smoothers_lowess.py:237: RuntimeWarning: invalid value encountered in divide
  res, _ = _lowess(


In [21]:
gerar_subplot_regplot(
    variaveis=variaveis_regplot,
    y='logMEDV',
    size=(15, 10),
    name='logMEDV_vs_variaveis'
)

In [22]:
grafico_categoria('CHAS', 'MEDV_vs_CHAS')

In [23]:
resumo_chas = df.groupby('CHAS', observed=True)['MEDV'].agg(['count', 'median', 'mean', 'std'])
print(resumo_chas)

medv_chas_0 = df.loc[df['CHAS'] == 0, 'MEDV']
medv_chas_1 = df.loc[df['CHAS'] == 1, 'MEDV']

estatistica, p_valor = mannwhitneyu(medv_chas_1, medv_chas_0, alternative='two-sided')
print(f"\nMann-Whitney U: U = {estatistica:.1f} | p-valor = {p_valor:.4f}")

      count  median       mean        std
CHAS                                     
0       476   20.95  22.258824   9.163078
1        35   23.30  28.440000  11.816643

Mann-Whitney U: U = 10942.5 | p-valor = 0.0019


In [24]:
grafico_categoria('RAD', 'MEDV_vs_RAD')

In [25]:
resumo_rad = df.groupby('RAD', observed=True)['MEDV'].agg(['count', 'median', 'mean', 'std'])
print(resumo_rad)

grupos_rad = [grupo['MEDV'] for _, grupo in df.groupby('RAD', observed=True)]

estatistica, p_valor = kruskal(*grupos_rad)
print(f"\nKruskal-Wallis: H = {estatistica:.2f} | p-valor = {p_valor:.4g}")

     count  median       mean        std
RAD                                     
1       20   22.20  24.365000   8.024454
2       24   23.85  26.833333   7.874376
3       43   26.60  29.076744  10.910302
4      110   20.45  21.387273   6.957883
5      115   23.00  25.706957   9.328401
6       26   21.20  20.976923   2.312801
7       17   26.20  27.105882   6.493215
8       24   28.25  30.358333   9.727724
24     132   14.40  16.403788   8.539745

Kruskal-Wallis: H = 157.18 | p-valor = 6.214e-30


### Conclusão — Relação dos Preditores com `MEDV`

A análise dos gráficos de dispersão, acompanhados das curvas de suavização LOWESS, revelou que as relações entre os preditores e a variável resposta `MEDV` apresentam diferentes graus de linearidade. Esse resultado é importante para a construção dos modelos de regressão, pois indica quais variáveis podem exigir transformações ou a inclusão de termos não lineares.

As variáveis `CRIM` e `LSTAT` apresentaram relações claramente não lineares com `MEDV`. Em ambos os casos, as transformações logarítmicas (`logCRIM` e `logLSTAT`) tornaram as relações observadas mais próximas de uma reta, indicando que essas transformações são alternativas promissoras para a modelagem.

A variável `DIS` apresentou uma relação com formato de joelho, caracterizada por um crescimento inicial mais acentuado seguido de uma tendência à estabilização. Embora a transformação `logDIS` tenha suavizado esse comportamento, ainda foi observada curvatura residual. Portanto, vale avaliar a inclusão de um termo quadrático como alternativa à transformação logarítmica.

Para `ZN`, a transformação `logZN` ajudou a evidenciar um padrão mais monotônico. Entretanto, a elevada concentração de valores iguais a zero, identificada na análise univariada, dificulta seu tratamento como uma variável contínua convencional. Uma alternativa a ser investigada é representá-la por meio de uma variável indicadora que diferencie observações com `ZN = 0` daquelas com `ZN > 0`, preservando a possibilidade de avaliar separadamente o efeito da presença de valores positivos.

Em contrapartida, `RM` apresentou uma relação aproximadamente linear e positiva com `MEDV` na escala original, não havendo, a princípio, uma justificativa clara para transformá-la. Essa relação poderá ser mantida na forma original e posteriormente avaliada pelos diagnósticos do modelo.

A variável `B` merece cautela devido à sua distribuição extremamente concentrada em torno de 400, acompanhada de uma cauda longa em direção a zero. Seu padrão de associação com `MEDV` deve ser interpretado considerando essa distribuição e a possibilidade de agrupamentos ou particularidades dos dados, sem presumir uma relação causal. Assim, sua contribuição para a modelagem deverá ser avaliada com atenção.

#### Análise de `CHAS` e `RAD` como fatores de agrupamento

Como `CHAS` é binária, sua relação com `MEDV` foi analisada pela comparação entre os dois grupos, e não por gráfico de dispersão ou correlação. O boxplot e o resumo por grupo mostraram uma mediana mais elevada para `CHAS = 1` (23,30, média 28,44) em comparação com `CHAS = 0` (20,95, média 22,26). O teste de Mann-Whitney, adequado devido à assimetria e à censura de `MEDV`, indicou diferença significativa entre as distribuições (U = 10942,5; p-valor = 0,0019). Esse resultado sugere uma associação entre a proximidade do rio Charles e o valor mediano dos imóveis, embora o grupo `CHAS = 1` seja pequeno (35 observações), o que exige cautela na interpretação. Portanto, é pertinente manter `CHAS` como candidata a preditora, verificando sua contribuição nos modelos ajustados.

Pelo mesmo motivo, `RAD` foi analisada pela comparação entre seus níveis, e não por gráfico de dispersão. A mediana de `MEDV` não acompanha a ordem do índice: ela varia entre 20,45 (`RAD = 4`) e 28,25 (`RAD = 8`) nos níveis de 1 a 8 e cai para **14,40** em `RAD = 24`, que é também o nível com mais observações (132). O teste de Kruskal-Wallis, extensão do Mann-Whitney para mais de dois grupos, indicou diferença significativa entre os níveis (H = 157,18; p-valor < 0,001). Como a relação não é monotônica, uma única inclinação para `RAD` numérica não representaria esse padrão. Além disso, a estrutura dessa variável deverá ser considerada em conjunto com sua elevada associação com `TAX`, investigada na análise de multicolinearidade.

Diante disso, `RAD` passa a ser representada como **fator categórico** (8 variáveis indicadoras, com `RAD = 1` como referência). Como alguns níveis têm poucas observações (17 em `RAD = 7`, 20 em `RAD = 1`), vale avaliar na modelagem o agrupamento dos níveis, por exemplo `RAD = 24` contra os demais, desde que essa decisão seja fundamentada nos dados e avaliada quanto à sua contribuição para o modelo.

Em síntese, a análise indica que **`logCRIM` e `logLSTAT` são alternativas promissoras para aproximar relações não lineares de relações lineares**, enquanto `DIS` pode exigir uma abordagem adicional para representar sua curvatura. `RM` pode ser mantida inicialmente na escala original, e `ZN`, `B`, `CHAS` e `RAD` merecem tratamentos específicos de acordo com suas características e padrões observados.

Essas conclusões orientam a seleção das formas funcionais que serão testadas na modelagem. A decisão final deverá considerar não apenas o formato dos gráficos, mas também a significância dos termos, o ajuste dos modelos e os diagnósticos dos resíduos.

## **Parte 4: Multicolinearidade**

> `CHAS` não entra na matriz de correlação de Pearson, pois é uma variável categórica (binária). Sua associação com `MEDV` foi avaliada na Parte 3 pela comparação entre grupos, e sua relação com os demais preditores é avaliada pelo VIF, onde entra como variável indicadora.

In [26]:
corr = df.drop(columns='CHAS').corr(numeric_only=True)
heatmap_correlacao(corr, 'Matriz de Correlação', 'heatmap_de_correlacao')

Como várias variáveis são assimétricas ou discretas (`CRIM`, `ZN`, `RAD`, `TAX`), a correlação de **Spearman** (baseada em postos) complementa a de Pearson, pois capta relações monotônicas não lineares e é menos sensível a valores extremos. `RAD`, por ser categórica, não entra na matriz de Pearson; por ser ordinal, entra na de Spearman, que depende apenas da ordem dos níveis.

In [27]:
# RAD é ordinal: seus códigos preservam a ordem dos níveis, o que basta para Spearman
corr_spearman = (
    df.drop(columns='CHAS')
    .assign(RAD=df['RAD'].cat.codes)
    .corr(method='spearman', numeric_only=True)
)
heatmap_correlacao(corr_spearman, 'Matriz de Correlação de Spearman', 'heatmap_spearman')

Como `RAD` é categórica, sua associação com `TAX` é avaliada pela distribuição de `TAX` em cada nível de `RAD`, e não pela correlação de Pearson. Para os preditores numéricos e `CHAS`, o VIF é calculado normalmente; para `RAD`, que entra como 8 indicadoras, é usado o **VIF generalizado (GVIF)**, que mede a colinearidade do bloco de indicadoras como um todo. Para comparar variáveis com diferentes graus de liberdade (GL), usa-se `GVIF^(1/(2·GL))`, cujos valores de referência são **2,24** (equivalente a VIF 5) e **3,16** (equivalente a VIF 10).

In [28]:
df.groupby('RAD', observed=True)['TAX'].agg(['count', 'min', 'median', 'max'])

,count,min,median,max
RAD,,,,
1,20,198,284.5,422
2,24,188,270.0,348
3,43,193,233.0,469
4,110,224,306.0,711
5,115,187,358.0,403
6,26,293,391.0,432
7,17,222,330.0,330
8,24,284,307.0,307
24,132,666,666.0,666


In [29]:
colunas_explicativas = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
    'PTRATIO', 'B', 'LSTAT']

calcular_vif(df, colunas_explicativas)

,Variável,GL,GVIF,GVIF^(1/(2·GL))
0,TAX,1,9.903660,3.147008
1,NOX,1,4.666367,2.160177
2,INDUS,1,4.401600,2.097999
3,DIS,1,4.177949,2.044003
4,AGE,1,2.910909,1.706139
5,ZN,1,2.492544,1.578779
6,LSTAT,1,2.282035,1.510641
7,PTRATIO,1,2.187746,1.479103
8,CRIM,1,1.797892,1.340855
9,RM,1,1.750337,1.323003


In [30]:
colunas_explicativas = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'B', 'LSTAT']

calcular_vif(df, colunas_explicativas)

,Variável,GL,GVIF,GVIF^(1/(2·GL))
0,NOX,1,4.656489,2.157890
1,DIS,1,4.169139,2.041847
2,INDUS,1,3.599460,1.897224
3,AGE,1,2.904770,1.704339
4,ZN,1,2.414857,1.553981
5,LSTAT,1,2.281637,1.510509
6,PTRATIO,1,2.178208,1.475875
7,CRIM,1,1.797564,1.340733
8,RM,1,1.750173,1.322941
9,B,1,1.343795,1.159222


In [31]:
colunas_explicativas2 = ['logCRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'logLSTAT']

print(calcular_vif(df, colunas_explicativas2))

    Variável  GL       GVIF  GVIF^(1/(2·GL))
0    logCRIM   1   8.309379         2.882599
1        NOX   1   4.749737         2.179389
2        DIS   1   4.142365         2.035280
3      INDUS   1   3.618410         1.902212
4        AGE   1   3.122714         1.767120
5   logLSTAT   1   3.058498         1.748856
6      logZN   1   2.825872         1.681033
7    PTRATIO   1   2.402946         1.550144
8         RM   1   2.023197         1.422391
9        RAD   8  16.303419         1.190604
10      CHAS   1   1.093650         1.045777


In [32]:
colunas_explicativas2 = ['CRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'PTRATIO', 'logLSTAT']

print(calcular_vif(df, colunas_explicativas2))

    Variável  GL      GVIF  GVIF^(1/(2·GL))
0        NOX   1  4.643204         2.154809
1        DIS   1  4.156783         2.038819
2      INDUS   1  3.624497         1.903811
3        AGE   1  3.102527         1.761399
4   logLSTAT   1  3.032868         1.741513
5      logZN   1  2.835089         1.683772
6    PTRATIO   1  2.340675         1.529926
7         RM   1  2.018737         1.420823
8       CRIM   1  1.744971         1.320974
9        RAD   8  6.296693         1.121875
10      CHAS   1  1.088447         1.043286


Número de condição da matriz de design (preditores padronizados + intercepto). Valores acima de ~30 indicam multicolinearidade relevante:

In [33]:
especificacoes = {
    "Original (com TAX)": ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'],
    "Original (sem TAX)": ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'B', 'LSTAT'],
    "Transformada (logCRIM)": ['logCRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'logLSTAT'],
    "Transformada (CRIM)": ['CRIM', 'logZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'PTRATIO', 'logLSTAT'],
}

pd.Series(
    {nome: calcular_numero_condicao(df, cols) for nome, cols in especificacoes.items()},
    name="Número de condição"
).round(2)

Original (com TAX)        13.09
Original (sem TAX)        11.24
Transformada (logCRIM)    12.34
Transformada (CRIM)       11.13
Name: Número de condição, dtype: float64

### Conclusão — Multicolinearidade

O VIF foi calculado **apenas com as variáveis explicativas**. A variável resposta `MEDV` não deve entrar nesse cálculo, pois o VIF mede quanto cada preditor é explicado pelos demais preditores. Como `RAD` é categórica, foi usado o **VIF generalizado (GVIF)**, que coincide com o VIF para as variáveis com um grau de liberdade; para comparar variáveis com graus de liberdade diferentes, usa-se `GVIF^(1/(2·GL))`, com valores de referência de 2,24 (VIF 5) e 3,16 (VIF 10).

A análise identificou uma associação relevante entre as variáveis `TAX` e `RAD`, sendo esse o principal ponto de atenção para a construção dos modelos de regressão. No modelo contendo ambas as variáveis, `TAX` apresentou GVIF de aproximadamente **9,90** (`GVIF^(1/(2·GL))` = 3,15, praticamente no limite de 3,16), e o bloco de `RAD` apresentou GVIF de **18,88** com 8 graus de liberdade (`GVIF^(1/(2·GL))` = 1,20). A tabela de `TAX` por nível de `RAD` explica esse resultado: **todas as 132 observações com `RAD = 24` têm `TAX = 666`**, enquanto nos demais níveis `TAX` varia entre 187 e 711. Assim, grande parte de `TAX` é determinada pela indicadora de `RAD = 24`. A correlação de Spearman entre as duas variáveis, calculada com `RAD` ordinal, é de **0,70**.

Para investigar esse comportamento, foi recalculado o GVIF após a remoção de `TAX`. Nessa configuração, o GVIF de `RAD` caiu para aproximadamente **6,35** (`GVIF^(1/(2·GL))` = 1,12), enquanto os VIFs das demais variáveis permaneceram abaixo de 4,7. Essa redução indica que uma parcela importante da multicolinearidade identificada no modelo inicial está associada à presença simultânea de `TAX` e `RAD`.

Nas especificações com variáveis transformadas, surgiu um segundo ponto de atenção: **`logCRIM` apresentou VIF de 8,31** (`GVIF^(1/(2·GL))` = 2,88, acima da referência de 2,24), e o GVIF de `RAD` subiu para 16,30, enquanto `CRIM` na escala original, na mesma especificação, apresentou VIF de apenas 1,74 (com `RAD` em 6,30). Isso ocorre porque a transformação logarítmica torna `CRIM` mais linearmente associada aos demais preditores, em especial ao bloco de indicadoras de `RAD`. Como `logCRIM` foi apontada como uma transformação promissora nas Partes 2 e 3, esse aumento de colinearidade deve ser considerado na escolha da forma funcional.

O **número de condição** da matriz de design (preditores padronizados + intercepto, com `RAD` em indicadoras) foi de 13,09 com `TAX`, 11,24 sem `TAX`, 12,34 na especificação com `logCRIM` e 11,13 na especificação com `CRIM`. Os valores são maiores que os obtidos com `RAD` numérica, pois as 8 indicadoras acrescentam colunas e alguns níveis têm poucas observações, mas todos estão abaixo da referência de 30. Isso indica que não há multicolinearidade severa no conjunto como um todo; os problemas identificados são localizados (`TAX`/`RAD` e `logCRIM`/`RAD`).

A variável `CHAS`, incluída no cálculo como indicadora (0/1), apresentou VIF entre **1,09 e 1,10** em todas as especificações, indicando que não possui redundância relevante com os demais preditores.

A correlação de Spearman também reforça conclusões da Parte 3: a associação entre `LSTAT` e `MEDV` é de **−0,82** por Spearman, contra **−0,56** por Pearson, o que confirma uma relação monotônica forte, mas não linear, compatível com a transformação `logLSTAT`.

Diante desses resultados, **a remoção de `TAX` é uma alternativa a ser considerada**, mantendo `RAD` no modelo como fator categórico. A decisão, entretanto, não deve se basear exclusivamente nos valores de GVIF: também é importante considerar a relevância teórica de cada variável e comparar o ajuste e os diagnósticos dos modelos com diferentes especificações. Uma alternativa é ajustar modelos com `TAX` e sem `TAX`, mantendo as demais condições comparáveis, e avaliar as diferenças encontradas. A remoção de `RAD`, ou sua redução a uma indicadora de `RAD = 24`, também pode ser investigada caso exista justificativa para essa escolha.

Em síntese, **a multicolinearidade entre `TAX` e `RAD` é o principal problema identificado nesta etapa**, seguida do aumento de colinearidade causado pela transformação `logCRIM`. Os demais preditores apresentam níveis de VIF relativamente controlados. Esses resultados foram obtidos com as 5 observações suspeitas mantidas (ver Parte 1) e devem ser revistos caso elas sejam removidas.

## Adicionando informações ao conjunto de dados

> O arquivo processado mantém todas as 511 observações, incluindo as 5 linhas suspeitas (índices 506 a 510) e as 5 observações com `RM` ausente. O tratamento dessas observações deve ser feito antes do ajuste do modelo.

In [34]:
df.to_csv('../data/housing_processed.csv', index=False)

## **Parte 5: Modelo completo e diagnóstico dos resíduos**

Nesta parte é ajustado o modelo de Regressão Linear Múltipla com **todas as 13 variáveis explicativas** e `MEDV` como resposta, na escala original:

$$
MEDV = \beta_0 + \beta_1CRIM + \beta_2ZN + \cdots + \beta_{13}LSTAT + \varepsilon
$$

Decisões tomadas a partir da EDA:

- `CHAS` entra como indicadora (0/1) e `RAD` como fator categórico (8 indicadoras, `RAD = 1` como referência). Assim, o modelo tem **p = 21 parâmetros** (20 coeficientes + intercepto);
- as 5 observações com `RM` ausente são removidas, pois o MQO exige linhas completas (n = 506);
- as 5 observações suspeitas (índices 506 a 510) são **mantidas** no ajuste principal, para verificar se o diagnóstico as identifica. No final, o modelo é reajustado sem elas (análise de sensibilidade).

O diagnóstico é feito **numericamente** (tabelas com o índice de cada observação sinalizada e testes estatísticos) e graficamente.

In [35]:
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.sandbox.stats.runs import runstest_1samp
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.stattools import durbin_watson

formula = (
    "MEDV ~ CRIM + ZN + INDUS + C(CHAS) + NOX + RM + AGE + DIS + C(RAD)"
    " + TAX + PTRATIO + B + LSTAT"
)

colunas_modelo = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD',
    'TAX', 'PTRATIO', 'B', 'LSTAT', 'MEDV']
df_modelo = df[colunas_modelo].dropna()

modelo = smf.ols(formula, data=df_modelo).fit()

n = int(modelo.nobs)
p = int(modelo.df_model) + 1  # coeficientes + intercepto
print(f"n = {n} | p = {p} | observações por parâmetro = {n / p:.1f}")

modelo.summary()

n = 506 | p = 21 | observações por parâmetro = 24.1


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   MEDV   R-squared:                       0.647
Model:                            OLS   Adj. R-squared:                  0.632
Method:                 Least Squares   F-statistic:                     44.41
Date:                Wed, 07 Oct 2026   Prob (F-statistic):           4.74e-96
Time:                        21:24:35   Log-Likelihood:                -1594.4
No. Observations:                 506   AIC:                             3231.
Df Residuals:                     485   BIC:                             3320.
Df Model:                          20                                         
Covariance Type:            nonrobust                                         
================================================================================
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept       16.4597      6.378      2.581      0.010       3.928      28.992
C(CHAS)[T.1]     2.9514      1.062      2.779      0.006       0.865       5.038
C(RAD)[T.2]      1.2177      1.836      0.663      0.508      -2.391       4.826
C(RAD)[T.3]      6.0448      1.622      3.727      0.000       2.858       9.231
C(RAD)[T.4]      1.5360      1.457      1.054      0.292      -1.327       4.399
C(RAD)[T.5]      3.2361      1.488      2.174      0.030       0.312       6.160
C(RAD)[T.6]      0.3769      1.800      0.209      0.834      -3.161       3.915
C(RAD)[T.7]      4.1332      1.931      2.140      0.033       0.338       7.928
C(RAD)[T.8]      4.5919      1.857      2.473      0.014       0.943       8.241
C(RAD)[T.24]     5.1025      2.186      2.334      0.020       0.807       9.398
CRIM            -0.1496      0.040     -3.735      0.000      -0.228      -0.071
ZN               0.0608      0.017      3.499      0.001       0.027       0.095
INDUS            0.0068      0.079      0.086      0.932      -0.148       0.162
NOX            -17.8755      4.800     -3.724      0.000     -27.308      -8.443
RM               5.5395      0.483     11.470      0.000       4.591       6.488
AGE             -0.0474      0.016     -3.046      0.002      -0.078      -0.017
DIS             -1.7658      0.251     -7.049      0.000      -2.258      -1.274
TAX             -0.0074      0.005     -1.548      0.122      -0.017       0.002
PTRATIO         -0.6550      0.173     -3.796      0.000      -0.994      -0.316
B                0.0118      0.003      3.606      0.000       0.005       0.018
LSTAT           -0.1184      0.050     -2.387      0.017      -0.216      -0.021
==============================================================================
Omnibus:                      369.357   Durbin-Watson:                   0.948
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             7422.524
Skew:                           2.942   Prob(JB):                         0.00
Kurtosis:                      20.817   Cond. No.                     1.55e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.55e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

### 5.1 Significância do modelo

- **Teste F global**: $H_0: \beta_1 = \cdots = \beta_{20} = 0$ (nenhuma variável explica `MEDV`);
- **Testes t individuais**: $H_0: \beta_j = 0$, dado que as demais variáveis estão no modelo;
- **Teste F parcial (ANOVA tipo II)**: para `RAD`, as 8 indicadoras devem ser testadas **em bloco**, pois o teste t de cada indicadora compara apenas um nível com a referência (`RAD = 1`).

In [36]:
print(f"R²             = {modelo.rsquared:.4f}")
print(f"R² ajustado    = {modelo.rsquared_adj:.4f}")
print(f"Erro padrão residual = {np.sqrt(modelo.mse_resid):.4f}")
print(f"F global = {modelo.fvalue:.2f} (GL = {int(modelo.df_model)}, {int(modelo.df_resid)}) | p-valor = {modelo.f_pvalue:.3g}")

ic = modelo.conf_int()
tabela_coef = pd.DataFrame({
    "Coeficiente": modelo.params,
    "Erro padrão": modelo.bse,
    "IC 95% inf": ic[0],
    "IC 95% sup": ic[1],
    "t": modelo.tvalues,
    "p-valor": modelo.pvalues,
})
tabela_coef["Significativo (5%)"] = np.where(tabela_coef["p-valor"] < 0.05, "sim", "não")
tabela_coef.round(4)

R²             = 0.6468
R² ajustado    = 0.6323
Erro padrão residual = 5.7734
F global = 44.41 (GL = 20, 485) | p-valor = 4.74e-96


,Coeficiente,Erro padrão,IC 95% inf,IC 95% sup,t,p-valor,Significativo (5%)
Intercept,16.4597,6.3781,3.9277,28.9918,2.5807,0.0102,sim
C(CHAS)[T.1],2.9514,1.0619,0.8649,5.0378,2.7793,0.0057,sim
C(RAD)[T.2],1.2177,1.8364,-2.3905,4.8260,0.6631,0.5076,não
C(RAD)[T.3],6.0448,1.6217,2.8584,9.2312,3.7275,0.0002,sim
C(RAD)[T.4],1.5360,1.4571,-1.3269,4.3990,1.0542,0.2923,não
C(RAD)[T.5],3.2361,1.4884,0.3116,6.1605,2.1742,0.0302,sim
C(RAD)[T.6],0.3769,1.8005,-3.1607,3.9146,0.2094,0.8343,não
C(RAD)[T.7],4.1332,1.9315,0.3381,7.9282,2.1399,0.0329,sim
C(RAD)[T.8],4.5919,1.8570,0.9431,8.2407,2.4727,0.0138,sim
C(RAD)[T.24],5.1025,2.1863,0.8067,9.3984,2.3338,0.0200,sim


In [37]:
# Teste F parcial de cada variável (ANOVA tipo II): RAD é testada como um bloco de 8 indicadoras
anova_ii = anova_lm(modelo, typ=2)
anova_ii["Significativo (5%)"] = np.where(anova_ii["PR(>F)"] < 0.05, "sim", "não")
anova_ii.round(4)

,sum_sq,df,F,PR(>F),Significativo (5%)
C(CHAS),257.4818,1.0,7.7247,0.0057,sim
C(RAD),1150.8255,8.0,4.3158,0.0000,sim
CRIM,465.0831,1.0,13.9530,0.0002,sim
ZN,408.0458,1.0,12.2418,0.0005,sim
INDUS,0.2452,1.0,0.0074,0.9317,não
NOX,462.1969,1.0,13.8664,0.0002,sim
RM,4385.4753,1.0,131.5692,0.0000,sim
AGE,309.3415,1.0,9.2806,0.0024,sim
DIS,1656.0113,1.0,49.6822,0.0000,sim
TAX,79.8853,1.0,2.3966,0.1222,não


### 5.2 Medidas de diagnóstico

Para cada observação são calculados:

| Medida | Definição | Ponto de corte |
| --- | --- | --- |
| Alavanca $h_{ii}$ | diagonal da matriz chapéu $H = X(X'X)^{-1}X'$ | $h_{ii} > 2p/n$ (e $3p/n$) |
| Resíduo padronizado $r_i$ | $e_i / \sqrt{QME(1-h_{ii})}$ | $\lvert r_i \rvert > 2$ |
| Resíduo studentizado $t_i$ | como $r_i$, mas com $QME_{(i)}$ calculado sem a observação $i$ | $\lvert t_i \rvert > 3$ (outlier) |
| Distância de Cook $D_i$ | influência da observação em todos os coeficientes | $D_i > 4/n$ (e $D_i > 1$) |
| DFFITS | influência da observação no próprio valor ajustado | $\lvert DFFITS \rvert > 2\sqrt{p/n}$ |

Os pontos de corte dependem de **n** e **p**, ou seja, são ajustados ao tamanho do conjunto de dados.

In [38]:
influencia = modelo.get_influence()

diagnostico = pd.DataFrame({
    "MEDV": df_modelo["MEDV"],
    "Ajustado": modelo.fittedvalues,
    "Resíduo": modelo.resid,
    "Alavanca": influencia.hat_matrix_diag,
    "Res. padronizado": influencia.resid_studentized_internal,
    "Res. studentizado": influencia.resid_studentized_external,
    "Cook": influencia.cooks_distance[0],
    "DFFITS": influencia.dffits[0],
}, index=df_modelo.index)

corte_alavanca = 2 * p / n
corte_alavanca_alto = 3 * p / n
corte_cook = 4 / n
corte_dffits = 2 * np.sqrt(p / n)

print(f"Soma das alavancas = {diagnostico['Alavanca'].sum():.2f} (deve ser igual a p = {p})")
print(f"Alavanca média p/n = {p / n:.4f}")
print(f"Cortes: alavanca 2p/n = {corte_alavanca:.4f} | 3p/n = {corte_alavanca_alto:.4f} | "
      f"Cook 4/n = {corte_cook:.4f} | DFFITS 2·sqrt(p/n) = {corte_dffits:.4f}")

Soma das alavancas = 21.00 (deve ser igual a p = 21)
Alavanca média p/n = 0.0415
Cortes: alavanca 2p/n = 0.0830 | 3p/n = 0.1245 | Cook 4/n = 0.0079 | DFFITS 2·sqrt(p/n) = 0.4074


### 5.3 Pontos de alavanca

Pontos de alavanca são observações com valores **atípicos nas variáveis explicativas** (longe do centro da nuvem de $X$). Eles não são necessariamente ruins: só se tornam um problema quando também têm resíduo grande, o que os torna influentes.

In [39]:
alavanca = diagnostico[diagnostico["Alavanca"] > corte_alavanca].sort_values("Alavanca", ascending=False)
print(f"Observações com h > 2p/n: {len(alavanca)} ({len(alavanca) / n:.1%})")
print(f"Observações com h > 3p/n: {(diagnostico['Alavanca'] > corte_alavanca_alto).sum()}")

alavanca.join(df_modelo.drop(columns="MEDV")).round(3)

Observações com h > 2p/n: 23 (4.5%)
Observações com h > 3p/n: 4


,MEDV,Ajustado,Resíduo,Alavanca,Res. padronizado,Res. studentizado,Cook,DFFITS,CRIM,ZN,...,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
509,67.0,18.338,48.662,0.320,10.219,11.524,2.337,7.900,0.778,0.0,...,0,0.561,6.222,34.0,2.543,3,329,23.0,343.00,76.00
380,10.4,12.571,-2.171,0.308,-0.452,-0.452,0.004,-0.301,88.976,0.0,...,0,0.671,6.968,91.9,1.416,24,666,20.2,396.90,17.21
418,8.8,3.190,5.610,0.190,1.080,1.080,0.013,0.522,73.534,0.0,...,0,0.679,5.957,100.0,1.803,24,666,20.2,16.45,20.62
405,5.0,6.986,-1.986,0.157,-0.375,-0.374,0.001,-0.161,67.921,0.0,...,0,0.693,5.683,100.0,1.425,24,666,20.2,384.97,22.98
410,15.0,8.668,6.332,0.114,1.165,1.166,0.008,0.419,51.136,0.0,...,0,0.597,5.757,100.0,1.413,24,666,20.2,2.60,10.11
510,24.0,22.281,1.719,0.107,0.315,0.315,0.001,0.109,0.654,0.0,...,0,0.561,6.760,67.0,2.987,3,345,23.0,321.00,45.00
283,50.0,43.949,6.051,0.104,1.107,1.108,0.007,0.377,0.015,90.0,...,1,0.401,7.923,24.8,5.885,1,198,13.6,395.52,3.16
342,16.5,19.556,-3.056,0.098,-0.557,-0.557,0.002,-0.183,0.025,0.0,...,0,0.518,6.540,59.7,6.267,1,422,15.9,389.96,8.65
253,42.8,33.126,9.674,0.097,1.763,1.767,0.016,0.580,0.369,22.0,...,0,0.431,8.259,8.4,8.907,7,330,19.1,396.90,3.54
353,30.1,23.467,6.633,0.096,1.208,1.209,0.007,0.394,0.017,90.0,...,0,0.410,6.728,36.1,12.126,5,187,17.0,384.46,4.50


In [40]:

# Alavanca contra resíduo studentizado; o tamanho do ponto é proporcional à distância de Cook
medidas_62 = diagnostico[["Ajustado", "Resíduo", "Res. studentizado", "Alavanca", "Cook"]]
sinalizados = diagnostico[
    (diagnostico["Alavanca"] > corte_alavanca_alto)
    | (diagnostico["Res. studentizado"].abs() > 3)
    | (diagnostico["Cook"] > corte_cook * 5)
]

fig = go.Figure()
fig.add_trace(pontos(diagnostico["Alavanca"], diagnostico["Res. studentizado"], diagnostico.index, medidas_62,
                     tamanho=np.sqrt(20 + 2000 * diagnostico["Cook"])))
fig.add_trace(go.Scatter(
    x=sinalizados["Alavanca"], y=sinalizados["Res. studentizado"], mode="text", name="Rótulo das sinalizadas",
    text=[str(i) for i in sinalizados.index], textposition="top right", textfont=dict(size=10, color="#52514e"),
    hoverinfo="skip",
))
y_min, y_max = faixa(diagnostico["Res. studentizado"])
x_min, x_max = faixa(diagnostico["Alavanca"])
linha_ref(fig, corte_alavanca, y_min, y_max, "Alavanca 2p/n", cor=COR_ALERTA, vertical=True)
linha_ref(fig, corte_alavanca_alto, y_min, y_max, "Alavanca 3p/n", cor=COR_CRITICO, vertical=True)
for limite in (-3, -2, 2, 3):
    linha_ref(fig, limite, x_min, x_max, f"|resíduo| = {abs(limite)}",
              cor=COR_CRITICO if abs(limite) == 3 else COR_ALERTA, traco="dot")

fig.update_xaxes(title_text="Alavanca (h_ii)")
fig.update_yaxes(title_text="Resíduo studentizado")
mostrar_e_salvar(fig, "alavanca_vs_residuo", titulo="Alavanca x resíduo studentizado (tamanho = distância de Cook)",
                 altura=600)

### 5.4 Outliers e observações influentes

Outliers são observações com **resíduo grande** (valor de `MEDV` mal explicado pelo modelo). Com n = 506, mesmo com erros normais é esperado que cerca de $n \cdot P(|Z| > 3) \approx 1{,}4$ observação passe de $|t_i| > 3$ só por acaso. Por isso, também é usado o **teste de Bonferroni**, que corrige o p-valor pelo número de observações.

In [41]:
outliers = diagnostico[diagnostico["Res. studentizado"].abs() > 3].sort_values("Res. studentizado", key=abs, ascending=False)
print(f"|t_i| > 3: {len(outliers)} observações (esperado sob normalidade: {n * 2 * stats.norm.sf(3):.1f})")

teste_bonferroni = modelo.outlier_test(method="bonf")
outliers_bonf = teste_bonferroni[teste_bonferroni["bonf(p)"] < 0.05]
print(f"Outliers pelo teste de Bonferroni (p < 0,05): {list(outliers_bonf.index)}")

outliers.join(teste_bonferroni["bonf(p)"]).join(df_modelo.drop(columns="MEDV")).round(3)

|t_i| > 3: 10 observações (esperado sob normalidade: 1.4)


Outliers pelo teste de Bonferroni (p < 0,05): [365, 368, 371, 372, 508, 509]


,MEDV,Ajustado,Resíduo,Alavanca,Res. padronizado,Res. studentizado,Cook,DFFITS,bonf(p),CRIM,...,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
509,67.0,18.338,48.662,0.320,10.219,11.524,2.337,7.900,0.000,0.778,...,0,0.561,6.222,34.0,2.543,3,329,23.0,343.00,76.00
368,50.0,15.962,34.038,0.050,6.049,6.284,0.092,1.442,0.000,4.898,...,0,0.631,4.970,100.0,1.332,24,666,20.2,375.52,3.26
508,54.0,20.599,33.401,0.044,5.916,6.136,0.076,1.313,0.000,0.444,...,0,0.561,6.123,98.0,2.987,3,320,23.0,343.00,21.00
371,50.0,21.652,28.348,0.018,4.955,5.080,0.021,0.689,0.000,9.232,...,0,0.631,6.216,100.0,1.169,24,666,20.2,366.15,9.53
372,50.0,22.622,27.378,0.045,4.854,4.971,0.053,1.084,0.000,8.267,...,1,0.668,5.875,89.6,1.130,24,666,20.2,347.88,8.88
365,27.5,6.029,21.471,0.082,3.881,3.938,0.064,1.174,0.048,4.556,...,0,0.718,3.561,87.9,1.613,24,666,20.2,354.70,7.12
369,50.0,28.339,21.661,0.050,3.848,3.905,0.037,0.891,0.055,5.670,...,1,0.631,6.683,96.8,1.357,24,666,20.2,375.33,3.73
370,50.0,30.581,19.419,0.051,3.453,3.493,0.031,0.811,0.264,6.539,...,1,0.631,7.016,97.5,1.202,24,666,20.2,392.05,2.96
367,23.1,4.159,18.941,0.068,3.398,3.436,0.040,0.929,0.324,13.522,...,0,0.631,3.863,100.0,1.511,24,666,20.2,131.42,13.33
506,12.0,28.951,-16.951,0.050,-3.012,-3.038,0.023,-0.696,1.000,0.988,...,0,0.561,6.980,89.0,2.098,3,320,23.0,396.00,12.00


In [42]:
influentes = diagnostico[
    (diagnostico["Cook"] > corte_cook) | (diagnostico["DFFITS"].abs() > corte_dffits)
].sort_values("Cook", ascending=False)

print(f"Cook > 4/n: {(diagnostico['Cook'] > corte_cook).sum()} | Cook > 1: {(diagnostico['Cook'] > 1).sum()}")
print(f"|DFFITS| > 2·sqrt(p/n): {(diagnostico['DFFITS'].abs() > corte_dffits).sum()}")

influentes.head(15).round(4)

Cook > 4/n: 24 | Cook > 1: 1
|DFFITS| > 2·sqrt(p/n): 24


,MEDV,Ajustado,Resíduo,Alavanca,Res. padronizado,Res. studentizado,Cook,DFFITS
509,67.0,18.3375,48.6625,0.3197,10.2190,11.5242,2.3367,7.8998
368,50.0,15.9623,34.0377,0.0500,6.0488,6.2842,0.0917,1.4419
508,54.0,20.5991,33.4009,0.0438,5.9164,6.1358,0.0764,1.3133
365,27.5,6.0291,21.4709,0.0816,3.8806,3.9382,0.0637,1.1737
372,50.0,22.6220,27.3780,0.0454,4.8536,4.9708,0.0534,1.0844
367,23.1,4.1593,18.9407,0.0681,3.3984,3.4361,0.0402,0.9287
369,50.0,28.3385,21.6615,0.0495,3.8484,3.9046,0.0367,0.8912
364,21.9,37.9904,-16.0904,0.0796,-2.9050,-2.9276,0.0347,-0.8608
370,50.0,30.5805,19.4195,0.0512,3.4531,3.4928,0.0306,0.8112
506,12.0,28.9514,-16.9514,0.0499,-3.0122,-3.0377,0.0227,-0.6961


### 5.5 Resíduos entre 2 e 3

Se os erros forem normais, cerca de $P(2 < |Z| \le 3) \approx 4{,}3\%$ dos resíduos padronizados ficam entre 2 e 3 em módulo. Esses pontos são esperados e **não** são outliers, desde que:

1. a quantidade observada seja compatível com a esperada (teste binomial);
2. eles estejam **distribuídos ao acaso**, sem se concentrar em uma faixa de valores ajustados, em um grupo de observações ou em um lado do zero.

A aleatoriedade é verificada no gráfico de resíduos por índice e por valor ajustado e pelo **teste das sequências (runs test)** sobre o sinal dos resíduos.

In [43]:
res_pad = diagnostico["Res. padronizado"]
faixa_2_3 = diagnostico[(res_pad.abs() > 2) & (res_pad.abs() <= 3)]

prob_2_3 = 2 * (stats.norm.cdf(3) - stats.norm.cdf(2))
esperado_2_3 = n * prob_2_3
teste_binomial = stats.binomtest(len(faixa_2_3), n, prob_2_3)

print(f"2 < |r| <= 3: {len(faixa_2_3)} observações ({len(faixa_2_3) / n:.1%})")
print(f"Esperado sob normalidade: {esperado_2_3:.1f} ({prob_2_3:.1%}) | teste binomial p-valor = {teste_binomial.pvalue:.4f}")
print(f"Positivos: {(faixa_2_3['Res. padronizado'] > 0).sum()} | Negativos: {(faixa_2_3['Res. padronizado'] < 0).sum()}")
print(f"Com MEDV = 50 (censura): {(faixa_2_3['MEDV'] == 50).sum()}")

estatistica_runs, p_runs = runstest_1samp(np.sign(modelo.resid.values), cutoff=0)
print(f"\nTeste das sequências (sinal dos resíduos na ordem do arquivo): z = {estatistica_runs:.2f} | p-valor = {p_runs:.4f}")

faixa_2_3.sort_values("Res. padronizado").round(3)

2 < |r| <= 3: 6 observações (1.2%)
Esperado sob normalidade: 21.7 (4.3%) | teste binomial p-valor = 0.0001
Positivos: 4 | Negativos: 2
Com MEDV = 50 (censura): 3

Teste das sequências (sinal dos resíduos na ordem do arquivo): z = -8.11 | p-valor = 0.0000


,MEDV,Ajustado,Resíduo,Alavanca,Res. padronizado,Res. studentizado,Cook,DFFITS
364,21.9,37.990,-16.090,0.080,-2.905,-2.928,0.035,-0.861
401,7.2,18.867,-11.667,0.014,-2.035,-2.042,0.003,-0.241
162,50.0,38.385,11.615,0.061,2.076,2.083,0.013,0.530
412,17.9,4.437,13.463,0.053,2.397,2.409,0.015,0.573
166,50.0,35.747,14.253,0.044,2.524,2.539,0.014,0.542
161,50.0,34.073,15.927,0.038,2.812,2.832,0.015,0.559


In [44]:

medidas_68 = diagnostico[["Ajustado", "Resíduo", "Res. padronizado", "Res. studentizado", "Alavanca", "Cook"]]
faixas_68 = [
    ("|r| ≤ 2", res_pad.abs() <= 2, COR_PONTO),
    ("2 < |r| ≤ 3", (res_pad.abs() > 2) & (res_pad.abs() <= 3), COR_ALERTA),
    ("|r| > 3", res_pad.abs() > 3, COR_CRITICO),
]

fig = make_subplots(rows=1, cols=2, subplot_titles=["Resíduos padronizados por índice",
                                                    "Resíduos padronizados x valores ajustados"])
for col, x, rotulo in [(1, pd.Series(diagnostico.index, index=diagnostico.index), "Índice da observação"),
                       (2, diagnostico["Ajustado"], "Valor ajustado")]:
    for nome, mascara, cor in faixas_68:
        fig.add_trace(pontos(x[mascara], res_pad[mascara], diagnostico.index[mascara], medidas_68[mascara],
                             nome=nome, cor=cor, tamanho=6), row=1, col=col)
    x_min, x_max = faixa(x)
    linha_ref(fig, 0, x_min, x_max, "Resíduo zero", cor="#0b0b0b", traco="solid", row=1, col=col)
    for limite in (-3, -2, 2, 3):
        linha_ref(fig, limite, x_min, x_max, f"|resíduo| = {abs(limite)}",
                  cor=COR_CRITICO if abs(limite) == 3 else COR_ALERTA, row=1, col=col)
    fig.update_xaxes(title_text=rotulo, row=1, col=col)
fig.update_yaxes(title_text="Resíduo padronizado", row=1, col=1)
mostrar_e_salvar(fig, "residuos_padronizados", altura=500)

### 5.6 Linearidade dos resíduos normalizados (gráfico QQ)

Se os erros forem normais, os resíduos studentizados ordenados, comparados com os quantis teóricos da normal padrão, devem formar uma **reta** (linha de 45°). Para julgar se os desvios são maiores que os esperados pelo acaso, é desenhado um **envelope simulado** (100 amostras normais de tamanho n, bandas de 2,5% e 97,5%).

Numericamente, a linearidade do gráfico é medida pela **correlação entre os resíduos ordenados e os quantis teóricos** (estatística de Filliben), e a normalidade pelos testes de Shapiro-Wilk, Jarque-Bera e Anderson-Darling.

> Com n ≈ 500, os testes de normalidade têm poder alto e rejeitam $H_0$ mesmo com desvios pequenos e sem importância prática. Por isso, o gráfico QQ e a correlação são a evidência principal; pelo Teorema Central do Limite, os testes t e F continuam aproximadamente válidos com esse tamanho de amostra se o desvio da normalidade for moderado.

In [45]:
res_stud = np.sort(diagnostico["Res. studentizado"].values)
quantis_teoricos = stats.norm.ppf((np.arange(1, n + 1) - 0.375) / (n + 0.25))

rng = np.random.default_rng(42)
simulacoes = np.sort(rng.standard_normal((100, n)), axis=1)
envelope_inf = np.percentile(simulacoes, 2.5, axis=0)
envelope_sup = np.percentile(simulacoes, 97.5, axis=0)

fora_envelope = ((res_stud < envelope_inf) | (res_stud > envelope_sup)).sum()

tracos, _ = tracos_qq(diagnostico["Res. studentizado"].values, diagnostico.index, quantis_teoricos,
                      envelope_inf, envelope_sup)
fig = go.Figure(tracos)
fig.update_xaxes(title_text="Quantis teóricos N(0, 1)")
fig.update_yaxes(title_text="Resíduos studentizados ordenados")
mostrar_e_salvar(fig, "qqplot_residuos", titulo="Gráfico QQ dos resíduos studentizados", altura=650, largura=750)
correlacao_qq = np.corrcoef(quantis_teoricos, res_stud)[0, 1]
shapiro = stats.shapiro(modelo.resid)
jarque_bera = stats.jarque_bera(modelo.resid)
anderson = stats.anderson(modelo.resid, dist="norm", method="interpolate")

print(f"Pontos fora do envelope: {fora_envelope} de {n} ({fora_envelope / n:.1%})")
print(f"Correlação do gráfico QQ (Filliben): {correlacao_qq:.4f}")
print(f"Assimetria dos resíduos: {stats.skew(modelo.resid):.2f} | Curtose (excesso): {stats.kurtosis(modelo.resid):.2f}")
print(f"Shapiro-Wilk: W = {shapiro.statistic:.4f} | p-valor = {shapiro.pvalue:.3g}")
print(f"Jarque-Bera: JB = {jarque_bera.statistic:.2f} | p-valor = {jarque_bera.pvalue:.3g}")
print(f"Anderson-Darling: A² = {anderson.statistic:.3f} | p-valor = {anderson.pvalue:.3g}")

Pontos fora do envelope: 394 de 506 (77.9%)
Correlação do gráfico QQ (Filliben): 0.8481
Assimetria dos resíduos: 2.94 | Curtose (excesso): 17.82
Shapiro-Wilk: W = 0.7820 | p-valor = 1.97e-25
Jarque-Bera: JB = 7422.52 | p-valor = 0
Anderson-Darling: A² = 20.689 | p-valor = 0.01


### 5.7 Outros pressupostos

- **Linearidade do modelo**: resíduos x valores ajustados com curva LOWESS (deve ser horizontal em torno de zero) e teste **RESET de Ramsey** ($H_0$: a forma funcional linear é adequada);
- **Homocedasticidade**: teste de **Breusch-Pagan** ($H_0$: variância constante);
- **Independência**: estatística de **Durbin-Watson** (valores perto de 2 indicam ausência de autocorrelação de primeira ordem).

In [46]:
fig = go.Figure()
fig.add_trace(pontos(modelo.fittedvalues, modelo.resid, modelo.fittedvalues.index, medidas_diagnostico(modelo)))
fig.add_trace(curva_lowess(modelo.fittedvalues, modelo.resid))
linha_ref(fig, 0, *faixa(modelo.fittedvalues), "Resíduo zero", cor="#0b0b0b", traco="solid")
fig.update_xaxes(title_text="Valor ajustado")
fig.update_yaxes(title_text="Resíduo")
mostrar_e_salvar(fig, "residuos_vs_ajustados", titulo="Resíduos x valores ajustados (LOWESS)", altura=500)
reset = linear_reset(modelo, power=2, use_f=True)
bp_lm, bp_p, _, _ = het_breuschpagan(modelo.resid, modelo.model.exog)

print(f"RESET (ŷ²): F = {reset.fvalue:.2f} | p-valor = {reset.pvalue:.3g}")
print(f"Breusch-Pagan: LM = {bp_lm:.2f} | p-valor = {bp_p:.3g}")
print(f"Durbin-Watson: {durbin_watson(modelo.resid):.3f}")

RESET (ŷ²): F = 74.61 | p-valor = 8.39e-17
Breusch-Pagan: LM = 73.63 | p-valor = 4.61e-08
Durbin-Watson: 0.948


### 5.8 Adequação ao tamanho do conjunto de dados

A tabela abaixo compara a quantidade de observações sinalizadas por cada critério com a quantidade **esperada** para n = 506. Com muitas observações, sempre haverá alguns pontos acima dos cortes; o problema é quando a quantidade observada fica muito acima da esperada.

In [47]:
resumo_tamanho = pd.DataFrame([
    {"Critério": "|r| > 2", "Observado": int((res_pad.abs() > 2).sum()), "Esperado (normal)": n * 2 * stats.norm.sf(2)},
    {"Critério": "2 < |r| <= 3", "Observado": len(faixa_2_3), "Esperado (normal)": esperado_2_3},
    {"Critério": "|t| > 3", "Observado": len(outliers), "Esperado (normal)": n * 2 * stats.norm.sf(3)},
    {"Critério": "Outlier Bonferroni", "Observado": len(outliers_bonf), "Esperado (normal)": 0.05},
    {"Critério": "h > 2p/n", "Observado": len(alavanca), "Esperado (normal)": np.nan},
    {"Critério": "Cook > 4/n", "Observado": int((diagnostico['Cook'] > corte_cook).sum()), "Esperado (normal)": np.nan},
])
resumo_tamanho["Esperado (normal)"] = resumo_tamanho["Esperado (normal)"].round(1)

print(f"n = {n}, p = {p}, n/p = {n / p:.1f} observações por parâmetro")
print(f"GL do resíduo = {int(modelo.df_resid)}")
print(f"Menor nível de RAD: {df_modelo['RAD'].value_counts().min()} observações | CHAS = 1: {(df_modelo['CHAS'] == 1).sum()} observações")
resumo_tamanho

n = 506, p = 21, n/p = 24.1 observações por parâmetro
GL do resíduo = 485
Menor nível de RAD: 17 observações | CHAS = 1: 35 observações


,Critério,Observado,Esperado (normal)
0,|r| > 2,16,23.0
1,2 < |r| <= 3,6,21.7
2,|t| > 3,10,1.4
3,Outlier Bonferroni,6,0.0
4,h > 2p/n,23,NaN
5,Cook > 4/n,24,NaN


### 5.9 Análise de sensibilidade: modelo sem as observações suspeitas

O modelo é reajustado sem as 5 linhas suspeitas (índices 506 a 510) identificadas na Parte 1, para medir o efeito delas nos coeficientes, no ajuste e no diagnóstico.

In [48]:
df_modelo_limpo = df_modelo.drop(index=[i for i in range(506, 511) if i in df_modelo.index])
modelo_limpo = smf.ols(formula, data=df_modelo_limpo).fit()
influencia_limpo = modelo_limpo.get_influence()
n_limpo = int(modelo_limpo.nobs)

def resumir(m, infl, n_obs):
    rp = infl.resid_studentized_internal
    rs = infl.resid_studentized_external
    return {
        "n": n_obs,
        "R²": m.rsquared,
        "R² ajustado": m.rsquared_adj,
        "Erro padrão residual": np.sqrt(m.mse_resid),
        "Coef. significativos (5%)": int((m.pvalues.drop("Intercept") < 0.05).sum()),
        "2 < |r| <= 3": int(((np.abs(rp) > 2) & (np.abs(rp) <= 3)).sum()),
        "|t| > 3": int((np.abs(rs) > 3).sum()),
        "h > 2p/n": int((infl.hat_matrix_diag > 2 * p / n_obs).sum()),
        "Shapiro-Wilk p": stats.shapiro(m.resid).pvalue,
    }

comparacao = pd.DataFrame({
    "Com suspeitas": resumir(modelo, influencia, n),
    "Sem suspeitas": resumir(modelo_limpo, influencia_limpo, n_limpo),
})
print(comparacao.round(4))

pd.DataFrame({
    "Coef. com suspeitas": modelo.params,
    "p com suspeitas": modelo.pvalues,
    "Coef. sem suspeitas": modelo_limpo.params,
    "p sem suspeitas": modelo_limpo.pvalues,
}).round(4)

                           Com suspeitas  Sem suspeitas
n                               506.0000       501.0000
R²                                0.6468         0.7504
R² ajustado                       0.6323         0.7400
Erro padrão residual              5.7734         4.7074
Coef. significativos (5%)        15.0000        16.0000
2 < |r| <= 3                      6.0000        19.0000
|t| > 3                          10.0000         7.0000
h > 2p/n                         23.0000        24.0000
Shapiro-Wilk p                    0.0000         0.0000


,Coef. com suspeitas,p com suspeitas,Coef. sem suspeitas,p sem suspeitas
Intercept,16.4597,0.0102,35.6135,0.0000
C(CHAS)[T.1],2.9514,0.0057,2.5273,0.0037
C(RAD)[T.2],1.2177,0.5076,1.5936,0.2878
C(RAD)[T.3],6.0448,0.0002,4.6426,0.0006
C(RAD)[T.4],1.5360,0.2923,2.5664,0.0317
C(RAD)[T.5],3.2361,0.0302,2.9869,0.0142
C(RAD)[T.6],0.3769,0.8343,1.1558,0.4318
C(RAD)[T.7],4.1332,0.0329,4.8671,0.0021
C(RAD)[T.8],4.5919,0.0138,4.7132,0.0020
C(RAD)[T.24],5.1025,0.0200,7.4981,0.0000


In [49]:
# Posição das linhas suspeitas no diagnóstico do modelo completo
diagnostico.loc[[i for i in range(506, 511) if i in diagnostico.index]].round(4)

,MEDV,Ajustado,Resíduo,Alavanca,Res. padronizado,Res. studentizado,Cook,DFFITS
506,12.0,28.9514,-16.9514,0.0499,-3.0122,-3.0377,0.0227,-0.6961
507,32.0,26.5353,5.4647,0.0531,0.9727,0.9726,0.0025,0.2303
508,54.0,20.5991,33.4009,0.0438,5.9164,6.1358,0.0764,1.3133
509,67.0,18.3375,48.6625,0.3197,10.2190,11.5242,2.3367,7.8998
510,24.0,22.2813,1.7187,0.1066,0.3149,0.3147,0.0006,0.1087


### Conclusão — Modelo completo e diagnóstico dos resíduos

**Significância.** O modelo com as 13 variáveis é **globalmente significativo** (F = 44,41 com 20 e 485 GL; p-valor ≈ 4,7 × 10⁻⁹⁶), com R² = 0,647, R² ajustado = 0,632 e erro padrão residual de 5,77 (mil dólares). Pelos testes individuais e pela ANOVA tipo II, **11 das 13 variáveis são significativas a 5%**: `RM` (F = 131,6), `DIS`, `PTRATIO`, `CRIM`, `NOX`, `B`, `ZN`, `AGE`, `CHAS`, `LSTAT` e o bloco de `RAD` (F = 4,32 com 8 GL, p < 0,001). **`INDUS` (p = 0,93) e `TAX` (p = 0,12) não são significativas**. No caso de `TAX`, o resultado concorda com a Parte 4: a colinearidade com `RAD = 24` infla o erro padrão do seu coeficiente. Dentro de `RAD`, os níveis 2, 4 e 6 não diferem da referência, o que reforça a ideia de agrupar níveis.

**Pontos de alavanca.** Com p = 21 e n = 506, a alavanca média é p/n = 0,042 e o corte 2p/n é 0,083. **23 observações (4,5%)** passam desse corte e 4 passam de 3p/n = 0,125. A maioria tem resíduo pequeno: são regiões com `CRIM` muito alto (380, 418, 405, 410, todas com `RAD = 24`), `ZN = 90` ou o grupo 120–126 (`RAD = 2`, nível pequeno). Elas estão longe do centro de $X$, mas seguem o modelo, portanto **não são influentes**. A exceção é a **observação 509** (`LSTAT = 76`, `MEDV = 67`): alavanca de 0,32 (quase 8 vezes a média), resíduo studentizado de 11,5 e **distância de Cook de 2,34**, a única acima de 1. Ela é ao mesmo tempo ponto de alavanca e outlier, e **sozinha altera o modelo**.

**Outliers.** 10 observações têm |t| > 3, contra cerca de 1,4 esperada sob normalidade, e o teste de Bonferroni confirma 6 (365, 368, 371, 372, 508, 509). Elas formam dois grupos: (i) as linhas suspeitas **506, 508 e 509**, já identificadas na Parte 1; e (ii) as observações **365 a 372**, todas com `RAD = 24` e quase todas com `MEDV = 50`, ou seja, efeito da **censura em 50**: o modelo prevê 16 a 30, mas o valor registrado foi truncado no limite.

**Resíduos entre 2 e 3.** Somente **6 observações (1,2%)** têm 2 < |r| ≤ 3, contra 21,7 (4,3%) esperadas (teste binomial, p = 0,0001). O número **baixo** não indica um bom ajuste: os outliers extremos (sobretudo 509) inflam o QME e reduzem todos os resíduos padronizados. Sem as 5 linhas suspeitas, a contagem sobe para **19**, próxima do esperado para n = 501 (≈ 21,5). Os pontos da faixa são de ambos os sinais (4 positivos e 2 negativos), mas 3 deles têm `MEDV = 50` e os gráficos mostram um agrupamento nos índices 360–415 (`RAD = 24`). O teste das sequências rejeita a aleatoriedade dos sinais na ordem do arquivo (z = −8,11), e o Durbin-Watson de 0,95 indica autocorrelação positiva. Como o arquivo está ordenado por região, isso indica **dependência espacial** entre regiões vizinhas, e não erro de coleta. Assim, **os resíduos não estão distribuídos ao acaso**.

**Linearidade dos resíduos normalizados.** O gráfico QQ **não é linear**: a cauda direita sobe muito acima da reta (assimetria de 2,94 e excesso de curtose de 17,8), 78% dos pontos ficam fora do envelope simulado e a correlação do gráfico QQ é de 0,848, muito abaixo dos ≈ 0,99 esperados para dados normais com esse n. Shapiro-Wilk (W = 0,78; p ≈ 2 × 10⁻²⁵) e Jarque-Bera (p ≈ 0) rejeitam a normalidade, e Anderson-Darling também (A² = 20,7; o p-valor interpolado ≤ 0,01 é o limite inferior da tabela). Com n ≈ 500, esses testes rejeitam até desvios pequenos, mas aqui o desvio é grande e o gráfico o confirma. Ele vem da cauda causada pelas linhas suspeitas, da censura em 50 e da assimetria de `MEDV`.

**Outros pressupostos.** O teste RESET (F = 74,6; p ≈ 10⁻¹⁷) e a curva LOWESS em forma de U nos resíduos x ajustados mostram que a **forma linear não é adequada** para algumas variáveis, como a Parte 3 já indicava (`LSTAT`, `CRIM`, `DIS`). O teste de Breusch-Pagan (p ≈ 5 × 10⁻⁸) indica **heterocedasticidade**.

**Tamanho do conjunto de dados.** Com **24 observações por parâmetro** (485 GL no resíduo), n é adequado para estimar os 21 parâmetros: está acima da regra usual de 10 a 20 observações por parâmetro, e os cortes usados (2p/n, 4/n, 2√(p/n), Bonferroni) foram ajustados a n e p. Com esse n, pelo Teorema Central do Limite, os testes t e F ficam aproximadamente válidos mesmo com desvios moderados da normalidade. **O tamanho não compensa**, porém, a dependência entre as observações e a heterocedasticidade, que afetam os erros padrão. Há ainda dois limites locais: `RAD = 7` tem só 17 observações e `CHAS = 1` só 35, então os coeficientes desses grupos são estimados com pouca precisão.

**Sensibilidade.** A retirada das 5 linhas suspeitas muda o modelo de forma relevante: R² ajustado passa de 0,632 para **0,740**, o erro padrão residual cai de 5,77 para 4,71, o coeficiente de `LSTAT` passa de −0,12 para −0,53, `AGE` deixa de ser significativa (p = 0,90) e `TAX` passa a ser (p = 0,027). Isso confirma que essas linhas devem ser **removidas**.

**Em síntese**, o modelo completo é significativo, mas **não atende aos pressupostos**: há um ponto influente (509), outliers ligados às linhas suspeitas e à censura em 50, resíduos não normais, não aleatórios e heterocedásticos, e não linearidade na forma funcional. Os próximos passos indicados são: remover as linhas 506–510; tratar a censura em `MEDV = 50` (remoção ou modelo censurado/Tobit); usar `log(MEDV)` e as transformações `logLSTAT`/`logCRIM` sugeridas na EDA; reavaliar `INDUS`, `TAX` e o agrupamento de `RAD`; e usar erros padrão robustos (HC3) se a heterocedasticidade continuar.

## **Parte 5 (continuação): Modelos sem outliers e pontos de alavanca**

A conclusão da Parte 5 mostrou que o modelo completo tem outliers, pontos de alavanca e observações influentes. Nesta parte, o modelo é reajustado **sem cada combinação desses grupos de linhas**, e os modelos são comparados por três critérios: **R²**, **análise dos resíduos** e **significância**.

Os grupos são marcados **uma única vez**, no diagnóstico do modelo completo:

| Grupo | Critério |
| --- | --- |
| **S** — suspeitas | linhas 506 a 510 (Parte 1) |
| **O** — outliers | resíduo studentizado $\lvert t_i \rvert > 3$ |
| **L** — alavanca | $h_{ii} > 2p/n$ |
| **I** — influentes | $D_i > 4/n$ ou $\lvert DFFITS_i \rvert > 2\sqrt{p/n}$ |

Com 4 grupos, há $2^4 = 16$ modelos candidatos: do modelo completo (nenhum grupo removido) até a remoção de S+O+L+I. Os grupos **não** são recalculados depois de cada remoção: ao retirar outliers, outras observações passam a parecer atípicas, e a remoção repetida consome os dados sem fim.

Dois cuidados na comparação:

- retirar linhas com resíduo grande **sempre** aumenta o R², e os R² de amostras diferentes não são estritamente comparáveis. Por isso, a fração removida é mostrada, e os candidatos que removem **mais de 10% de n** não entram na escolha;
- cada critério recebe o mesmo peso. A nota final é a média de três postos: R² ajustado; resíduos (média dos postos da correlação do gráfico QQ, do p-valor binomial da faixa $2 < |r| \le 3$, do número de outliers de Bonferroni e do p-valor de Breusch-Pagan); e significância (número de variáveis significativas na ANOVA tipo II, com desempate pelo F global).

In [50]:
from itertools import combinations

sel_grupos = {
    "S": {i for i in range(506, 511) if i in diagnostico.index},
    "O": set(diagnostico.index[diagnostico["Res. studentizado"].abs() > 3]),
    "L": set(diagnostico.index[diagnostico["Alavanca"] > corte_alavanca]),
    "I": set(diagnostico.index[(diagnostico["Cook"] > corte_cook) | (diagnostico["DFFITS"].abs() > corte_dffits)]),
}
for nome, linhas in sel_grupos.items():
    print(f"{nome}: {len(linhas)} linhas")
print(f"O contido em I: {sel_grupos['O'] <= sel_grupos['I']} | S ∩ I: {sorted(sel_grupos['S'] & sel_grupos['I'])}")

sel_variaveis = ["CRIM", "ZN", "INDUS", "C(CHAS)", "NOX", "RM", "AGE", "DIS", "C(RAD)", "TAX", "PTRATIO", "B", "LSTAT"]
sel_prob_2_3 = 2 * (stats.norm.cdf(3) - stats.norm.cdf(2))


def avaliar(m):
    infl = m.get_influence()
    n_m = int(m.nobs)
    rp = infl.resid_studentized_internal
    rs = infl.resid_studentized_external
    qq = np.corrcoef(np.sort(rs), stats.norm.ppf((np.arange(1, n_m + 1) - 0.375) / (n_m + 0.25)))[0, 1]
    n_2_3 = int(((np.abs(rp) > 2) & (np.abs(rp) <= 3)).sum())
    anova = anova_lm(m, typ=2).loc[sel_variaveis, "PR(>F)"]
    return {
        "n": n_m,
        "R²": m.rsquared,
        "R² ajustado": m.rsquared_adj,
        "Erro padrão residual": np.sqrt(m.mse_resid),
        "F global": m.fvalue,
        "Variáveis signif. (de 13)": int((anova < 0.05).sum()),
        "INDUS p": anova["INDUS"],
        "TAX p": anova["TAX"],
        "Correlação QQ": qq,
        "Shapiro W": stats.shapiro(rs).statistic,
        "2<|r|<=3": n_2_3,
        "2<|r|<=3 esperado": n_m * sel_prob_2_3,
        "Binomial p": stats.binomtest(n_2_3, n_m, sel_prob_2_3).pvalue,
        "|t|>3": int((np.abs(rs) > 3).sum()),
        "|t|>3 esperado": n_m * 2 * stats.norm.sf(3),
        "Outliers Bonferroni": int((m.outlier_test(method="bonf")["bonf(p)"] < 0.05).sum()),
        "Breusch-Pagan p": het_breuschpagan(m.resid, m.model.exog)[1],
        "RESET F": linear_reset(m, power=2, use_f=True).fvalue,
        "Durbin-Watson": durbin_watson(m.resid),
    }


# Combinações que removem as mesmas linhas geram o mesmo modelo: fica o nome mais curto
sel_conjuntos = {}
for k in range(len(sel_grupos) + 1):
    for combinacao in combinations(sel_grupos, k):
        nome = "+".join(combinacao) if combinacao else "Completo"
        removidas = frozenset().union(*(sel_grupos[g] for g in combinacao))
        sel_conjuntos.setdefault(removidas, []).append(nome)

sel_linhas = []
sel_modelos = {}
for removidas, nomes in sel_conjuntos.items():
    m = smf.ols(formula, data=df_modelo.drop(index=sorted(removidas))).fit()
    sel_modelos[nomes[0]] = m
    nao_signif = anova_lm(m, typ=2).loc[sel_variaveis, "PR(>F)"].pipe(lambda s: s[s >= 0.05].index)
    sel_linhas.append({
        "Modelo": nomes[0],
        "Equivalentes": ", ".join(nomes[1:]),
        "Removidas": len(removidas),
        "% removido": 100 * len(removidas) / n,
        **avaliar(m),
        "Não significativas": ", ".join(nao_signif),
    })

candidatos = pd.DataFrame(sel_linhas).set_index("Modelo")
print(f"{len(sel_conjuntos)} modelos distintos entre as 16 combinações (O está contido em I)")
candidatos.round(4).T

S: 5 linhas
O: 10 linhas
L: 23 linhas
I: 24 linhas
O contido em I: True | S ∩ I: [506, 508, 509]


12 modelos distintos entre as 16 combinações (O está contido em I)


Modelo,Completo,S,O,L,I,S+O,S+L,S+I,O+L,L+I,S+O+L,S+L+I
Equivalentes,,,,,O+I,,,S+O+I,,O+L+I,,S+O+L+I
Removidas,0,5,10,23,24,12,26,26,32,43,33,44
% removido,0.0,0.9881,1.9763,4.5455,4.7431,2.3715,5.1383,5.1383,6.3241,8.498,6.5217,8.6957
n,506,501,496,483,482,494,480,480,474,463,473,462
R²,0.6468,0.7504,0.8161,0.7074,0.8343,0.8196,0.7448,0.8381,0.8136,0.8315,0.816,0.8343
R² ajustado,0.6323,0.74,0.8084,0.6947,0.8271,0.812,0.7337,0.8311,0.8053,0.8239,0.8078,0.8268
Erro padrão residual,5.7734,4.7074,3.8797,5.1112,3.4505,3.8458,4.7197,3.4118,3.8679,3.4515,3.8421,3.4212
F global,44.4126,72.1654,105.4214,55.8406,116.0259,107.4803,66.9782,118.842,98.8309,109.0825,100.2006,111.0403
Variáveis signif. (de 13),11,11,11,9,11,11,9,11,10,11,10,11
INDUS p,0.9317,0.7934,0.4771,0.9151,0.5817,0.5165,0.6257,0.537,0.8061,0.2316,0.9213,0.1709


In [51]:
# Nota de cada candidato elegível (até 10% de n removido); posto 1 = melhor
elegiveis = candidatos[candidatos["% removido"] <= 10].copy()

posto_r2 = elegiveis["R² ajustado"].rank(ascending=False)
posto_residuos = pd.concat([
    elegiveis["Correlação QQ"].rank(ascending=False),
    elegiveis["Binomial p"].rank(ascending=False),
    elegiveis["Outliers Bonferroni"].rank(ascending=True),
    elegiveis["Breusch-Pagan p"].rank(ascending=False),
], axis=1).mean(axis=1)
posto_signif = (elegiveis["Variáveis signif. (de 13)"] + elegiveis["F global"] / 1e4).rank(ascending=False)

elegiveis["Posto R²"] = posto_r2
elegiveis["Posto resíduos"] = posto_residuos.rank()
elegiveis["Posto significância"] = posto_signif
elegiveis["Nota"] = elegiveis[["Posto R²", "Posto resíduos", "Posto significância"]].mean(axis=1)
elegiveis = elegiveis.sort_values(["Nota", "Removidas"])

print(f"Candidatos elegíveis: {len(elegiveis)} de {len(candidatos)}")
print(f"Excluídos (> 10% removido): {list(candidatos.index[candidatos['% removido'] > 10])}")
melhores = list(elegiveis.index[:2])
print(f"Dois melhores: {melhores}")
pd.set_option("display.max_columns", None)

elegiveis[[
    "Removidas", "% removido", "R² ajustado", "Erro padrão residual", "Variáveis signif. (de 13)", "F global",
    "Correlação QQ", "2<|r|<=3", "2<|r|<=3 esperado", "Binomial p", "|t|>3", "Outliers Bonferroni",
    "Breusch-Pagan p", "Não significativas", "Posto R²", "Posto resíduos", "Posto significância", "Nota",
]].head(5).round(4)

Candidatos elegíveis: 12 de 12
Excluídos (> 10% removido): []
Dois melhores: ['S+I', 'S+L+I']


,Removidas,% removido,R² ajustado,Erro padrão residual,Variáveis signif. (de 13),F global,Correlação QQ,2<|r|<=3,2<|r|<=3 esperado,Binomial p,|t|>3,Outliers Bonferroni,Breusch-Pagan p,Não significativas,Posto R²,Posto resíduos,Posto significância,Nota
Modelo,,,,,,,,,,,,,,,,,,
S+I,26,5.1383,0.8311,3.4118,11,118.8420,0.9853,23,20.5442,0.5717,6,0,0.0,"INDUS, C(CHAS)",1.0,2.0,1.0,1.3333
S+L+I,44,8.6957,0.8268,3.4212,11,111.0403,0.9852,20,19.7738,0.9084,5,0,0.0,"INDUS, C(CHAS)",3.0,1.0,3.0,2.3333
I,24,4.7431,0.8271,3.4505,11,116.0259,0.9853,25,20.6298,0.3105,6,0,0.0,"INDUS, C(CHAS)",2.0,4.0,2.0,2.6667
S+O,12,2.3715,0.8120,3.8458,11,107.4803,0.9777,21,21.1434,1.0000,8,1,0.0,"INDUS, C(CHAS)",5.0,3.0,5.0,4.3333
L+I,43,8.4980,0.8239,3.4515,11,109.0825,0.9845,22,19.8166,0.5662,5,0,0.0,"INDUS, C(CHAS)",4.0,6.0,4.0,4.6667


In [52]:

# Gráfico QQ com envelope simulado: modelo completo x os dois melhores
rng_sel = np.random.default_rng(42)
paineis_82 = []
for nome in ["Completo"] + melhores:
    m = sel_modelos[nome]
    n_m = int(m.nobs)
    rs = m.get_influence().resid_studentized_external
    q, inf, sup = envelope_qq(n_m, rng_sel)
    tracos, fora = tracos_qq(rs, m.fittedvalues.index, q, inf, sup)
    titulo = f"{nome} (n = {n_m})<br>correlação QQ = {np.corrcoef(np.sort(rs), q)[0, 1]:.3f} | fora do envelope: {fora}"
    paineis_82.append((titulo, tracos))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, subplot_titles=[titulo for titulo, _ in paineis_82])
for col, (_, tracos) in enumerate(paineis_82, start=1):
    for traco in tracos:
        fig.add_trace(traco, row=1, col=col)
    fig.update_xaxes(title_text="Quantil teórico N(0, 1)", row=1, col=col)
fig.update_yaxes(title_text="Resíduo studentizado", row=1, col=1)
mostrar_e_salvar(fig, "qqplot_modelos_selecionados", altura=550)

In [53]:
# Coeficientes e p-valores: modelo completo x os dois melhores
pd.concat(
    {nome: pd.DataFrame({"Coef.": sel_modelos[nome].params, "p-valor": sel_modelos[nome].pvalues})
     for nome in ["Completo"] + melhores},
    axis=1,
).round(4)

Completo              S+I            S+L+I        
                Coef. p-valor    Coef. p-valor    Coef. p-valor
Intercept     16.4597  0.0102  15.8377  0.0004  16.6812  0.0003
C(CHAS)[T.1]   2.9514  0.0057   0.8658  0.2157   0.7089  0.3309
C(RAD)[T.2]    1.2177  0.5076   1.2742  0.2472   1.0834  0.3823
C(RAD)[T.3]    6.0448  0.0002   4.5323  0.0000   4.6170  0.0000
C(RAD)[T.4]    1.5360  0.2923   2.6207  0.0025   2.8109  0.0020
C(RAD)[T.5]    3.2361  0.0302   2.9231  0.0010   2.9462  0.0018
C(RAD)[T.6]    0.3769  0.8343   1.2395  0.2455   1.2484  0.2673
C(RAD)[T.7]    4.1332  0.0329   3.6055  0.0020   4.2062  0.0008
C(RAD)[T.8]    4.5919  0.0138   4.1621  0.0002   4.1819  0.0004
C(RAD)[T.24]   5.1025  0.0200   6.3381  0.0000   6.2775  0.0001
CRIM          -0.1496  0.0002  -0.1527  0.0000  -0.1489  0.0014
ZN             0.0608  0.0005   0.0479  0.0000   0.0421  0.0002
INDUS          0.0068  0.9317  -0.0298  0.5370  -0.0847  0.1709
NOX          -17.8755  0.0002  -9.8927  0.0008  -8.8857  0.0032
RM             5.5395  0.0000   5.3471  0.0000   5.2523  0.0000
AGE           -0.0474  0.0024  -0.0314  0.0022  -0.0308  0.0031
DIS           -1.7658  0.0000  -1.2680  0.0000  -1.2956  0.0000
TAX           -0.0074  0.1222  -0.0112  0.0001  -0.0100  0.0087
PTRATIO       -0.6550  0.0002  -0.8333  0.0000  -0.8537  0.0000
B              0.0118  0.0003   0.0128  0.0000   0.0124  0.0000
LSTAT         -0.1184  0.0174  -0.2383  0.0000  -0.2493  0.0000

### Conclusão — Seleção dos modelos sem outliers e alavancas

**Candidatos.** Das 16 combinações, só **12 modelos são distintos**, pois todos os outliers (O) também são influentes (I). Por exemplo, S+I e S+O+I removem as mesmas linhas. Nenhum candidato remove mais de 10% de n (o máximo é 44 linhas, 8,7%), então todos entraram na escolha.

**1º lugar: S+I (sem as linhas suspeitas e sem as influentes; 26 linhas, 5,1% de n).** É o melhor em dois dos três critérios e o segundo no terceiro:

- **R²**: tem o maior R² ajustado (0,632 → **0,831**). O erro padrão residual cai de 5,77 para 3,41 mil dólares;
- **Significância**: tem o maior F global (44,4 → **118,8**). 11 das 13 variáveis são significativas. `TAX` passa a ser significativa (p = 0,0001), e `INDUS` (p = 0,54) e `CHAS` (p = 0,22) não são. Alguns coeficientes mudam muito: `LSTAT` dobra (−0,12 → −0,24) e `NOX` cai quase pela metade (−17,9 → −9,9). Isso mostra que as linhas influentes distorciam o modelo completo;
- **Resíduos**: a correlação do gráfico QQ sobe de 0,848 para **0,985**, e o W de Shapiro-Wilk de 0,73 para 0,97. Há 23 resíduos em $2 < |r| \le 3$, para 20,5 esperados (teste binomial p = 0,57, compatível com a normal). Não há outlier pelo teste de Bonferroni. Ainda há 6 resíduos com $|t| > 3$, para 1,3 esperados.

**2º lugar: S+L+I (também sem os pontos de alavanca; 44 linhas, 8,7% de n).** Tem os **melhores resíduos**: 20 resíduos em $2 < |r| \le 3$ para 19,8 esperados (p = 0,91), apenas 5 com $|t| > 3$, nenhum outlier de Bonferroni e correlação QQ de 0,985. R² ajustado (0,827) e F (111,0) são quase iguais aos do 1º lugar, com as mesmas 11 variáveis significativas. Ficou em 2º lugar porque remove **18 linhas a mais** sem ganho em R² ou significância. Essas linhas são pontos de alavanca que não são influentes: estão longe do centro dos $X$, mas o modelo os ajusta bem, e por isso ajudam a estimar os coeficientes com precisão.

**Por que não os outros.** O modelo **I** (sem as influentes) é quase igual ao S+I, mas mantém as linhas suspeitas 507 e 510, e a faixa 2–3 tem mais pontos que o esperado (25 para 20,6). O modelo **S+O** remove só 12 linhas, mas fica atrás em R² ajustado (0,812) e na correlação QQ (0,978). Ele também ainda tem 1 outlier de Bonferroni. Os modelos que removem **L** sem **I** (L, S+L) são os piores: perdem `CRIM`, `AGE` e `TAX`, e os resíduos continuam longe da normal.

**O que a remoção não resolve.** Em todos os candidatos:

- a variância continua não constante (Breusch-Pagan p ≈ 10⁻⁶ no S+I);
- a autocorrelação continua (Durbin-Watson ≈ 1,24);
- o teste RESET **piora** (F = 74,6 → 231; p ≈ 10⁻⁴²). Sem os pontos extremos, a curvatura que sobra na relação fica mais clara.

Das 26 linhas removidas no S+I, 11 são observações censuradas em `MEDV = 50` (de 18 no total), então a remoção trata a censura só em parte.

**Recomendação.** Usar o **S+I como modelo principal** e o **S+L+I como verificação**: os dois levam às mesmas conclusões sobre quais variáveis são significativas. Só as linhas 506–510 são erros de dados. As outras linhas removidas são casas reais, então a remoção deve ser registrada no relatório. Por causa da heterocedasticidade, os erros padrão devem ser robustos (HC3). A forma funcional (transformação log e o termo `RM²`) é tratada na Parte 6.

## **Parte 6: Termo quadrático de `RM`**

A Parte 5 mostrou que a forma linear não é adequada (RESET com p ≈ 10⁻¹⁷ e resíduos em forma de U). Nesta parte é avaliado o termo **`RM²`**, que foi o termo de potência com melhor evidência na verificação exploratória (ver `TERMOS_NAO_LINEARES.md`).

A ideia é simples: **um cômodo a mais não vale o mesmo em todas as casas**. No modelo linear, o efeito de `RM` é uma constante $\beta_1$. Com o termo quadrático,

$$
\log(MEDV) = \cdots + \beta_1 (RM - \overline{RM}) + \beta_2 (RM - \overline{RM})^2 + \cdots
$$

e o efeito de um cômodo a mais passa a depender do tamanho da casa:

$$
\frac{\partial \log(MEDV)}{\partial RM} = \beta_1 + 2\beta_2 (RM - \overline{RM})
$$

Decisões desta parte:

- resposta `logMEDV` e preditores `logLSTAT` e `logDIS`, como sugerido nas Partes 2 e 3;
- remoção das 5 linhas suspeitas (índices 506 a 510), como indicado na Parte 5, e das 5 linhas com `RM` ausente (n = 501);
- `RM` **centralizado** (subtraída a média) antes de elevar ao quadrado.

In [54]:
df_rm = df.loc[:505].dropna(subset=["RM"]).copy()
media_rm = df_rm["RM"].mean()
df_rm["RM_c"] = df_rm["RM"] - media_rm
print(f"n = {len(df_rm)} | média de RM = {media_rm:.2f}")

# Mediana de MEDV por faixa de RM: o ganho por faixa cresce com RM (curva convexa)
faixas_rm = pd.cut(df_rm["RM"], [3, 5, 5.5, 6, 6.5, 7, 7.5, 9])
resumo_faixas = df_rm.groupby(faixas_rm, observed=True)["MEDV"].agg(["count", "median"])
resumo_faixas["Ganho da faixa anterior"] = resumo_faixas["median"].diff()
resumo_faixas

n = 501 | média de RM = 6.28


,count,median,Ganho da faixa anterior
RM,,,
"(3.0, 5.0]",16,13.80,NaN
"(5.0, 5.5]",26,14.40,0.60
"(5.5, 6.0]",130,18.95,4.55
"(6.0, 6.5]",178,21.15,2.20
"(6.5, 7.0]",87,26.60,5.45
"(7.0, 7.5]",37,33.30,6.70
"(7.5, 9.0]",27,46.70,13.40


Ajuste do modelo com `RM` linear e com `RM` centralizado + `RM²`, comparados pelo teste F parcial (modelos aninhados), AIC, R² ajustado e teste RESET:

In [55]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

base_rm = (
    "logMEDV ~ CRIM + ZN + INDUS + C(CHAS) + NOX + AGE + logDIS + C(RAD)"
    " + PTRATIO + B + logLSTAT"
)

modelo_rm_linear = smf.ols(base_rm + " + RM", data=df_rm).fit()
modelo_rm_quad = smf.ols(base_rm + " + RM_c + I(RM_c**2)", data=df_rm).fit()

print(anova_lm(modelo_rm_linear, modelo_rm_quad))
print()
print(pd.DataFrame({
    "R² ajustado": [modelo_rm_linear.rsquared_adj, modelo_rm_quad.rsquared_adj],
    "AIC": [modelo_rm_linear.aic, modelo_rm_quad.aic],
    "RESET p-valor": [
        linear_reset(modelo_rm_linear, power=2, use_f=True).pvalue,
        linear_reset(modelo_rm_quad, power=2, use_f=True).pvalue,
    ],
}, index=["RM linear", "RM + RM²"]).round(4))

b1 = modelo_rm_quad.params["RM_c"]
b2 = modelo_rm_quad.params["I(RM_c ** 2)"]
print(f"\nβ1 (RM_c) = {b1:.4f} | β2 (RM_c²) = {b2:.4f} | p-valor de β2 = {modelo_rm_quad.pvalues['I(RM_c ** 2)']:.2g}")

   df_resid        ssr  df_diff   ss_diff          F    Pr(>F)
0     481.0  16.567990      0.0       NaN        NaN       NaN
1     480.0  15.992446      1.0  0.575544  17.274461  0.000038

           R² ajustado       AIC  RESET p-valor
RM linear       0.7955 -246.1995         0.2879
RM + RM²        0.8022 -261.9129         0.0002

β1 (RM_c) = 0.0696 | β2 (RM_c²) = 0.0395 | p-valor de β2 = 3.8e-05


Efeito de **um cômodo a mais** sobre `MEDV`, em %, para casas de tamanhos diferentes. No modelo linear o efeito é o mesmo para todas; no modelo quadrático ele é $e^{\beta_1 + 2\beta_2 (RM - \overline{RM})} - 1$:

In [56]:
efeito_linear = 100 * (np.exp(modelo_rm_linear.params["RM"]) - 1)
tamanhos = {"5": 5.0, "6": 6.0, f"{media_rm:.2f} (média)": media_rm, "7": 7.0, "8": 8.0}

pd.DataFrame({
    "Modelo linear (%)": [efeito_linear] * len(tamanhos),
    "Modelo com RM² (%)": [100 * (np.exp(b1 + 2 * b2 * (rm - media_rm)) - 1) for rm in tamanhos.values()],
}, index=pd.Index(tamanhos.keys(), name="RM")).round(1)

,Modelo linear (%),Modelo com RM² (%)
RM,,
5,7.1,-3.1
6,7.1,4.8
6.28 (média),7.1,7.2
7,7.1,13.4
8,7.1,22.8


In [57]:
vertice = media_rm - b1 / (2 * b2)
print(f"Vértice da parábola: RM = {vertice:.2f}")
print(f"Observações com RM < 5,5: {(df_rm['RM'] < 5.5).sum()}")

Vértice da parábola: RM = 5.40
Observações com RM < 5,5: 42


Por que **centralizar**: sem centralização, `RM` e `RM²` são quase a mesma coluna. As duas versões têm o mesmo ajuste e as mesmas previsões; muda apenas a estabilidade e a interpretação dos coeficientes:

In [58]:
modelo_rm_bruto = smf.ols(base_rm + " + RM + I(RM**2)", data=df_rm).fit()

linhas = []
for nome, x in [("RM bruto", df_rm["RM"]), ("RM centralizado", df_rm["RM_c"])]:
    X = np.column_stack([np.ones(len(x)), x, x ** 2])
    linhas.append({
        "Correlação (X, X²)": np.corrcoef(x, x ** 2)[0, 1],
        "VIF entre X e X²": variance_inflation_factor(X, 1),
    })

comparacao = pd.DataFrame(linhas, index=["RM bruto", "RM centralizado"])
comparacao["Coef. de RM"] = [modelo_rm_bruto.params["RM"], b1]
comparacao["Erro padrão"] = [modelo_rm_bruto.bse["RM"], modelo_rm_quad.bse["RM_c"]]
comparacao["Mesmo AIC"] = [modelo_rm_bruto.aic, modelo_rm_quad.aic]
comparacao.round(3)

,"Correlação (X, X²)",VIF entre X e X²,Coef. de RM,Erro padrão,Mesmo AIC
RM bruto,0.995,91.685,-0.427,0.120,-261.913
RM centralizado,0.206,1.044,0.070,0.017,-261.913


In [59]:
# Curvas de RM com os demais preditores fixados na média (numéricos) ou na categoria mais comum (fatores)
def fixar_demais(dados):
    fixo = dados.copy()
    for v in ["CRIM", "ZN", "INDUS", "NOX", "AGE", "logDIS", "PTRATIO", "B", "logLSTAT"]:
        fixo[v] = df_rm[v].mean()
    for v in ["CHAS", "RAD"]:
        fixo[v] = pd.Categorical([df_rm[v].mode()[0]] * len(fixo), categories=df_rm[v].cat.categories,
                                 ordered=df_rm[v].cat.ordered)
    return fixo

grade = pd.DataFrame({"RM": np.linspace(df_rm["RM"].min(), df_rm["RM"].max(), 200)})
grade["RM_c"] = grade["RM"] - media_rm
grade = fixar_demais(grade)

# Resíduo parcial: resíduo do modelo quadrático somado à previsão com os demais preditores fixados
residuo_parcial = modelo_rm_quad.resid + modelo_rm_quad.predict(fixar_demais(df_rm))

fig = go.Figure()
fig.add_trace(pontos(df_rm["RM"], residuo_parcial, df_rm.index, {"Resíduo parcial": residuo_parcial},
                     nome="Resíduo parcial", tamanho=6))
for nome, m, cor, traco in [("RM linear", modelo_rm_linear, COR_REFERENCIA, "dash"),
                            ("RM + RM²", modelo_rm_quad, COR_CURVA, "solid")]:
    fig.add_trace(go.Scatter(x=grade["RM"], y=m.predict(grade), mode="lines", name=nome,
                             line=dict(color=cor, width=2.5, dash=traco),
                             hovertemplate=f"{nome}<br>RM = %{{x:.2f}}<br>log(MEDV) = %{{y:.3f}}<extra></extra>"))
fig.update_xaxes(title_text="RM (número médio de cômodos)")
fig.update_yaxes(title_text="log(MEDV) (efeito parcial de RM)")
mostrar_e_salvar(fig, "logMEDV_vs_RM_quadratico", titulo="Efeito parcial de RM: linear x quadrático", altura=500)

O teste RESET piora com `RM²`. Para entender o motivo, os resíduos médios são comparados por faixa de `RM` e por faixa de valor ajustado. O ajuste também é repetido **sem as observações censuradas** (`MEDV = 50`) e com um termo `logLSTAT²`:

In [60]:
faixas_rm_res = pd.cut(df_rm["RM"], [3, 5.5, 6, 6.5, 7, 7.5, 9])
print("Resíduo médio por faixa de RM:")
print(pd.DataFrame({
    "RM linear": modelo_rm_linear.resid.groupby(faixas_rm_res, observed=True).mean(),
    "RM + RM²": modelo_rm_quad.resid.groupby(faixas_rm_res, observed=True).mean(),
}).round(3))

print("\nResíduo médio do modelo com RM² por quintil do valor ajustado:")
print(modelo_rm_quad.resid.groupby(pd.qcut(modelo_rm_quad.fittedvalues, 5)).mean().round(3).to_string())

df_rm_sem50 = df_rm[df_rm["MEDV"] < 50]
modelo_rm_quad_sem50 = smf.ols(base_rm + " + RM_c + I(RM_c**2)", data=df_rm_sem50).fit()
df_rm["logLSTAT_c"] = df_rm["logLSTAT"] - df_rm["logLSTAT"].mean()
modelo_rm_quad_lstat = smf.ols(base_rm + " + RM_c + I(RM_c**2) + I(logLSTAT_c**2)", data=df_rm).fit()

print(f"\nSem MEDV = 50 (n = {len(df_rm_sem50)}): β2 = {modelo_rm_quad_sem50.params['I(RM_c ** 2)']:.4f}"
      f" | p-valor = {modelo_rm_quad_sem50.pvalues['I(RM_c ** 2)']:.2g}")
print(f"Com RM² + logLSTAT²: RESET p-valor = {linear_reset(modelo_rm_quad_lstat, power=2, use_f=True).pvalue:.3f}"
      f" | p-valor de logLSTAT² = {modelo_rm_quad_lstat.pvalues['I(logLSTAT_c ** 2)']:.2g}")

Resíduo médio por faixa de RM:
            RM linear  RM + RM²
RM                             
(3.0, 5.5]      0.020    -0.027
(5.5, 6.0]      0.014     0.015
(6.0, 6.5]     -0.022    -0.009
(6.5, 7.0]     -0.023    -0.006
(7.0, 7.5]      0.026     0.029
(7.5, 9.0]      0.083     0.007

Resíduo médio do modelo com RM² por quintil do valor ajustado:
(1.723, 2.723]   -0.074
(2.723, 2.927]    0.083
(2.927, 3.105]    0.023
(3.105, 3.339]   -0.016
(3.339, 4.114]   -0.016

Sem MEDV = 50 (n = 485): β2 = 0.0466 | p-valor = 7.6e-06
Com RM² + logLSTAT²: RESET p-valor = 0.138 | p-valor de logLSTAT² = 8.9e-07


### Conclusão — Termo quadrático de `RM`

**Os dados mostram uma curva, não uma reta.** A cada meia unidade de `RM`, o ganho na mediana de `MEDV` cresce: cerca de +0,6 mil dólares entre as duas primeiras faixas e +13,4 mil entre 7–7,5 e 7,5–9 cômodos. Esse padrão, chamado de **convexo**, não pode ser representado por uma reta.

**O modelo com `RM²` é melhor.** O coeficiente de `RM²` é positivo ($\beta_2 \approx 0{,}039$) e significativo (teste F parcial, p ≈ 4 × 10⁻⁵). O AIC cai de −246 para −262 e o R² ajustado sobe de 0,796 para 0,802.

**Interpretação.** O modelo linear dá o mesmo valor médio a todo cômodo extra: **+7,1%**. Com `RM²`, um cômodo a mais vale cerca de **+4,8%** em uma casa com 6 cômodos, **+7,2%** na casa média (6,28), **+13,4%** com 7 e **+22,8%** com 8. Assim, o modelo linear **superestima** o preço das casas pequenas e **subestima** o das casas grandes. Na prática, regiões com 7–8 cômodos por casa são um mercado diferente (imóveis grandes em bairros ricos), e o tamanho funciona como um sinal desse padrão. O estudo original do Boston Housing (Harrison & Rubinfeld, 1978) já usava `RM²`.

**Centralização.** Sem centralizar, `RM` e `RM²` têm correlação de 0,99 e VIF de **91,7** entre si, e o coeficiente de `RM` (−0,43) é o efeito em uma casa com zero cômodos, que não existe. Com `RM` centralizado, o VIF cai para **1,0**, e o coeficiente de `RM` (0,070, ou +7,2%) passa a ser o efeito de um cômodo a mais na casa média. O ajuste e as previsões são os mesmos (mesmo AIC).

**RESET.** O teste RESET piora com `RM²` (p-valor de 0,29 para 0,0002), mas isso **não** indica um problema com `RM²`. Por faixa de `RM`, o modelo linear tem resíduo médio de +0,083 nas casas maiores (subestima o preço); com `RM²`, esse valor cai para +0,007 e as faixas ficam todas perto de zero. A rejeição do RESET vem das casas **mais baratas** (menor quintil do valor ajustado, resíduo médio de −0,07), ou seja, de outra curvatura, ligada principalmente a `LSTAT`: com um termo `logLSTAT²`, o p-valor do RESET sobe para 0,14 e o teste deixa de rejeitar a forma funcional. No modelo linear, as duas curvaturas se compensavam e o RESET não as detectava.

**Censura.** Sem as 16 observações com `MEDV = 50` (n = 485), `RM²` continua positivo e significativo ($\beta_2 \approx 0{,}047$; p ≈ 8 × 10⁻⁶). Assim, o efeito convexo de `RM` **não** é causado pela censura.

**Cuidados.** O vértice da parábola fica em `RM` ≈ 5,4. Abaixo dele, o modelo indica um efeito negativo (−3% por cômodo em `RM` = 5), mas há apenas 42 observações com `RM` < 5,5, então isso é efeito da forma da parábola e não um resultado real. O modelo não deve ser usado fora da faixa de 5 a 8,5 cômodos.

**Em síntese**, `RM²` centralizado deve entrar no modelo: tem forte evidência estatística, uma interpretação prática clara e não cria problema de multicolinearidade. A curvatura que ainda resta (casas mais baratas, ligada a `LSTAT`) deve ser tratada na modelagem final, por exemplo com `logLSTAT²`.

## **Parte 7: Transformações, centralização e pressupostos**

As Partes 5 e 6 rejeitaram os pressupostos de normalidade, homocedasticidade e independência dos erros. Nesta parte é verificado **o que cada tipo de transformação consegue corrigir**:

| Transformação | O que muda | O que **não** muda |
| --- | --- | --- |
| **Centralizar** ($X - \bar X$) ou **padronizar** ($(X - \bar X)/s$) um preditor | intercepto, leitura dos coeficientes, colinearidade entre $X$ e $X^2$ ou $XZ$ | ajuste, previsões, resíduos, R² e todos os testes dos resíduos |
| **Box-Cox na resposta** ($Y^{(\lambda)}$) | forma da distribuição dos erros, variância e linearidade | — |
| **Box-Cox / log em um preditor** | linearidade da relação de $X$ com $Y$ | distribuição dos erros (só indiretamente) |

**Quando centralizar.** A centralização é uma **mudança de origem**: a matriz $X$ centralizada gera o mesmo espaço de colunas (o intercepto absorve as médias), então $\hat y$ e os resíduos são idênticos. Ela é útil apenas em dois casos:

1. **antes de criar potências e produtos** ($X^2$, $X \cdot Z$). Sem centralizar, $X$ e $X^2$ são quase a mesma coluna (Parte 6: VIF de 91,7 para `RM`);
2. **para interpretar** o intercepto e os efeitos principais no ponto médio dos dados (a "casa média"), e não em $X = 0$, que muitas vezes não existe (`RM = 0`, `NOX = 0`).

Neste conjunto, os candidatos são `RM` (já feito na Parte 6), `logLSTAT` (se `logLSTAT²` entrar) e qualquer interação (`RM·PTRATIO`, `NOX·DIS`). As indicadoras (`CHAS` e as 8 de `RAD`) **não** são centralizadas: o valor 0 já tem significado (a categoria de referência). **Padronizar** tem o mesmo efeito e apenas coloca os coeficientes na mesma escala (efeito de 1 desvio padrão), o que permite comparar seus tamanhos.

O conjunto de dados é o da Parte 6 (`df_rm`: sem as 5 linhas suspeitas e sem `RM` ausente, n = 501).

### 7.1 Centralização e padronização não mudam o ajuste

O modelo completo da Parte 5 é reajustado com todos os preditores numéricos padronizados. Se a afirmação acima estiver correta, os resíduos e os testes devem ser idênticos:

In [61]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

numericas = ["CRIM", "ZN", "INDUS", "NOX", "RM", "AGE", "DIS", "TAX", "PTRATIO", "B", "LSTAT"]

df_t = df_rm.copy()
df_padronizado = df_t.copy()
df_padronizado[numericas] = (df_t[numericas] - df_t[numericas].mean()) / df_t[numericas].std()

modelo_bruto = smf.ols(formula, data=df_t).fit()
modelo_padronizado = smf.ols(formula, data=df_padronizado).fit()

print(f"Resíduos idênticos: {np.allclose(modelo_bruto.resid, modelo_padronizado.resid)}")
pd.DataFrame({
    "X original": [modelo_bruto.rsquared_adj, stats.shapiro(modelo_bruto.resid).pvalue,
                   het_breuschpagan(modelo_bruto.resid, modelo_bruto.model.exog)[1],
                   linear_reset(modelo_bruto, power=2, use_f=True).pvalue,
                   variance_inflation_factor(modelo_bruto.model.exog, modelo_bruto.model.exog_names.index("RM")),
                   modelo_bruto.params["Intercept"], modelo_bruto.params["RM"]],
    "X padronizado": [modelo_padronizado.rsquared_adj, stats.shapiro(modelo_padronizado.resid).pvalue,
                      het_breuschpagan(modelo_padronizado.resid, modelo_padronizado.model.exog)[1],
                      linear_reset(modelo_padronizado, power=2, use_f=True).pvalue,
                      variance_inflation_factor(modelo_padronizado.model.exog, modelo_padronizado.model.exog_names.index("RM")),
                      modelo_padronizado.params["Intercept"], modelo_padronizado.params["RM"]],
}, index=["R² ajustado", "Shapiro-Wilk p", "Breusch-Pagan p", "RESET p", "VIF de RM", "Intercepto", "Coef. de RM"]).map(lambda v: f"{v:.4g}")

Resíduos idênticos: True


,X original,X padronizado
R² ajustado,0.74,0.74
Shapiro-Wilk p,1.186e-17,1.186e-17
Breusch-Pagan p,1.323e-07,1.323e-07
RESET p,8.111e-37,8.111e-37
VIF de RM,2.023,2.023
Intercepto,35.61,18.31
Coef. de RM,3.648,2.574


### 7.2 Box-Cox na resposta

A transformação de Box-Cox procura o expoente $\lambda$ que torna os erros do modelo **mais próximos da normal com variância constante**:

$$
Y^{(\lambda)} = \begin{cases} \dfrac{Y^\lambda - 1}{\lambda}, & \lambda \neq 0 \\[4pt] \log Y, & \lambda = 0 \end{cases}
$$

O $\lambda$ é escolhido pela **máxima verossimilhança perfilada do modelo de regressão**, e não da distribuição de `MEDV` sozinha: o pressuposto é sobre os **erros**, não sobre $Y$. Para cada $\lambda$, a resposta é dividida por $\dot y^{\lambda-1}$ ($\dot y$ = média geométrica de `MEDV`), para que as somas de quadrados de $\lambda$ diferentes fiquem na mesma escala, e

$$
\ell(\lambda) = -\frac{n}{2} \log\left(\frac{SQR(\lambda)}{n}\right)
$$

O intervalo de 95% contém os $\lambda$ com $\ell(\lambda) \ge \ell(\hat\lambda) - \tfrac12 \chi^2_{1;\,0,95}$. Valores "redondos" dentro do intervalo são preferidos, pois são mais fáceis de interpretar ($\lambda = 0$: log; $\lambda = 0{,}5$: raiz; $\lambda = 1$: sem transformação).

In [62]:
def perfil_boxcox(dados, lado_direito, lambdas):
    y = dados["MEDV"]
    media_geom = np.exp(np.log(y).mean())
    linhas = []
    for lam in lambdas:
        if abs(lam) < 1e-9:
            z = media_geom * np.log(y)
        else:
            z = (y ** lam - 1) / (lam * media_geom ** (lam - 1))
        sqr = smf.ols("z ~ " + lado_direito, data=dados.assign(z=z)).fit().ssr
        linhas.append({"lambda": lam, "logver": -len(y) / 2 * np.log(sqr / len(y))})
    return pd.DataFrame(linhas)


# O λ depende dos preditores: o perfil é calculado para o modelo completo (Parte 5)
# e para o modelo com logDIS, logLSTAT, RM² e logLSTAT² (modelo E da Seção 7.4)
lados_direitos = {
    "Modelo completo (X originais)": formula.split("~")[1],
    "Modelo E (X transformados)": base_rm.split("~")[1] + " + RM_c + I(RM_c**2) + I(logLSTAT_c**2)",
}
lambdas = np.round(np.arange(-0.6, 1.0001, 0.01), 2)
corte_chi2 = stats.chi2.ppf(0.95, 1) / 2
lambdas_otimos = {}

fig = make_subplots(rows=1, cols=2, subplot_titles=[f"Box-Cox de MEDV: {nome}" for nome in lados_direitos])
for col, (nome, lado) in enumerate(lados_direitos.items(), start=1):
    perfil = perfil_boxcox(df_t, lado, lambdas)
    lambda_otimo = perfil.loc[perfil["logver"].idxmax(), "lambda"]
    lambdas_otimos[nome] = lambda_otimo
    corte_ic = perfil["logver"].max() - corte_chi2
    ic_lambda = perfil.loc[perfil["logver"] >= corte_ic, "lambda"]
    print(f"{nome}: λ ótimo = {lambda_otimo:.2f} | IC 95%: [{ic_lambda.min():.2f}; {ic_lambda.max():.2f}]"
          f" | log no IC: {ic_lambda.min() <= 0 <= ic_lambda.max()}"
          f" | raiz no IC: {ic_lambda.min() <= 0.5 <= ic_lambda.max()}"
          f" | sem transformação no IC: {ic_lambda.min() <= 1 <= ic_lambda.max()}")

    fig.add_trace(go.Scatter(x=perfil["lambda"], y=perfil["logver"], mode="lines", name="Log-verossimilhança",
                             line=dict(color=COR_PONTO, width=2.5),
                             hovertemplate="λ = %{x:.2f}<br>log-verossimilhança = %{y:.2f}<extra></extra>"),
                  row=1, col=col)
    y_min, y_max = faixa(perfil["logver"])
    linha_ref(fig, corte_ic, lambdas.min(), lambdas.max(), "Limite do IC 95%", row=1, col=col)
    for lam, rotulo in [(0, "log"), (0.5, "raiz"), (1, "sem transformação")]:
        linha_ref(fig, lam, y_min, y_max, f"λ = {lam} ({rotulo})", cor=COR_CRITICO, vertical=True, traco="dot",
                  row=1, col=col)
    linha_ref(fig, lambda_otimo, y_min, y_max, f"λ ótimo ({nome.split(' (')[0]})", cor="#008300", vertical=True,
              traco="solid", row=1, col=col)
    fig.update_xaxes(title_text="λ", row=1, col=col)
fig.update_yaxes(title_text="Log-verossimilhança perfilada", row=1, col=1)

print(f"λ ótimo só da distribuição de MEDV (sem preditores): {stats.boxcox(df_t['MEDV'])[1]:.2f}")

mostrar_e_salvar(fig, "boxcox_perfil_lambda", altura=500)

Modelo completo (X originais): λ ótimo = 0.13 | IC 95%: [0.02; 0.24] | log no IC: False | raiz no IC: False | sem transformação no IC: False


Modelo E (X transformados): λ ótimo = 0.41 | IC 95%: [0.29; 0.53] | log no IC: False | raiz no IC: True | sem transformação no IC: False
λ ótimo só da distribuição de MEDV (sem preditores): 0.22


### 7.3 Box-Cox nos preditores

Nos preditores, o $\lambda$ de Box-Cox serve como **guia** para a escolha da transformação que deixa cada variável mais simétrica. Isso ajuda a **linearidade** da relação com a resposta e reduz a alavanca dos valores extremos, mas não é um pressuposto do modelo: a RLM não exige preditores normais. Para variáveis com zeros (`ZN`) é usada a transformação de **Yeo-Johnson**, uma extensão de Box-Cox que aceita zero e valores negativos.

In [63]:
linhas = []
for v in ["CRIM", "ZN", "INDUS", "NOX", "AGE", "DIS", "TAX", "PTRATIO", "B", "LSTAT"]:
    x = df_t[v]
    if (x > 0).all():
        transformado, lam = stats.boxcox(x)
        metodo = "Box-Cox"
    else:
        transformado, lam = stats.yeojohnson(x)
        metodo = "Yeo-Johnson"
    linhas.append({
        "Variável": v,
        "Método": metodo,
        "λ": lam,
        "Assimetria original": stats.skew(x),
        "Assimetria transformada": stats.skew(transformado),
        "Assimetria com log": stats.skew(np.log1p(x) if (x == 0).any() else np.log(x)),
    })

pd.DataFrame(linhas).set_index("Variável").round(2)

,Método,λ,Assimetria original,Assimetria transformada,Assimetria com log
Variável,,,,,
CRIM,Box-Cox,-0.11,5.18,0.09,0.39
ZN,Yeo-Johnson,-0.91,2.21,1.08,1.20
INDUS,Box-Cox,0.44,0.29,-0.14,-0.65
NOX,Box-Cox,-0.90,0.72,0.06,0.35
AGE,Box-Cox,1.34,-0.59,-0.42,-1.66
DIS,Box-Cox,-0.16,1.02,0.03,0.16
TAX,Box-Cox,-0.51,0.66,0.08,0.32
PTRATIO,Box-Cox,4.36,-0.80,-0.22,-1.01
B,Box-Cox,3.63,-2.86,-1.92,-4.79


### 7.4 Comparação das especificações

Cada especificação acrescenta uma transformação à anterior. Todas usam os mesmos 501 dados, então R², AIC e testes são comparáveis (o AIC só entre modelos com a **mesma resposta**: o AIC de `MEDV` está em outra escala).

| Modelo | Resposta | Preditores |
| --- | --- | --- |
| A | `MEDV` | os 13 da Parte 5, na escala original |
| B | `logMEDV` | os mesmos de A |
| C | `logMEDV` | `logDIS`, `logLSTAT` e sem `TAX` (modelo linear da Parte 6) |
| D | `logMEDV` | C com `RM` centralizado + `RM²` (modelo da Parte 6) |
| E | `logMEDV` | D com `logLSTAT` centralizado + `logLSTAT²` (sugerido na conclusão da Parte 6) |

In [64]:
especificacoes_t = {
    "A: MEDV": formula,
    "B: logMEDV": formula.replace("MEDV ~", "logMEDV ~"),
    "C: + log(DIS, LSTAT)": base_rm + " + RM",
    "D: + RM² centralizado": base_rm + " + RM_c + I(RM_c**2)",
    "E: + logLSTAT² centralizado": base_rm + " + RM_c + I(RM_c**2) + I(logLSTAT_c**2)",
}
q_t = stats.norm.ppf((np.arange(1, len(df_t) + 1) - 0.375) / (len(df_t) + 0.25))


def diagnosticar(m):
    rs = m.get_influence().resid_studentized_external
    return {
        "R² ajustado": m.rsquared_adj,
        "AIC": m.aic,
        "Assimetria": stats.skew(m.resid),
        "Curtose (excesso)": stats.kurtosis(m.resid),
        "Correlação QQ": np.corrcoef(np.sort(rs), q_t)[0, 1],
        "Shapiro-Wilk p": stats.shapiro(rs).pvalue,
        "|t| > 3": int((np.abs(rs) > 3).sum()),
        "Shapiro-Wilk p sem |t| > 3": stats.shapiro(rs[np.abs(rs) <= 3]).pvalue,
        "Breusch-Pagan p": het_breuschpagan(m.resid, m.model.exog)[1],
        "RESET p": linear_reset(m, power=2, use_f=True).pvalue,
        "Durbin-Watson": durbin_watson(m.resid),
    }


modelos_t = {nome: smf.ols(f, data=df_t).fit() for nome, f in especificacoes_t.items()}
pd.DataFrame({nome: diagnosticar(m) for nome, m in modelos_t.items()}).map(lambda v: f"{v:.3g}")

,A: MEDV,B: logMEDV,"C: + log(DIS, LSTAT)",D: + RM² centralizado,E: + logLSTAT² centralizado
R² ajustado,0.74,0.787,0.795,0.802,0.812
AIC,2.99e+03,-225,-246,-262,-285
Assimetria,1.55,0.476,-0.162,-0.206,-0.161
Curtose (excesso),5.54,3.27,2.85,3.48,3.52
Correlação QQ,0.943,0.972,0.976,0.969,0.969
Shapiro-Wilk p,3.16e-18,1.82e-12,3.9e-11,4.16e-13,4.28e-13
|t| > 3,7,12,11,13,12
Shapiro-Wilk p sem |t| > 3,5.83e-11,0.000716,0.00418,0.00195,0.00184
Breusch-Pagan p,1.32e-07,1.47e-08,3.89e-12,1.41e-11,6.39e-10
RESET p,8.11e-37,1.58e-07,0.288,0.000243,0.138


No modelo E, o λ ótimo de Box-Cox (Seção 7.2) fica perto de 0,5 (raiz) e não de 0 (log). O modelo E é reajustado com `log(MEDV)`, com `raiz(MEDV)` e com o λ ótimo, para ver se essa diferença muda o diagnóstico dos resíduos. O R² e o AIC **não** são comparáveis entre respostas em escalas diferentes; a comparação é feita pelos resíduos:

In [65]:
lambda_e = lambdas_otimos["Modelo E (X transformados)"]
lado_e = especificacoes_t["E: + logLSTAT² centralizado"].split("~")[1]
respostas = {
    "log(MEDV)": np.log(df_t["MEDV"]),
    "raiz(MEDV)": np.sqrt(df_t["MEDV"]),
    f"Box-Cox λ = {lambda_e:.2f}": (df_t["MEDV"] ** lambda_e - 1) / lambda_e,
}
pd.DataFrame({
    nome: diagnosticar(smf.ols("resposta ~" + lado_e, data=df_t.assign(resposta=y)).fit())
    for nome, y in respostas.items()
}).drop(index=["R² ajustado", "AIC"]).map(lambda v: f"{v:.3g}")

,log(MEDV),raiz(MEDV),Box-Cox λ = 0.41
Assimetria,-0.161,0.365,0.281
Curtose (excesso),3.52,5.16,4.6
Correlação QQ,0.969,0.964,0.967
Shapiro-Wilk p,4.28e-13,5.53e-14,2.51e-13
|t| > 3,12,6,7
Shapiro-Wilk p sem |t| > 3,0.00184,0.00508,0.00336
Breusch-Pagan p,6.39e-10,6.65e-08,7.15e-08
RESET p,0.138,0.0884,0.257
Durbin-Watson,1.15,1.19,1.18


In [66]:

# Gráfico QQ com envelope simulado: modelo A (escala original) x modelo E (transformado)
rng_t = np.random.default_rng(42)
sim_t = np.sort(rng_t.standard_normal((100, len(df_t))), axis=1)
inf_t, sup_t = np.percentile(sim_t, 2.5, axis=0), np.percentile(sim_t, 97.5, axis=0)

paineis_109 = []
for nome in ["A: MEDV", "E: + logLSTAT² centralizado"]:
    m = modelos_t[nome]
    rs = m.get_influence().resid_studentized_external
    tracos, fora = tracos_qq(rs, m.fittedvalues.index, q_t, inf_t, sup_t)
    paineis_109.append((f"{nome}<br>correlação QQ = {np.corrcoef(np.sort(rs), q_t)[0, 1]:.3f} | fora do envelope: {fora}",
                        tracos))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, subplot_titles=[titulo for titulo, _ in paineis_109])
for col, (_, tracos) in enumerate(paineis_109, start=1):
    for traco in tracos:
        fig.add_trace(traco, row=1, col=col)
    fig.update_xaxes(title_text="Quantil teórico N(0, 1)", row=1, col=col)
fig.update_yaxes(title_text="Resíduo studentizado", row=1, col=1)
mostrar_e_salvar(fig, "qqplot_transformacoes", altura=550)

In [67]:
# Fonte da heterocedasticidade no modelo E: regressão de e² sobre cada termo (Breusch-Pagan de Koenker por termo)
modelo_e = modelos_t["E: + logLSTAT² centralizado"]
aux_bp = smf.ols("e2 ~ " + especificacoes_t["E: + logLSTAT² centralizado"].split("~")[1],
                 data=df_t.assign(e2=modelo_e.resid ** 2)).fit()
termos_bp = aux_bp.wald_test_terms(scalar=True).table
termos_bp.drop(index="Intercept").sort_values("pvalue").head(5).round(4)

,statistic,pvalue,df_constraint,df_denom
logDIS,16.1435,0.0001,1,479.0
NOX,4.3235,0.0381,1,479.0
AGE,3.6078,0.0581,1,479.0
C(CHAS),3.3188,0.0691,1,479.0
logLSTAT,3.2161,0.0735,1,479.0


### 7.5 Erros padrão robustos (HC3)

Como a variância dos erros não é constante (Breusch-Pagan rejeita em todas as especificações), os erros padrão "clássicos" do MQO não são confiáveis. Os **erros padrão robustos à heterocedasticidade** (estimador sanduíche de White, versão **HC3**, recomendada para amostras finitas) não mudam os coeficientes, só os erros padrão, os p-valores e os intervalos de confiança.

O modelo E é reajustado com `cov_type="HC3"`. Os testes de cada termo usam o teste de Wald (com `RAD` testado como bloco de 8 indicadoras, como na ANOVA tipo II).

In [68]:
modelo_e_hc3 = smf.ols(especificacoes_t["E: + logLSTAT² centralizado"], data=df_t).fit(cov_type="HC3")

wald_classico = modelo_e.wald_test_terms(scalar=True).table["pvalue"]
wald_hc3 = modelo_e_hc3.wald_test_terms(scalar=True).table["pvalue"]

comparacao_hc3 = pd.DataFrame({"p clássico": wald_classico, "p HC3": wald_hc3}).drop(index="Intercept")
comparacao_hc3["Significativo (clássico)"] = np.where(comparacao_hc3["p clássico"] < 0.05, "sim", "não")
comparacao_hc3["Significativo (HC3)"] = np.where(comparacao_hc3["p HC3"] < 0.05, "sim", "não")
comparacao_hc3["Muda?"] = np.where(
    comparacao_hc3["Significativo (clássico)"] != comparacao_hc3["Significativo (HC3)"], "SIM", "")

print(f"Razão média entre os erros padrão HC3 e clássicos: {(modelo_e_hc3.bse / modelo_e.bse).mean():.2f}")
comparacao_hc3.round(4)

Razão média entre os erros padrão HC3 e clássicos: 1.23


,p clássico,p HC3,Significativo (clássico),Significativo (HC3),Muda?
C(CHAS),0.0009,0.0125,sim,sim,
C(RAD),0.0020,0.0009,sim,sim,
CRIM,0.0000,0.0001,sim,sim,
ZN,0.8727,0.8373,não,não,
INDUS,0.7492,0.7577,não,não,
NOX,0.0000,0.0000,sim,sim,
AGE,0.6049,0.6867,não,não,
logDIS,0.0000,0.0000,sim,sim,
PTRATIO,0.0000,0.0000,sim,sim,
B,0.0007,0.0179,sim,sim,


### Conclusão — Transformações, centralização e pressupostos

**Centralização e padronização.** Com todos os preditores numéricos padronizados, os resíduos são **idênticos** aos do modelo original, e também o R², o VIF e os p-valores de Shapiro-Wilk, Breusch-Pagan e RESET. Mudam apenas o intercepto (35,6 → 18,3) e a escala dos coeficientes (o de `RM` passa de 3,65 por cômodo para 2,57 por desvio padrão). Assim, **centralizar não ajuda a passar nos testes dos resíduos**. Ela é usada para outra finalidade: antes de criar potências e produtos (`RM²`, `logLSTAT²`), para reduzir a colinearidade com o termo linear e manter a leitura dos coeficientes no ponto médio dos dados.

**Box-Cox na resposta.** É a transformação que mais melhora os resíduos. Passar de `MEDV` para `logMEDV` (modelo A → B) reduz a assimetria dos resíduos de 1,55 para 0,48, a curtose de 5,5 para 3,3, eleva a correlação do gráfico QQ de 0,943 para 0,972 e o R² ajustado de 0,740 para 0,787. Sem os resíduos com $|t| > 3$, o p-valor de Shapiro-Wilk sobe de 10⁻¹¹ para 7 × 10⁻⁴. O valor de $\lambda$, porém, **depende dos preditores**: com os preditores originais, $\hat\lambda = 0{,}13$ (IC 95% de 0,02 a 0,24); com os preditores transformados do modelo E, $\hat\lambda = 0{,}41$ (IC de 0,29 a 0,53). Nos dois casos $\lambda = 1$ (sem transformação) é rejeitado. No modelo E, `raiz(MEDV)` e o $\lambda$ ótimo trocam um problema por outro: têm menos resíduos com $|t| > 3$ (6 e 7 contra 12) e um Breusch-Pagan um pouco menos extremo, mas a assimetria (0,37 e 0,28 contra −0,16), a curtose (5,2 e 4,6 contra 3,5) e a correlação QQ (0,964 e 0,967 contra 0,969) ficam piores. Nenhuma das respostas passa em Shapiro-Wilk ou em Breusch-Pagan. Como o ganho não é claro, **`log(MEDV)` é mantido**, pela interpretação simples (efeitos em %) e pela coerência com a Parte 6.

**Box-Cox nos preditores.** Os $\lambda$ de `CRIM` (−0,11), `DIS` (−0,16) e `LSTAT` (0,22) estão perto de 0, o que confirma o log para essas variáveis. `NOX` (−0,90) e `TAX` (−0,51) ficariam mais simétricos com $1/X$ ou $1/\sqrt X$, mas já têm assimetria moderada (0,72 e 0,66) e não mostraram curvatura na Parte 3. Para `B` (3,63), `PTRATIO` (4,36) e `AGE` (1,34), o $\lambda$ é extremo ou perto de 1, e a transformação não tem interpretação útil: elas ficam na escala original. `ZN` continua assimétrica mesmo com Yeo-Johnson (2,21 → 1,08), por causa da concentração em zero; a alternativa é uma indicadora `ZN > 0` (Parte 3).

**Comparação das especificações.** Cada passo melhora a linearidade:

- o log nos preditores (C) leva o RESET de p = 10⁻⁷ para p = 0,29 e o R² ajustado para 0,795;
- `RM²` centralizado (D) melhora o ajuste (AIC −246 → −262), mas faz o RESET rejeitar de novo (p = 0,0002), como explicado na Parte 6;
- `logLSTAT²` centralizado (E) corrige essa curvatura: o RESET volta a não rejeitar (p = 0,14), o AIC cai para −285 e o R² ajustado sobe para **0,812**.

**O que as transformações não resolvem.**

- **Normalidade.** Shapiro-Wilk rejeita em todas as especificações. A causa são as caudas: há cerca de 12 resíduos com $|t| > 3$ (1,4 esperados) e curtose de 3,5. Sem eles, os resíduos são aproximadamente normais (correlação QQ de 0,969). Com n = 501, o teste rejeita desvios pequenos, e os testes t e F continuam aproximadamente válidos pelo Teorema Central do Limite. A evidência principal é o gráfico QQ com envelope.
- **Homocedasticidade.** Breusch-Pagan rejeita em **todas** as especificações (p entre 10⁻¹² e 10⁻⁷). A variância dos erros muda principalmente com `logDIS` (p = 0,0001 na regressão auxiliar de $e^2$). A correção adequada é usar **erros padrão robustos (HC3)**: no modelo E, eles são em média 23% maiores que os clássicos, mas **nenhuma variável muda de conclusão** a 5%. `CHAS` (p de 0,001 para 0,013), `B` (0,0007 → 0,018) e `RM` (p < 10⁻⁴ → 0,009) perdem força, mas continuam significativas. `ZN`, `INDUS` e `AGE` não são significativas nos dois casos.
- **Independência.** O Durbin-Watson fica entre 1,09 e 1,19 em todas as especificações. O arquivo está ordenado por região, e regiões vizinhas têm erros parecidos (**correlação espacial**). Nenhuma transformação de variáveis corrige isso, pois o problema está na ordem e na vizinhança das observações. Esse pressuposto deve ser apresentado como uma **limitação** do modelo; os p-valores devem ser lidos como aproximados.

**Em síntese**, o modelo recomendado usa `log(MEDV)` como resposta, `logDIS` e `logLSTAT` como preditores, `RM` e `logLSTAT` **centralizados** antes de criar os quadrados, e **erros padrão HC3** para a inferência. Com isso, a linearidade é aceita (RESET p = 0,14), a assimetria dos resíduos é eliminada e as conclusões sobre a significância não dependem da heterocedasticidade. Normalidade estrita e independência continuam rejeitadas e são limitações conhecidas do conjunto de dados (caudas pesadas, censura em `MEDV = 50` e correlação espacial).


## **Parte 8: Modelo final**

As Partes 5 a 7 definiram cada peça do modelo em separado: a remoção das observações influentes (Parte 5, continuação), o termo `RM²` centralizado (Parte 6) e as transformações com erros padrão HC3 (Parte 7). Nesta parte essas peças são combinadas em um **único modelo final**, que é então:

- simplificado (variáveis não significativas);
- diagnosticado de novo (pontos influentes, resíduos, independência);
- interpretado (coeficientes com IC 95% robusto e efeitos em %);
- avaliado na **capacidade preditiva** por validação cruzada.

Todas as etapas usam os mesmos 501 dados da Parte 7 (`df_t`: sem as linhas suspeitas 506 a 510 e sem `RM` ausente).

### 8.1 Seleção de variáveis

No modelo E com HC3 (Seção 7.5), `ZN`, `INDUS` e `AGE` não são significativas. Como os testes individuais podem esconder um efeito conjunto, as três são testadas **juntas** pelo teste de Wald com a matriz de covariância HC3 ($H_0: \beta_{ZN} = \beta_{INDUS} = \beta_{AGE} = 0$). O modelo sem elas também é comparado pelo AIC e pelo BIC (menor é melhor; o BIC penaliza mais os parâmetros extras).

In [69]:
formula_e = especificacoes_t["E: + logLSTAT² centralizado"]
formula_final = formula_e.replace(" + ZN + INDUS", "").replace(" + AGE", "")
print(f"Modelo final: {formula_final}")

wald_reducao = modelo_e_hc3.wald_test("ZN = 0, INDUS = 0, AGE = 0", scalar=True)
print(f"Wald conjunto (HC3), ZN = INDUS = AGE = 0: χ² = {wald_reducao.statistic:.2f} (3 GL) | p-valor = {wald_reducao.pvalue:.3f}")

modelo_reduzido = smf.ols(formula_final, data=df_t).fit()
pd.DataFrame({
    nome: {"Parâmetros": int(m.df_model) + 1, "R² ajustado": m.rsquared_adj, "AIC": m.aic, "BIC": m.bic,
           "RESET p": linear_reset(m, power=2, use_f=True).pvalue}
    for nome, m in [("Modelo E", modelo_e), ("Sem ZN, INDUS e AGE", modelo_reduzido)]
}).round(4)

Modelo final: logMEDV ~ CRIM + C(CHAS) + NOX + logDIS + C(RAD) + PTRATIO + B + logLSTAT + RM_c + I(RM_c**2) + I(logLSTAT_c**2)
Wald conjunto (HC3), ZN = INDUS = AGE = 0: χ² = 0.32 (3 GL) | p-valor = 0.955


,Modelo E,"Sem ZN, INDUS e AGE"
Parâmetros,22.0000,19.0000
R² ajustado,0.8115,0.8125
AIC,-285.1925,-290.7709
BIC,-192.4272,-210.6554
RESET p,0.1378,0.1406


### 8.2 Observações influentes no modelo final

As observações influentes da Parte 5 foram encontradas no modelo **linear em `MEDV`**. Com a resposta em log e os novos termos, outras linhas podem ser influentes. Por isso, os mesmos critérios (Cook > 4/n ou |DFFITS| > 2·√(p/n)) são aplicados de novo ao modelo reduzido, e o modelo é reajustado sem essas linhas, como no S+I da Parte 5 (as linhas suspeitas S já estão fora de `df_t`).

In [70]:
def linhas_influentes(m):
    infl = m.get_influence()
    n_m, p_m = int(m.nobs), int(m.df_model) + 1
    influente = (infl.cooks_distance[0] > 4 / n_m) | (np.abs(infl.dffits[0]) > 2 * np.sqrt(p_m / n_m))
    return m.model.data.frame.index[influente]


def resumir_residuos(m):
    rs = m.get_influence().resid_studentized_external
    n_m = int(m.nobs)
    q = stats.norm.ppf((np.arange(1, n_m + 1) - 0.375) / (n_m + 0.25))
    return {
        "n": n_m,
        "R² ajustado": m.rsquared_adj,
        "Erro padrão residual": np.sqrt(m.mse_resid),
        "Assimetria": stats.skew(m.resid),
        "Curtose (excesso)": stats.kurtosis(m.resid),
        "Correlação QQ": np.corrcoef(np.sort(rs), q)[0, 1],
        "Shapiro-Wilk p": stats.shapiro(rs).pvalue,
        "|t| > 3": int((np.abs(rs) > 3).sum()),
        "|t| > 3 esperado": n_m * 2 * stats.norm.sf(3),
        "Outliers Bonferroni": int((m.outlier_test(method="bonf")["bonf(p)"] < 0.05).sum()),
        "Breusch-Pagan p": het_breuschpagan(m.resid, m.model.exog)[1],
        "RESET p": linear_reset(m, power=2, use_f=True).pvalue,
        "Durbin-Watson": durbin_watson(m.resid),
    }


influentes_final = linhas_influentes(modelo_reduzido)
df_final = df_t.drop(index=influentes_final)
modelo_final_classico = smf.ols(formula_final, data=df_final).fit()
modelo_final = smf.ols(formula_final, data=df_final).fit(cov_type="HC3")

print(f"Influentes no modelo reduzido: {len(influentes_final)} linhas ({len(influentes_final) / len(df_t):.1%} de n)")
print(f"Em comum com as influentes da Parte 5: {len(set(influentes_final) & sel_grupos['I'])} de {len(sel_grupos['I'])}")
print(f"Com MEDV = 50 (censura): {(df_t.loc[influentes_final, 'MEDV'] == 50).sum()} de {(df_t['MEDV'] == 50).sum()}")
print(f"Maior distância de Cook: {modelo_reduzido.get_influence().cooks_distance[0].max():.3f}")

# VIF dos termos que não são de RAD (as indicadoras de RAD formam um bloco, avaliado pelo GVIF na Parte 4)
exog_final = modelo_final_classico.model.exog
vif_final = pd.Series({nome: variance_inflation_factor(exog_final, i)
                       for i, nome in enumerate(modelo_final_classico.model.exog_names)
                       if nome != "Intercept" and "RAD" not in nome})
print(f"Maior VIF no modelo final: {vif_final.max():.2f} ({vif_final.idxmax()})")

pd.DataFrame({
    "Com as influentes": resumir_residuos(modelo_reduzido),
    "Sem as influentes (final)": resumir_residuos(modelo_final_classico),
}).map(lambda v: f"{v:.3g}")

Influentes no modelo reduzido: 28 linhas (5.6% de n)
Em comum com as influentes da Parte 5: 10 de 24
Com MEDV = 50 (censura): 5 de 16
Maior distância de Cook: 0.225
Maior VIF no modelo final: 4.81 (NOX)


,Com as influentes,Sem as influentes (final)
n,501,473
R² ajustado,0.813,0.881
Erro padrão residual,0.178,0.129
Assimetria,-0.176,0.0196
Curtose (excesso),3.57,1.28
Correlação QQ,0.968,0.993
Shapiro-Wilk p,3.24e-13,0.00115
|t| > 3,12,5
|t| > 3 esperado,1.35,1.28
Outliers Bonferroni,5,1


### 8.3 Coeficientes do modelo final

Os erros padrão, os IC 95% e os p-valores usam **HC3**. Como a resposta é `log(MEDV)`, os coeficientes são lidos em % de `MEDV`:

- **preditor na escala original** (e indicadoras): um aumento de uma unidade (ou o nível em relação à referência) muda `MEDV` em $100 \cdot (e^{\beta} - 1)\%$;
- **preditor em log** (`logDIS`, `logLSTAT`): $\beta$ é a **elasticidade**: um aumento de 1% no preditor muda `MEDV` em cerca de $\beta\%$. Para `logLSTAT`, que tem o termo quadrático centralizado, esse valor vale no ponto médio de `logLSTAT`;
- **`RM_c`**: efeito de um cômodo a mais **na casa média** (Parte 6). Os termos quadráticos não têm leitura isolada.

In [71]:
ic_final = modelo_final.conf_int()
tabela_final = pd.DataFrame({
    "Coeficiente": modelo_final.params,
    "Erro padrão HC3": modelo_final.bse,
    "IC 95% inf": ic_final[0],
    "IC 95% sup": ic_final[1],
    "p-valor": modelo_final.pvalues,
    "Efeito em MEDV (%)": 100 * (np.exp(modelo_final.params) - 1),
})
for termo in ["logDIS", "logLSTAT"]:
    tabela_final.loc[termo, "Efeito em MEDV (%)"] = modelo_final.params[termo]
tabela_final.loc[["Intercept", "I(RM_c ** 2)", "I(logLSTAT_c ** 2)"], "Efeito em MEDV (%)"] = np.nan
tabela_final["Significativo (5%)"] = np.where(tabela_final["p-valor"] < 0.05, "sim", "não")

print(f"n = {int(modelo_final.nobs)} | R² = {modelo_final.rsquared:.4f} | R² ajustado = {modelo_final.rsquared_adj:.4f}")
print(f"F global (HC3) = {modelo_final.fvalue:.1f} | p-valor = {modelo_final.f_pvalue:.3g}")

media_rm_final = df_t["RM"].mean()
b1_final = modelo_final.params["RM_c"]
b2_final = modelo_final.params["I(RM_c ** 2)"]
print("Efeito de um cômodo a mais (%): " + " | ".join(
    f"RM = {rm}: {100 * (np.exp(b1_final + 2 * b2_final * (rm - media_rm_final)) - 1):+.1f}"
    for rm in [5, 6, 7, 8]))

print("\nWald por termo (HC3), RAD como bloco de 8 indicadoras:")
print(modelo_final.wald_test_terms(scalar=True).table["pvalue"].drop("Intercept").map(lambda v: f"{v:.2g}").to_string())

tabela_final.round(4)

n = 473 | R² = 0.8854 | R² ajustado = 0.8809
F global (HC3) = 184.1 | p-valor = 1.71e-195
Efeito de um cômodo a mais (%): RM = 5: -5.8 | RM = 6: +7.9 | RM = 7: +23.6 | RM = 8: +41.7

Wald por termo (HC3), RAD como bloco de 8 indicadoras:
C(CHAS)               5.9e-05
C(RAD)                9.3e-06
CRIM                  1.2e-07
NOX                   1.4e-08
logDIS                1.1e-08
PTRATIO               1.8e-20
B                     4.2e-05
logLSTAT              7.8e-50
RM_c                  3.1e-15
I(RM_c ** 2)          1.7e-16
I(logLSTAT_c ** 2)    3.5e-06


,Coeficiente,Erro padrão HC3,IC 95% inf,IC 95% sup,p-valor,Efeito em MEDV (%),Significativo (5%)
Intercept,4.5968,0.1403,4.3219,4.8718,0.0000,NaN,sim
C(CHAS)[T.1],0.0913,0.0227,0.0467,0.1358,0.0001,9.5558,sim
C(RAD)[T.2],0.0728,0.0352,0.0038,0.1418,0.0385,7.5524,sim
C(RAD)[T.3],0.1523,0.0333,0.0870,0.2177,0.0000,16.4548,sim
C(RAD)[T.4],0.0659,0.0284,0.0102,0.1217,0.0204,6.8144,sim
C(RAD)[T.5],0.0955,0.0292,0.0383,0.1526,0.0011,10.0170,sim
C(RAD)[T.6],0.0489,0.0351,-0.0200,0.1178,0.1641,5.0102,não
C(RAD)[T.7],0.1254,0.0347,0.0575,0.1933,0.0003,13.3614,sim
C(RAD)[T.8],0.0828,0.0345,0.0152,0.1503,0.0163,8.6309,sim
C(RAD)[T.24],0.1639,0.0425,0.0807,0.2471,0.0001,17.8106,sim


### 8.4 Independência: ordem do arquivo e agrupamento por cidade

O Durbin-Watson baixo (Seção 7.5) foi atribuído à ordem do arquivo, que agrupa regiões vizinhas. Duas verificações:

1. **Ordem aleatória.** Se a dependência vem da ordem das linhas (e não de um erro no modelo), o Durbin-Watson calculado com os resíduos em ordem aleatória deve ficar perto de 2. São usadas 1000 permutações.
2. **Erros padrão agrupados por cidade.** No Boston Housing, as regiões da mesma cidade têm valores idênticos de `TAX`, `PTRATIO`, `INDUS` e `ZN`, que são medidos por cidade. Essa combinação identifica, de forma aproximada, a cidade de cada linha. Os erros padrão **agrupados** (`cov_type="cluster"`) permitem que os erros sejam correlacionados dentro de cada grupo. Eles não corrigem os coeficientes, mas mostram se a inferência resiste à dependência entre vizinhos.

In [72]:
rng_dw = np.random.default_rng(42)
residuos_final = modelo_final.resid.values
dw_permutado = np.array([durbin_watson(rng_dw.permutation(residuos_final)) for _ in range(1000)])
print(f"Durbin-Watson na ordem do arquivo: {durbin_watson(residuos_final):.3f}")
print(f"Durbin-Watson em ordem aleatória: média = {dw_permutado.mean():.3f} | "
      f"faixa de 95% = [{np.percentile(dw_permutado, 2.5):.3f}; {np.percentile(dw_permutado, 97.5):.3f}]")

cidade = df_final.groupby(["TAX", "PTRATIO", "INDUS", "ZN"]).ngroup()
tamanho_grupos = cidade.value_counts()
print(f"\nGrupos (cidades aproximadas): {cidade.nunique()} | mediana de {tamanho_grupos.median():.0f} linhas por grupo"
      f" | maior grupo: {tamanho_grupos.max()} linhas")

modelo_final_cluster = smf.ols(formula_final, data=df_final).fit(cov_type="cluster", cov_kwds={"groups": cidade})
comparacao_cluster = pd.DataFrame({
    "p HC3": modelo_final.wald_test_terms(scalar=True).table["pvalue"],
    "p agrupado": modelo_final_cluster.wald_test_terms(scalar=True).table["pvalue"],
}).drop(index="Intercept")
comparacao_cluster["Significativo (HC3)"] = np.where(comparacao_cluster["p HC3"] < 0.05, "sim", "não")
comparacao_cluster["Significativo (agrupado)"] = np.where(comparacao_cluster["p agrupado"] < 0.05, "sim", "não")
# RAD é um bloco de 8 indicadoras e não tem uma razão única; CHAS tem uma indicadora
razao_ep = (modelo_final_cluster.bse / modelo_final.bse).rename(index={"C(CHAS)[T.1]": "C(CHAS)"})
comparacao_cluster["Razão dos erros padrão"] = razao_ep.reindex(comparacao_cluster.index)
comparacao_cluster.round(4)

Durbin-Watson na ordem do arquivo: 1.411
Durbin-Watson em ordem aleatória: média = 2.000 | faixa de 95% = [1.822; 2.179]

Grupos (cidades aproximadas): 78 | mediana de 3 linhas por grupo | maior grupo: 112 linhas


,p HC3,p agrupado,Significativo (HC3),Significativo (agrupado),Razão dos erros padrão
C(CHAS),0.0001,0.0330,sim,sim,1.8844
C(RAD),0.0000,0.0000,sim,sim,NaN
CRIM,0.0000,0.0000,sim,sim,0.4356
NOX,0.0000,0.0000,sim,sim,0.7695
logDIS,0.0000,0.0000,sim,sim,1.1170
PTRATIO,0.0000,0.0000,sim,sim,1.2723
B,0.0000,0.0000,sim,sim,0.5373
logLSTAT,0.0000,0.0000,sim,sim,2.2549
RM_c,0.0000,0.0000,sim,sim,1.5781
I(RM_c ** 2),0.0000,0.0000,sim,sim,1.1520


### 8.5 Diagnóstico gráfico do modelo final

Quatro gráficos resumem os pressupostos do modelo final:

- **gráfico QQ** dos resíduos studentizados com envelope simulado (normalidade);
- **resíduos x valores ajustados** com LOWESS (linearidade: a curva deve ficar horizontal em zero);
- **escala-locação**, $\sqrt{|r_i|}$ x valores ajustados (homocedasticidade: a curva deve ficar horizontal);
- **distância de Cook** por observação (influência: nenhum ponto deve dominar o ajuste).

In [73]:
infl_final = modelo_final_classico.get_influence()
n_final = int(modelo_final_classico.nobs)
ajustados_final = modelo_final_classico.fittedvalues
rs_final = np.sort(infl_final.resid_studentized_external)
q_final = stats.norm.ppf((np.arange(1, n_final + 1) - 0.375) / (n_final + 0.25))
sim_final = np.sort(np.random.default_rng(42).standard_normal((100, n_final)), axis=1)
inf_final, sup_final = np.percentile(sim_final, 2.5, axis=0), np.percentile(sim_final, 97.5, axis=0)

graficos_diagnostico(modelo_final_classico, df_final, "Modelo final", "diagnostico_modelo_final")

### 8.6 Capacidade preditiva (validação cruzada)

O R² mede o ajuste nos mesmos dados usados para estimar o modelo e, por isso, é otimista. A **validação cruzada com 10 partes** (`KFold`, com embaralhamento e semente 42) estima o erro em dados **não usados** no ajuste: o modelo é ajustado em 9 partes e prevê a parte restante, 10 vezes.

As previsões do modelo final estão em `log(MEDV)`. Como $E[MEDV] \neq e^{E[\log MEDV]}$, a volta para mil dólares usa o **fator de smearing de Duan**: $\widehat{MEDV} = e^{\hat y} \cdot \frac{1}{n}\sum e^{e_i}$, calculado só com os resíduos do treino.

São comparados:

- o **modelo completo linear** da Parte 5 (`MEDV` na escala original);
- o **modelo final** ajustado com todas as linhas do treino;
- o **modelo final sem as influentes**: em cada rodada, as influentes são encontradas e removidas **só no treino**. A parte de teste mantém todas as linhas, pois não é possível remover casas reais na hora de prever.

Todas as medidas estão em mil dólares, na escala de `MEDV`.

In [74]:
from sklearn.model_selection import KFold


def prever_log(m, dados):
    # Volta para a escala de MEDV com o fator de smearing de Duan
    return np.exp(m.predict(dados)) * np.mean(np.exp(m.resid))


validacao = {"Linear completo (Parte 5)": [], "Final (todas as linhas)": [], "Final (sem influentes no treino)": []}
for treino, teste in KFold(n_splits=10, shuffle=True, random_state=42).split(df_t):
    dados_treino, dados_teste = df_t.iloc[treino], df_t.iloc[teste]
    m_linear = smf.ols(formula, data=dados_treino).fit()
    m_log = smf.ols(formula_final, data=dados_treino).fit()
    m_log_sem = smf.ols(formula_final, data=dados_treino.drop(index=linhas_influentes(m_log))).fit()
    previsoes = [m_linear.predict(dados_teste), prever_log(m_log, dados_teste), prever_log(m_log_sem, dados_teste)]
    y_teste = dados_teste["MEDV"]
    for nome, previsto in zip(validacao, previsoes):
        erro = y_teste - previsto
        validacao[nome].append({
            "RMSE": np.sqrt(np.mean(erro ** 2)),
            "MAE": np.mean(np.abs(erro)),
            "R² fora da amostra": 1 - np.sum(erro ** 2) / np.sum((y_teste - y_teste.mean()) ** 2),
        })

resultado_validacao = pd.DataFrame({
    nome: {**pd.DataFrame(rodadas).mean().to_dict(), "Desvio padrão do RMSE": pd.DataFrame(rodadas)["RMSE"].std()}
    for nome, rodadas in validacao.items()
}).T
resultado_validacao.round(3)

,RMSE,MAE,R² fora da amostra,Desvio padrão do RMSE
Linear completo (Parte 5),4.802,3.385,0.705,0.909
Final (todas as linhas),3.996,2.753,0.788,1.193
Final (sem influentes no treino),4.072,2.661,0.778,1.312


### Conclusão - Modelo final

**Modelo final.**

$$
\log(MEDV) = \beta_0 + \beta_1 CRIM + \beta_2 CHAS + \beta_3 NOX + \beta_4 \log DIS + \textstyle\sum_k \gamma_k RAD_k + \beta_5 PTRATIO + \beta_6 B + \beta_7 \log LSTAT + \beta_8 RM_c + \beta_9 RM_c^2 + \beta_{10} (\log LSTAT_c)^2 + \varepsilon
$$

O modelo é ajustado em 473 linhas, com erros padrão HC3. `RM_c` e $\log LSTAT_c$ são centralizados na média dos 501 dados.

**Seleção de variáveis.** `ZN`, `INDUS` e `AGE` saem do modelo. O teste de Wald conjunto (HC3) não rejeita que os três coeficientes sejam zero (χ² = 0,32 com 3 GL; p = 0,96). O AIC (−285 → −291) e o BIC (−192 → −211) melhoram, e o R² ajustado fica igual (0,812 → 0,813). Com isso, `TAX`, `ZN`, `INDUS` e `AGE` ficam fora do modelo: `TAX` por colinearidade com `RAD` (Parte 4) e as outras três por não serem significativas.

**Observações influentes.** No modelo reduzido, 28 linhas (5,6% de n) são influentes. Só 10 delas também eram influentes no modelo linear da Parte 5, pois com a resposta em log as linhas influentes mudam. Nenhuma tem Cook > 1 (o máximo é 0,23), e 5 das 16 linhas com `MEDV = 50` estão entre elas. Sem essas linhas:

- os resíduos ficam quase normais: correlação QQ de 0,968 para **0,993**, curtose de 3,6 para 1,3, e 5 resíduos com $|t| > 3$ (1,3 esperados). O Shapiro-Wilk ainda rejeita (p = 0,001), mas o desvio é pequeno (18 de 473 pontos fora do envelope, 3,8%);
- o RESET deixa de indicar problema na forma funcional (p = **0,94**);
- o R² ajustado sobe para **0,881**, e o erro padrão residual cai de 0,178 para 0,129 em log, isto é, cerca de ±13% em `MEDV`.

**Coeficientes.** Com HC3, todos os termos são significativos a 5%: `RAD` como bloco (p ≈ 10⁻⁵), embora o nível 6 não seja diferente do nível 1 isoladamente. Mantidos os demais preditores:

- **`LSTAT`** é o efeito mais forte: +1% em `LSTAT` reduz `MEDV` em cerca de 0,31% (no ponto médio). O termo quadrático negativo mostra que o efeito fica mais forte nos extremos de `LSTAT`;
- **`RM`**: um cômodo a mais vale **+12%** na casa média, +8% com 6 cômodos, +24% com 7 e +42% com 8 (efeito convexo, Parte 6);
- **`NOX`**: +0,1 na concentração de óxidos de nitrogênio reduz `MEDV` em cerca de 6,4% ($e^{-0{,}066} - 1$). Este é o efeito que o estudo original (Harrison & Rubinfeld, 1978) procurava medir;
- **`CHAS`**: regiões às margens do rio Charles valem cerca de **9,6%** a mais;
- **`CRIM`**: cada ponto na taxa de criminalidade reduz `MEDV` em 1,7%;
- **`PTRATIO`**: um aluno a mais por professor reduz `MEDV` em 2,7%;
- **`DIS`**: +1% na distância aos centros de emprego reduz `MEDV` em 0,14%;
- **`RAD`**: todos os níveis valem mais que o nível 1 (de +5% a +18%);
- **`B`**: o efeito é positivo e pequeno.

**Independência.** Na ordem do arquivo, o Durbin-Watson é 1,41. Com os resíduos em ordem aleatória, ele fica em 2,00 (faixa de 95% de 1,82 a 2,18). Isso confirma que a dependência vem da **vizinhança entre as linhas**, e não da forma do modelo. Os 78 grupos com `TAX`, `PTRATIO`, `INDUS` e `ZN` iguais aproximam as cidades (o maior, com 112 linhas, é a cidade de Boston). Com erros padrão agrupados por esses grupos, **todos os termos continuam significativos**. Os erros padrão de `logLSTAT` (2,3 vezes), `logLSTAT²` (1,9) e `RM_c` (1,6) aumentam, e `CHAS` passa a p = 0,03. Assim, as conclusões resistem à correlação dentro das cidades. A correlação entre cidades vizinhas não pode ser verificada sem coordenadas e continua uma **limitação**.

**Pressupostos que ainda não são atendidos.** A variância não é constante (Breusch-Pagan p ≈ 10⁻¹¹): o gráfico escala-locação mostra resíduos maiores nas casas mais baratas (valor ajustado baixo). Por isso, a inferência usa HC3. A distância de Cook ainda se concentra nas linhas 380 a 420, que são regiões pobres do centro de Boston. Nenhum ponto domina o ajuste.

**Capacidade preditiva.** Na validação cruzada com 10 partes, o modelo final erra em média **4,0 mil dólares** (RMSE; MAE de 2,8), contra 4,8 mil (MAE de 3,4) do modelo linear completo da Parte 5. O R² fora da amostra sobe de 0,71 para **0,79**. A transformação log e os termos não lineares melhoram a **previsão**, e não só os testes. Remover as influentes do treino **não** melhora a previsão (RMSE de 4,07; MAE de 2,66). A remoção melhora a estimativa dos coeficientes para a população típica, mas as casas removidas continuam aparecendo nos dados novos. O R² dentro da amostra (0,885) é otimista em relação ao R² fora da amostra (0,79).

**Em síntese**, o modelo final explica cerca de 88% da variação de `log(MEDV)` nos dados usados no ajuste e cerca de 79% da variação de `MEDV` em dados novos. A forma funcional é adequada (RESET p = 0,94), e os resíduos são quase normais. A inferência usa HC3, e as conclusões também resistem aos erros padrão agrupados por cidade. As limitações são:

- a heterocedasticidade, tratada com HC3 e não eliminada;
- a correlação espacial entre cidades vizinhas;
- a censura em `MEDV = 50`, tratada só em parte (sem modelo Tobit);
- a remoção de 28 casas reais. Ela deve ser registrada no relatório e não deve ser usada para previsão.

## **Parte 9: Heterocedasticidade (MQGF)**

O modelo final da Parte 8 usa erros padrão **HC3**. O HC3 corrige os erros padrão, os intervalos de confiança e os testes quando a variância dos erros muda com $x$, mas **não remove** a heterocedasticidade (Breusch-Pagan p ≈ 10⁻¹¹ na Seção 8.2). Três problemas continuam:

- os coeficientes do MQO deixam de ser os mais **eficientes** (de menor variância);
- os **intervalos de previsão** supõem uma variância única para todas as casas;
- o **fator de smearing** de Duan (Seção 8.6) supõe que os erros têm a mesma distribuição para todo $x$.

Nesta parte, a variância é **modelada** e o modelo é ajustado por **mínimos quadrados generalizados factíveis (MQGF)**:

1. encontrar a forma da variância com uma regressão auxiliar;
2. ajustar o MQGF com pesos $w_i = 1/\hat\sigma^2_i$;
3. verificar se os resíduos ponderados têm variância constante;
4. comparar os coeficientes com os do MQO;
5. rever a volta à escala de `MEDV`;
6. comparar a previsão e a cobertura dos intervalos de previsão por validação cruzada.

### 9.1 Forma da variância

Se $\operatorname{Var}(\varepsilon_i \mid x_i) = \sigma^2 \exp(z_i'\delta)$, então $\log(e_i^2) \approx \text{constante} + z_i'\delta + \text{erro}$. Com os resíduos de `modelo_final_classico` (Seção 8.2, n = 473), três regressões auxiliares de $\log(e_i^2)$ são comparadas (Wooldridge, cap. 8):

- (a) $z$ = valor ajustado $\hat y$;
- (b) $z$ = `logLSTAT`;
- (c) $z$ = todos os preditores do modelo final.

A forma exponencial garante que a variância estimada seja positiva. O F parcial compara (c) com (b), pois (b) está contida em (c).

In [75]:
lado_direito_final = formula_final.split("~")[1].strip()
dados_aux = df_final.assign(ajustado=modelo_final_classico.fittedvalues,
                            log_e2=np.log(modelo_final_classico.resid ** 2))
formas_variancia = {
    "(a) valor ajustado": "ajustado",
    "(b) logLSTAT": "logLSTAT",
    "(c) todos os preditores": lado_direito_final,
}
aux_variancia = {nome: smf.ols(f"log_e2 ~ {forma}", data=dados_aux).fit() for nome, forma in formas_variancia.items()}

print(f"(a) log(e²) = {aux_variancia['(a) valor ajustado'].params['ajustado']:.3f} · ŷ + constante")
print(f"(b) log(e²) = {aux_variancia['(b) logLSTAT'].params['logLSTAT']:.3f} · logLSTAT + constante")
f_parcial = aux_variancia["(c) todos os preditores"].compare_f_test(aux_variancia["(b) logLSTAT"])
print(f"(c) contra (b), F parcial: F = {f_parcial[0]:.2f} ({f_parcial[2]:.0f} GL) | p-valor = {f_parcial[1]:.4f}")

pd.DataFrame({
    nome: {"GL": int(a.df_model), "R²": a.rsquared, "R² ajustado": a.rsquared_adj, "F": a.fvalue,
           "p-valor do F": a.f_pvalue, "BIC": a.bic}
    for nome, a in aux_variancia.items()
}).T.map(lambda v: f"{v:.4g}")

(a) log(e²) = -1.206 · ŷ + constante
(b) log(e²) = 0.819 · logLSTAT + constante
(c) contra (b), F parcial: F = 2.39 (17 GL) | p-valor = 0.0015


,GL,R²,R² ajustado,F,p-valor do F,BIC
(a) valor ajustado,1,0.03373,0.03168,16.44,5.875e-05,2125
(b) logLSTAT,1,0.04422,0.04219,21.79,3.973e-06,2120
(c) todos os preditores,18,0.1227,0.08792,3.528,1.722e-06,2184


### 9.2 Ajuste por MQGF

As três formas explicam uma parte pequena de $\log(e^2)$ (R² de 0,03 a 0,12), o que é comum: $\log(e_i^2)$ tem muito ruído mesmo quando a forma está certa. A forma **(b) `logLSTAT`** é a escolhida:

- com um só termo, explica mais que (a) (R² de 0,044 contra 0,034);
- tem o **menor BIC** das três. A forma (c) tem R² maior, e o F parcial rejeita (b) contra (c) (p = 0,002), mas ela usa 18 termos, a maioria não significativa. A Seção 9.6 mostra que (c) não melhora a previsão;
- o coeficiente de 0,82 indica que a variância cresce com `LSTAT` (aproximadamente $\sigma^2_i \propto LSTAT_i^{0{,}82}$): os erros são maiores nas regiões pobres, onde as casas são mais baratas. Isso é o que o gráfico escala-locação da Seção 8.5 mostra.

A função `ajustar_mqgf` faz as três etapas em qualquer conjunto de dados (também em cada treino da validação cruzada):

1. ajusta o MQO e calcula $\log(e_i^2)$;
2. ajusta a regressão auxiliar e calcula $\hat g_i$;
3. ajusta `smf.wls(formula_final, weights=w)` com $w_i = 1/\exp(\hat g_i)$.

A constante de $\hat g$ é viesada ($E[\log \chi^2_1] \approx -1{,}27$), mas isso não muda os coeficientes, pois os pesos só precisam ser proporcionais a $1/\sigma^2_i$. A variância na escala correta é $\hat\sigma^2(x) = s^2_{\text{MQGF}} \cdot \exp(\hat g(x))$ (função `variancia_log`). O MQGF também é ajustado com `cov_type="HC3"`, que protege a inferência caso a forma da variância esteja errada.

In [76]:
forma_variancia = formas_variancia["(b) logLSTAT"]


def ajustar_mqgf(dados, forma=forma_variancia, **kwargs_fit):
    m_ols = smf.ols(formula_final, data=dados).fit()
    dados_aux_m = dados.assign(ajustado=m_ols.fittedvalues, log_e2=np.log(m_ols.resid ** 2))
    aux = smf.ols(f"log_e2 ~ {forma}", data=dados_aux_m).fit()
    pesos = 1 / np.exp(aux.fittedvalues)
    m_wls = smf.wls(formula_final, data=dados, weights=pesos).fit(**kwargs_fit)
    return m_wls, aux, m_ols


def variancia_log(m_wls, aux, m_ols, dados):
    # Variância de log(MEDV) para cada linha de dados: s² do MQGF vezes exp(ĝ(x))
    return m_wls.scale * np.exp(aux.predict(dados.assign(ajustado=m_ols.predict(dados))))


modelo_mqgf, aux_mqgf, _ = ajustar_mqgf(df_final)
modelo_mqgf_hc3, _, _ = ajustar_mqgf(df_final, cov_type="HC3")
pesos_mqgf = modelo_mqgf.model.weights
print(f"Pesos: razão entre o maior e o menor = {pesos_mqgf.max() / pesos_mqgf.min():.1f}")
print(f"Desvio padrão estimado em log: de {np.sqrt(variancia_log(modelo_mqgf, aux_mqgf, modelo_final_classico, df_final)).min():.3f}"
      f" a {np.sqrt(variancia_log(modelo_mqgf, aux_mqgf, modelo_final_classico, df_final)).max():.3f}"
      f" (MQO: {np.sqrt(modelo_final_classico.scale):.3f} para todas as linhas)")

print("\nWald por termo, p-valores (RAD como bloco):")
pd.DataFrame({
    "MQO + HC3": modelo_final.wald_test_terms(scalar=True).table["pvalue"],
    "MQGF": modelo_mqgf.wald_test_terms(scalar=True).table["pvalue"],
    "MQGF + HC3": modelo_mqgf_hc3.wald_test_terms(scalar=True).table["pvalue"],
}).drop(index="Intercept").map(lambda v: f"{v:.2g}")

Pesos: razão entre o maior e o menor = 12.6
Desvio padrão estimado em log: de 0.058 a 0.207 (MQO: 0.129 para todas as linhas)

Wald por termo, p-valores (RAD como bloco):


,MQO + HC3,MQGF,MQGF + HC3
C(CHAS),5.9e-05,0.0028,0.0038
C(RAD),9.3e-06,1.6e-08,1.5e-07
CRIM,1.2e-07,1.6e-15,1.6e-07
NOX,1.4e-08,3.9e-08,3.4e-09
logDIS,1.1e-08,3.7e-12,4.9e-11
PTRATIO,1.8e-20,2e-16,4.2e-21
B,4.2e-05,5.4e-07,2.8e-06
logLSTAT,7.8e-50,9.3e-44,1.2e-42
RM_c,3.1e-15,9.7e-23,4.6e-17
I(RM_c ** 2),1.7e-16,5.8e-18,8.6e-11


### 9.3 Resíduos ponderados

Se a forma da variância está certa, os resíduos ponderados $\sqrt{w_i}\,e_i$ (`wresid`) têm variância constante. O Breusch-Pagan é aplicado a eles, com os mesmos regressores do teste no MQO, e o gráfico escala-locação é refeito com os resíduos ponderados padronizados ($\sqrt{w_i}\,e_i / s$). A correlação de Spearman entre $\sqrt{|r_i|}$ e o valor ajustado resume a inclinação de cada gráfico (zero = plano).

In [77]:
bp_mqo = het_breuschpagan(modelo_final_classico.resid, modelo_final_classico.model.exog)[1]
bp_mqgf = het_breuschpagan(modelo_mqgf.wresid, modelo_mqgf.model.exog)[1]
print(f"Breusch-Pagan p-valor: MQO = {bp_mqo:.2g} | MQGF (resíduos ponderados) = {bp_mqgf:.2g}")

escala_mqo = np.sqrt(np.abs(modelo_final_classico.get_influence().resid_studentized_internal))
escala_mqgf = np.sqrt(np.abs(modelo_mqgf.wresid / np.sqrt(modelo_mqgf.scale)))
paineis = [
    ("MQO (modelo final)", modelo_final_classico.fittedvalues, escala_mqo, "√|resíduo padronizado|"),
    ("MQGF (pesos por logLSTAT)", modelo_mqgf.fittedvalues, escala_mqgf, "√|resíduo ponderado padronizado|"),
]

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=[f"Escala-locação: {titulo}" for titulo, *_ in paineis])
for col, (titulo, ajustado, escala, rotulo) in enumerate(paineis, start=1):
    rho = stats.spearmanr(ajustado, escala).statistic
    print(f"{titulo}: Spearman(√|r|, ajustado) = {rho:+.3f}")
    fig.add_trace(pontos(ajustado, escala, ajustado.index, {"Ajustado": ajustado, rotulo: escala}), row=1, col=col)
    fig.add_trace(curva_lowess(ajustado, escala), row=1, col=col)
    fig.update_xaxes(title_text="Valor ajustado (log MEDV)", row=1, col=col)
    fig.update_yaxes(title_text=rotulo, row=1, col=col)
mostrar_e_salvar(fig, "escala_locacao_mqgf", altura=520)

Breusch-Pagan p-valor: MQO = 5.9e-12 | MQGF (resíduos ponderados) = 2e-06
MQO (modelo final): Spearman(√|r|, ajustado) = -0.220


MQGF (pesos por logLSTAT): Spearman(√|r|, ajustado) = -0.024


### 9.4 Comparação dos coeficientes

O MQO e o MQGF estimam os mesmos $\beta$ quando o modelo está bem especificado; só a eficiência muda. Por isso, os coeficientes devem ficar próximos. Uma diferença grande (várias vezes o erro padrão) indicaria erro de especificação da média, e não só heterocedasticidade. A coluna "Diferença / EP HC3" mede a distância entre as duas estimativas em erros padrão HC3 do MQO.

In [78]:
comparacao_mqgf = pd.DataFrame({
    "Coeficiente MQO": modelo_final.params,
    "EP HC3 (MQO)": modelo_final.bse,
    "Coeficiente MQGF": modelo_mqgf.params,
    "EP MQGF": modelo_mqgf.bse,
    "EP HC3 (MQGF)": modelo_mqgf_hc3.bse,
})
comparacao_mqgf["Diferença / EP HC3"] = (
    (comparacao_mqgf["Coeficiente MQGF"] - comparacao_mqgf["Coeficiente MQO"]) / comparacao_mqgf["EP HC3 (MQO)"])

print(f"Maior |diferença| em erros padrão HC3: {comparacao_mqgf['Diferença / EP HC3'].abs().max():.2f}"
      f" ({comparacao_mqgf['Diferença / EP HC3'].abs().idxmax()})")
print(f"Razão média EP HC3 (MQGF) / EP HC3 (MQO): {(modelo_mqgf_hc3.bse / modelo_final.bse).mean():.2f}")
comparacao_mqgf.round(4)

Maior |diferença| em erros padrão HC3: 1.17 (C(CHAS)[T.1])
Razão média EP HC3 (MQGF) / EP HC3 (MQO): 1.00


,Coeficiente MQO,EP HC3 (MQO),Coeficiente MQGF,EP MQGF,EP HC3 (MQGF),Diferença / EP HC3
Intercept,4.5968,0.1403,4.5786,0.1225,0.1318,-0.1296
C(CHAS)[T.1],0.0913,0.0227,0.0648,0.0215,0.0224,-1.1669
C(RAD)[T.2],0.0728,0.0352,0.0436,0.0324,0.0348,-0.8291
C(RAD)[T.3],0.1523,0.0333,0.1491,0.0290,0.0311,-0.0967
C(RAD)[T.4],0.0659,0.0284,0.0598,0.0264,0.0275,-0.2145
C(RAD)[T.5],0.0955,0.0292,0.0866,0.0264,0.0283,-0.3032
C(RAD)[T.6],0.0489,0.0351,0.0283,0.0348,0.0329,-0.5865
C(RAD)[T.7],0.1254,0.0347,0.1136,0.0344,0.0330,-0.3415
C(RAD)[T.8],0.0828,0.0345,0.0613,0.0322,0.0380,-0.6226
C(RAD)[T.24],0.1639,0.0425,0.1785,0.0368,0.0417,0.3428


### 9.5 Volta à escala de MEDV com variância modelada

Se $\log(MEDV) \mid x \sim N(\mu(x), \sigma^2(x))$, então $E[MEDV \mid x] = e^{\mu(x) + \sigma^2(x)/2}$. Com o MQGF, a previsão em mil dólares passa a ser

$$\widehat{MEDV} = e^{\hat y + \hat\sigma^2(x)/2},$$

com $\hat\sigma^2(x)$ dado por `variancia_log`. O fator de correção $e^{\hat\sigma^2(x)/2}$ passa a variar com `LSTAT`, enquanto o smearing de Duan aplica o mesmo fator a todas as casas. A opção lognormal é usada porque, sem as influentes, os resíduos são quase normais (correlação QQ de 0,993, Seção 8.2).

In [79]:
def prever_log_hetero(m_wls, aux, m_ols, dados):
    # Volta para a escala de MEDV com a correção lognormal e a variância de cada linha
    return np.exp(m_wls.predict(dados) + variancia_log(m_wls, aux, m_ols, dados) / 2)


fator_lognormal = np.exp(variancia_log(modelo_mqgf, aux_mqgf, modelo_final_classico, df_final) / 2)
fator_duan = np.mean(np.exp(modelo_final_classico.resid))
print(f"Fator de smearing de Duan (MQO, igual para todas as casas): {fator_duan:.4f}")
print(f"Fator lognormal (MQGF): média = {fator_lognormal.mean():.4f} | de {fator_lognormal.min():.4f} a {fator_lognormal.max():.4f}")

faixa_ajustado = pd.qcut(modelo_final_classico.fittedvalues, 4,
                         labels=["1º quartil (baratas)", "2º quartil", "3º quartil", "4º quartil (caras)"])
pd.DataFrame({
    "Fator lognormal médio": fator_lognormal.groupby(faixa_ajustado, observed=True).mean(),
    "Fator de Duan": fator_duan,
}).round(4)

Fator de smearing de Duan (MQO, igual para todas as casas): 1.0080
Fator lognormal (MQGF): média = 1.0084 | de 1.0017 a 1.0217


,Fator lognormal médio,Fator de Duan
1º quartil (baratas),1.0135,1.008
2º quartil,1.0091,1.008
3º quartil,1.0066,1.008
4º quartil (caras),1.0043,1.008


### 9.6 Validação cruzada e cobertura dos intervalos de previsão

A validação usa as **mesmas 10 partes** da Seção 8.6 (`df_t`, todas as linhas do treino). Em cada treino, o MQO, a regressão auxiliar e os pesos são estimados de novo; a parte de teste só é usada para prever. São comparados:

- o **modelo final (MQO)**, com o smearing de Duan (`prever_log`), que reproduz a linha "Final (todas as linhas)" da Seção 8.6;
- o **MQGF** com as três formas da variância da Seção 9.1, com a correção lognormal (`prever_log_hetero`).

Para cada casa da parte de teste é calculado o **intervalo de previsão de 95%** em log, com `get_prediction(...).summary_frame()` (colunas `obs_ci_*`), e levado para mil dólares com $e^x$. No MQO, a variância do erro é a mesma para todas as casas; no MQGF, ela é $\hat\sigma^2(x)$, passada pelos pesos da parte de teste. A **cobertura** é a proporção de `MEDV` dentro do intervalo; o ideal é 95% em todas as faixas de preço (quartis de `MEDV`).

In [80]:
faixa_preco = pd.qcut(df_t["MEDV"], 4)
modelos_cv = ["Final (MQO)"] + [f"MQGF {nome}" for nome in formas_variancia]
metricas_cv = {nome: [] for nome in modelos_cv}
cobertura_cv = []
for treino, teste in KFold(n_splits=10, shuffle=True, random_state=42).split(df_t):
    dados_treino, dados_teste = df_t.iloc[treino], df_t.iloc[teste]
    y_teste = dados_teste["MEDV"]
    m_ols = smf.ols(formula_final, data=dados_treino).fit()
    previsoes = {"Final (MQO)": prever_log(m_ols, dados_teste)}
    intervalos = {"Final (MQO)": m_ols.get_prediction(dados_teste).summary_frame(alpha=0.05)}
    for nome, forma in formas_variancia.items():
        m_wls, aux, m_ols_aux = ajustar_mqgf(dados_treino, forma)
        variancia_teste = variancia_log(m_wls, aux, m_ols_aux, dados_teste)
        previsoes[f"MQGF {nome}"] = prever_log_hetero(m_wls, aux, m_ols_aux, dados_teste)
        intervalos[f"MQGF {nome}"] = m_wls.get_prediction(
            dados_teste, weights=m_wls.scale / variancia_teste).summary_frame(alpha=0.05)
    rodada = pd.DataFrame({"Faixa de MEDV": faixa_preco.loc[dados_teste.index]}, index=dados_teste.index)
    for nome in modelos_cv:
        erro = y_teste - previsoes[nome]
        metricas_cv[nome].append({
            "RMSE": np.sqrt(np.mean(erro ** 2)),
            "MAE": np.mean(np.abs(erro)),
            "R² fora da amostra": 1 - np.sum(erro ** 2) / np.sum((y_teste - y_teste.mean()) ** 2),
        })
        inferior = np.exp(intervalos[nome]["obs_ci_lower"].values)
        superior = np.exp(intervalos[nome]["obs_ci_upper"].values)
        rodada[nome] = (y_teste >= inferior) & (y_teste <= superior)
        rodada[f"Largura: {nome}"] = superior - inferior
    cobertura_cv.append(rodada)

resultado_mqgf_cv = pd.DataFrame({
    nome: {**pd.DataFrame(rodadas).mean().to_dict(), "Desvio padrão do RMSE": pd.DataFrame(rodadas)["RMSE"].std()}
    for nome, rodadas in metricas_cv.items()
}).T
print("Erro de previsão (mil dólares), média das 10 rodadas:")
print(resultado_mqgf_cv.round(3).to_string())

cobertura_cv = pd.concat(cobertura_cv)
tabela_cobertura = cobertura_cv.groupby("Faixa de MEDV", observed=True)[modelos_cv].mean()
tabela_cobertura.loc["Todas"] = cobertura_cv[modelos_cv].mean()
largura = cobertura_cv.groupby("Faixa de MEDV", observed=True)[["Largura: Final (MQO)", "Largura: MQGF (b) logLSTAT"]].mean()
print("\nLargura média do intervalo de previsão (mil dólares):")
print(largura.round(1).to_string())
print("\nCobertura dos intervalos de previsão de 95%:")
(100 * tabela_cobertura).round(1)

Erro de previsão (mil dólares), média das 10 rodadas:
                               RMSE    MAE  R² fora da amostra  Desvio padrão do RMSE
Final (MQO)                   3.996  2.753               0.788                  1.193
MQGF (a) valor ajustado       3.917  2.710               0.800                  1.041
MQGF (b) logLSTAT             3.894  2.710               0.801                  1.095
MQGF (c) todos os preditores  4.012  2.625               0.788                  1.181

Largura média do intervalo de previsão (mil dólares):
               Largura: Final (MQO)  Largura: MQGF (b) logLSTAT
Faixa de MEDV                                                  
(4.999, 17.0]                  10.1                        12.5
(17.0, 21.2]                   13.8                        13.9
(21.2, 25.0]                   17.2                        14.7
(25.0, 50.0]                   24.4                        17.2

Cobertura dos intervalos de previsão de 95%:


,Final (MQO),MQGF (a) valor ajustado,MQGF (b) logLSTAT,MQGF (c) todos os preditores
Faixa de MEDV,,,,
"(4.999, 17.0]",87.3,94.4,91.3,90.5
"(17.0, 21.2]",99.2,99.2,99.2,97.6
"(21.2, 25.0]",96.8,98.4,96.8,92.7
"(25.0, 50.0]",94.4,91.9,94.4,90.3
Todas,94.4,96.0,95.4,92.8


### Conclusão - Heterocedasticidade

**Forma da variância.** A variância dos erros cresce com `LSTAT`: $\log(e^2) = 0{,}82 \cdot \log LSTAT + \text{constante}$, isto é, $\sigma^2_i \propto LSTAT_i^{0{,}82}$. Essa forma, com um só termo, explica mais que o valor ajustado (R² de 0,044 contra 0,034) e tem o menor BIC. A forma com todos os preditores tem R² maior (0,12), mas usa 18 termos e prevê pior (Seção 9.6). O desvio padrão estimado em log vai de 0,06 (`LSTAT` baixo) a 0,21 (`LSTAT` alto), contra 0,13 para todas as casas no MQO.

**Resíduos ponderados.** O Breusch-Pagan passa de p ≈ 6·10⁻¹² (MQO) para p ≈ 2·10⁻⁶ (MQGF). O teste ainda rejeita, mas o gráfico escala-locação fica **mais plano**: a tendência de queda desaparece, e a correlação de Spearman entre $\sqrt{|r|}$ e o valor ajustado cai de −0,22 para −0,02. Resta uma leve curva em U nas pontas, onde há poucas casas. A heterocedasticidade que resta não segue o nível de preço.

**Coeficientes.** Os coeficientes do MQGF ficam próximos dos do MQO: a maior diferença é de 1,2 erro padrão HC3 (`CHAS`, de +9,6% para +6,7%). Assim, não há sinal de erro de especificação da média. Com o MQGF, com ou sem HC3, **todos os termos continuam significativos** a 5%. Os erros padrão HC3 do MQGF são, em média, iguais aos do MQO (razão 1,00): o ganho de eficiência é pequeno.

**Volta à escala de MEDV.** Com a correção lognormal, o fator varia de 1,014 (casas baratas) a 1,004 (casas caras), contra um fator único de 1,008 no smearing de Duan. A diferença nas previsões é menor que 1%.

**Validação cruzada.** Com as mesmas 10 partes da Seção 8.6:

- o MQGF com pesos por `logLSTAT` erra menos que o modelo final: RMSE de 4,00 para **3,89**, MAE de 2,75 para 2,71 e R² fora da amostra de 0,79 para **0,80**;
- a cobertura dos intervalos de previsão de 95% passa de 94,4% para **95,4%** no total. Nas casas mais baratas (`MEDV` ≤ 17), ela sobe de 87,3% para 91,3%; nas mais caras, fica em 94,4%;
- os intervalos se ajustam ao preço: ficam mais largos nas casas baratas (de 10,1 para 12,5 mil dólares) e mais estreitos nas caras (de 24,4 para 17,2 mil dólares).

A cobertura nas casas mais baratas ainda fica abaixo de 95%. Nessa faixa estão as regiões pobres do centro de Boston, com os resíduos mais extremos (Seção 8.2). O intervalo normal não cobre essas caudas pesadas, mesmo com a variância modelada.

**Em síntese**, o RMSE do MQGF (3,89) é menor que o do modelo final (4,00), e os intervalos de previsão ficam mais próximos de 95% em todas as faixas de preço. Por isso, **para previsão e intervalos de previsão, o MQGF com pesos por `logLSTAT` é recomendado**. Para a interpretação, os coeficientes e as conclusões da Parte 8 (MQO com HC3) continuam válidos, pois o MQGF dá os mesmos sinais, a mesma significância e efeitos próximos. A heterocedasticidade foi **modelada** e não só corrigida nos erros padrão. Ainda há uma heterocedasticidade residual (Breusch-Pagan p ≈ 10⁻⁶) e caudas pesadas nas casas mais baratas.

## **Parte 10: Censura em `MEDV = 50` (Tobit)**

`MEDV` foi limitado em 50 (mil dólares) na coleta. Em 16 setores de `df_t`, o valor real é **50 ou mais**, mas o arquivo mostra só 50. O MQO trata esses valores como exatos: os resíduos dessas linhas ficam grandes e positivos, a parte de cima da reta é puxada para baixo e as inclinações tendem a ser **atenuadas**. Na Parte 6, a censura só foi verificada por **remoção** (modelo sem `MEDV = 50`), o que também gera viés: as linhas removidas são justamente as casas mais caras, e a amostra passa a ser truncada.

O **modelo Tobit** (regressão censurada, Tobin, 1958) usa a informação certa. Com $y_i = \log MEDV_i$, $c = \log 50$ e $y_i^* = x_i'\beta + \varepsilon_i$, $\varepsilon_i \sim N(0, \sigma^2)$, observa-se $y_i = \min(y_i^*, c)$. A log-verossimilhança de cada linha é

$$
\ell_i(\beta, \sigma) =
\begin{cases}
\log \phi\!\left(\dfrac{y_i - x_i'\beta}{\sigma}\right) - \log \sigma, & y_i < c \ \text{(não censurada)},\\[2ex]
\log\!\left[1 - \Phi\!\left(\dfrac{c - x_i'\beta}{\sigma}\right)\right], & y_i = c \ \text{(censurada)}.
\end{cases}
$$

Para uma linha censurada, entra a **probabilidade** de o valor ser 50 ou mais, e não a densidade no 50. A média é a mesma do modelo final (`formula_final`) e os dados são `df_t` com **todas as 501 linhas**, porque as linhas censuradas são o objeto desta parte.

### 10.1 Implementação

O statsmodels não tem um Tobit pronto. A classe `Tobit` abaixo é uma subclasse de `GenericLikelihoodModel`, que maximiza a soma de `loglikeobs` e calcula os erros padrão pela inversa da hessiana numérica:

- parâmetros: $\beta$ e $\gamma$, com $\log \sigma_i = z_i'\gamma$. Sem `exog_var`, $z_i = 1$ e $\gamma$ é só `log_sigma` (variância constante). O log garante $\sigma > 0$;
- linhas censuradas: `stats.norm.logsf`, que calcula $\log[1 - \Phi(\cdot)]$ sem perder precisão na cauda;
- ponto de partida: coeficientes do MQO e $\log \hat\sigma_{\text{MQO}}$; otimizador `bfgs`.

A matriz $X$ vem de `patsy.dmatrices(formula_final, df_t)`, a mesma do MQO.

In [81]:
import patsy
import statsmodels.api as sm
from statsmodels.base.model import GenericLikelihoodModel, LikelihoodResultsWrapper


class Tobit(GenericLikelihoodModel):
    # Regressão normal com censura superior em `limite`; log(sigma_i) = exog_var @ gamma
    def __init__(self, endog, exog, limite, exog_var=None, **kwds):
        if exog_var is None:
            exog_var = pd.DataFrame({"log_sigma": np.ones(len(exog))}, index=exog.index)
        self.limite = limite
        self.exog_var = np.asarray(exog_var, dtype=float)
        super().__init__(endog, exog, extra_params_names=list(exog_var.columns), **kwds)
        self.censurado = self.endog >= limite

    def loglikeobs(self, params):
        k = self.exog.shape[1]
        mu = self.exog @ params[:k]
        sigma = np.exp(self.exog_var @ params[k:])
        nao_censurado = stats.norm.logpdf((self.endog - mu) / sigma) - np.log(sigma)
        censurado = stats.norm.logsf((self.limite - mu) / sigma)  # log P(y* >= c)
        return np.where(self.censurado, censurado, nao_censurado)

    def fit(self, start_params=None, method="bfgs", maxiter=5000, **kwargs):
        if start_params is None:
            # Partida: MQO para beta e log do desvio padrão dos resíduos para gamma
            m_ols = sm.OLS(self.endog, self.exog).fit()
            gamma = np.zeros(self.exog_var.shape[1])
            gamma[0] = np.log(np.sqrt(m_ols.scale))
            start_params = np.r_[m_ols.params, gamma]
        resultado = super().fit(start_params=start_params, method=method, maxiter=maxiter, disp=0, **kwargs)
        return LikelihoodResultsWrapper(resultado)  # params, bse e pvalues com os nomes dos termos


limite_log = np.log(50)
y_t, X_t = patsy.dmatrices(formula_final, df_t, return_type="dataframe")
censurado_t = df_t["MEDV"] >= 50
print(f"Linhas censuradas (MEDV = 50): {censurado_t.sum()} de {len(df_t)}")
print(f"logMEDV >= log 50 exatamente nessas linhas: {((y_t['logMEDV'] >= limite_log) == censurado_t).all()}")

Linhas censuradas (MEDV = 50): 16 de 501
logMEDV >= log 50 exatamente nessas linhas: True


### 10.2 Conferência da implementação

Duas verificações antes de usar o Tobit nos dados reais:

1. **Simulação com $\beta$ conhecido.** Com a mesma matriz $X$ de `df_t`, os valores "verdadeiros" são os do MQO (`modelo_reduzido`): $y^* = X\beta + \sigma\varepsilon$, $\varepsilon \sim N(0, 1)$, censurado no quantil 95% (cerca de 5% de linhas censuradas, como os 16 de 501 dos dados reais). São feitas **200 réplicas** (semente 42). Em cada uma, o Tobit e o MQO são ajustados em $\min(y^*, c)$. A média das estimativas do Tobit deve ficar perto do valor verdadeiro (viés pequeno em relação ao desvio padrão entre réplicas), e o MQO deve mostrar a atenuação.
2. **Sem censura.** Com limite $+\infty$, nenhuma linha é censurada e a verossimilhança do Tobit é a da regressão normal. Os coeficientes devem ser **iguais** aos do MQO e $\hat\sigma$ igual ao estimador de máxima verossimilhança $\sqrt{SQR/n}$.

In [82]:
rng_tobit = np.random.default_rng(42)
nomes_tobit = list(X_t.columns) + ["log_sigma"]
verdadeiro = np.r_[modelo_reduzido.params.values, np.log(np.sqrt(modelo_reduzido.scale))]
media_latente = X_t.values @ verdadeiro[:-1]
sigma_verdadeiro = np.exp(verdadeiro[-1])
limite_sim = np.quantile(media_latente + sigma_verdadeiro * rng_tobit.standard_normal(len(X_t)), 0.95)

estimativas_sim = {"Tobit": [], "MQO": []}
ep_sim, convergiu_sim, fracao_sim = [], [], []
for _ in range(200):
    y_latente = media_latente + sigma_verdadeiro * rng_tobit.standard_normal(len(X_t))
    y_sim = pd.Series(np.minimum(y_latente, limite_sim), index=X_t.index)
    r_tobit = Tobit(y_sim, X_t, limite_sim).fit()
    r_mqo = sm.OLS(y_sim, X_t).fit()
    estimativas_sim["Tobit"].append(r_tobit.params.values)
    estimativas_sim["MQO"].append(np.r_[r_mqo.params.values, np.log(np.sqrt(r_mqo.scale))])
    ep_sim.append(r_tobit.bse.values)
    convergiu_sim.append(r_tobit.mle_retvals["converged"])
    fracao_sim.append((y_latente >= limite_sim).mean())

dp_sim = np.std(estimativas_sim["Tobit"], axis=0)
simulacao = pd.DataFrame({
    "Verdadeiro": verdadeiro,
    "Média Tobit": np.mean(estimativas_sim["Tobit"], axis=0),
    "Média MQO": np.mean(estimativas_sim["MQO"], axis=0),
    "DP Tobit (réplicas)": dp_sim,
    "EP Tobit (média)": np.mean(ep_sim, axis=0),
}, index=nomes_tobit)
simulacao["Viés Tobit / DP"] = (simulacao["Média Tobit"] - simulacao["Verdadeiro"]) / dp_sim
simulacao["Viés MQO / DP"] = (simulacao["Média MQO"] - simulacao["Verdadeiro"]) / dp_sim
assert all(convergiu_sim)
print(f"Simulação: 200 réplicas, todas convergiram = {all(convergiu_sim)}, "
      f"censura média = {np.mean(fracao_sim):.1%}")
print(f"Maior |viés| / DP: Tobit = {simulacao['Viés Tobit / DP'].abs().max():.2f} | "
      f"MQO = {simulacao['Viés MQO / DP'].abs().max():.2f}")

tobit_sem_censura = Tobit(y_t["logMEDV"], X_t, np.inf).fit()
assert tobit_sem_censura.mle_retvals["converged"]
dif_max = np.max(np.abs(tobit_sem_censura.params.values[:-1] - modelo_reduzido.params.values))
print(f"Sem censura: maior |beta Tobit - beta MQO| = {dif_max:.1e}; "
      f"sigma Tobit = {np.exp(tobit_sem_censura.params['log_sigma']):.5f}, "
      f"sqrt(SQR/n) = {np.sqrt(modelo_reduzido.ssr / modelo_reduzido.nobs):.5f}")
simulacao.round(3)

Simulação: 200 réplicas, todas convergiram = True, censura média = 5.2%
Maior |viés| / DP: Tobit = 0.53 | MQO = 1.60


Sem censura: maior |beta Tobit - beta MQO| = 7.7e-08; sigma Tobit = 0.17429, sqrt(SQR/n) = 0.17429


,Verdadeiro,Média Tobit,Média MQO,DP Tobit (réplicas),EP Tobit (média),Viés Tobit / DP,Viés MQO / DP
Intercept,5.041,5.034,5.054,0.181,0.170,-0.038,0.071
C(CHAS)[T.1],0.110,0.111,0.097,0.032,0.033,0.024,-0.386
C(RAD)[T.2],0.108,0.108,0.113,0.058,0.055,0.005,0.083
C(RAD)[T.3],0.182,0.182,0.181,0.055,0.049,-0.011,-0.013
C(RAD)[T.4],0.098,0.096,0.090,0.048,0.044,-0.047,-0.166
C(RAD)[T.5],0.131,0.132,0.125,0.048,0.045,0.013,-0.132
C(RAD)[T.6],0.095,0.093,0.085,0.055,0.053,-0.041,-0.176
C(RAD)[T.7],0.167,0.169,0.175,0.060,0.059,0.022,0.123
C(RAD)[T.8],0.129,0.133,0.135,0.060,0.056,0.074,0.113
C(RAD)[T.24],0.198,0.197,0.198,0.055,0.051,-0.019,0.005


### 10.3 Comparação de três ajustes

A mesma `formula_final` é ajustada de três formas em `df_t`:

- **MQO** com todas as 501 linhas (`modelo_reduzido`, célula 116), que trata `MEDV = 50` como valor exato;
- **MQO sem `MEDV = 50`** (485 linhas), a verificação por remoção da Parte 6;
- **Tobit** com todas as 501 linhas.

Os erros padrão do MQO são HC3, como no modelo final. Os do Tobit são dados de duas formas: pela inversa da hessiana (supõe o modelo correto) e pelo **sanduíche** $H^{-1}(S'S)H^{-1}$ (`cov_type="HC0"`, robusto a heterocedasticidade). Os efeitos são lidos em % de `MEDV`: $100\,(e^{\beta} - 1)$. A última coluna mede a mudança do Tobit em relação ao MQO em erros padrão HC3 do MQO.

In [83]:
modelo_tobit = Tobit(y_t["logMEDV"], X_t, limite_log).fit()
modelo_tobit_robusto = Tobit(y_t["logMEDV"], X_t, limite_log).fit(cov_type="HC0")
assert modelo_tobit.mle_retvals["converged"] and modelo_tobit_robusto.mle_retvals["converged"]
mqo_hc3 = smf.ols(formula_final, data=df_t).fit(cov_type="HC3")
mqo_sem_50 = smf.ols(formula_final, data=df_t[~censurado_t]).fit(cov_type="HC3")
print(f"Tobit convergiu: {modelo_tobit.mle_retvals['converged']}")
print(f"sigma: MQO = {np.sqrt(mqo_hc3.scale):.4f} | MQO sem MEDV = 50 = {np.sqrt(mqo_sem_50.scale):.4f} | "
      f"Tobit = {np.exp(modelo_tobit.params['log_sigma']):.4f}")

termos = mqo_hc3.params.index
comparacao_tobit = pd.DataFrame({
    "MQO": mqo_hc3.params,
    "EP HC3 MQO": mqo_hc3.bse,
    "MQO sem 50": mqo_sem_50.params,
    "EP HC3 sem 50": mqo_sem_50.bse,
    "Tobit": modelo_tobit.params[termos],
    "EP Tobit (hessiana)": modelo_tobit.bse[termos],
    "EP Tobit (sanduíche)": modelo_tobit_robusto.bse[termos],
})
for coluna in ["MQO", "MQO sem 50", "Tobit"]:
    comparacao_tobit[f"Efeito % {coluna}"] = 100 * (np.exp(comparacao_tobit[coluna]) - 1)
comparacao_tobit["(Tobit - MQO) / EP HC3"] = (comparacao_tobit["Tobit"] - comparacao_tobit["MQO"]) / comparacao_tobit["EP HC3 MQO"]
comparacao_tobit["(sem 50 - MQO) / EP HC3"] = (comparacao_tobit["MQO sem 50"] - comparacao_tobit["MQO"]) / comparacao_tobit["EP HC3 MQO"]

print("\nTermos que explicam as casas caras:")
print(comparacao_tobit.loc[["RM_c", "I(RM_c ** 2)", "logLSTAT", "I(logLSTAT_c ** 2)"]].T.round(4).to_string())
comparacao_tobit.drop(index="Intercept").round(4)

Tobit convergiu: True
sigma: MQO = 0.1777 | MQO sem MEDV = 50 = 0.1680 | Tobit = 0.1782

Termos que explicam as casas caras:
                            RM_c  I(RM_c ** 2)  logLSTAT  I(logLSTAT_c ** 2)
MQO                       0.0735        0.0560   -0.3827             -0.1089
EP HC3 MQO                0.0244        0.0169    0.0342              0.0286
MQO sem 50                0.0982        0.0675   -0.3190             -0.1294
EP HC3 sem 50             0.0276        0.0228    0.0284              0.0309
Tobit                     0.0823        0.0639   -0.3855             -0.1026
EP Tobit (hessiana)       0.0164        0.0102    0.0236              0.0215
EP Tobit (sanduíche)      0.0246        0.0180    0.0344              0.0290
Efeito % MQO              7.6269        5.7591  -31.7964            -10.3177
Efeito % MQO sem 50      10.3131        6.9805  -27.3157            -12.1396
Efeito % Tobit            8.5733        6.6025  -31.9885             -9.7554
(Tobit - MQO) / EP HC3    0.

,MQO,EP HC3 MQO,MQO sem 50,EP HC3 sem 50,Tobit,EP Tobit (hessiana),EP Tobit (sanduíche),Efeito % MQO,Efeito % MQO sem 50,Efeito % Tobit,(Tobit - MQO) / EP HC3,(sem 50 - MQO) / EP HC3
C(CHAS)[T.1],0.1099,0.0438,0.0627,0.0422,0.1213,0.0332,0.0447,11.6153,6.4758,12.8910,0.2596,-1.0770
C(RAD)[T.2],0.1078,0.0482,0.1229,0.0481,0.0951,0.0558,0.0463,11.3843,13.0753,9.9717,-0.2645,0.3123
C(RAD)[T.3],0.1822,0.0455,0.1898,0.0456,0.1768,0.0499,0.0439,19.9830,20.9016,19.3379,-0.1184,0.1674
C(RAD)[T.4],0.0982,0.0436,0.1015,0.0428,0.0976,0.0448,0.0419,10.3179,10.6814,10.2551,-0.0131,0.0755
C(RAD)[T.5],0.1312,0.0427,0.1414,0.0420,0.1301,0.0452,0.0411,14.0166,15.1858,13.8885,-0.0263,0.2387
C(RAD)[T.6],0.0951,0.0504,0.0970,0.0491,0.0946,0.0543,0.0481,9.9788,10.1810,9.9167,-0.0112,0.0365
C(RAD)[T.7],0.1674,0.0475,0.1610,0.0465,0.1599,0.0594,0.0452,18.2248,17.4674,17.3431,-0.1575,-0.1352
C(RAD)[T.8],0.1286,0.0491,0.1475,0.0526,0.1144,0.0563,0.0482,13.7273,15.8975,12.1172,-0.2905,0.3851
C(RAD)[T.24],0.1982,0.0601,0.1571,0.0567,0.1959,0.0518,0.0564,21.9254,17.0140,21.6372,-0.0394,-0.6846
CRIM,-0.0121,0.0031,-0.0112,0.0028,-0.0122,0.0013,0.0024,-1.2021,-1.1118,-1.2132,-0.0367,0.2992


### 10.4 Pressupostos do Tobit

Ao contrário do MQO, que continua consistente para $\beta$ com erros heterocedásticos ou não normais, o Tobit **depende** da forma da distribuição: a probabilidade de censura $1 - \Phi\big((c - x'\beta)/\sigma\big)$ só está certa se os erros forem normais e de variância constante. Com a heterocedasticidade encontrada na Parte 9 (a variância cresce com `LSTAT`), o Tobit homocedástico pode ser **viesado**. Duas verificações:

1. **Erros padrão sanduíche** (`cov_type="HC0"`): corrigem a inferência, mas não o viés de $\beta$. São comparados com os da hessiana e com o HC3 do MQO pelos p-valores de cada termo.
2. **Tobit heterocedástico**: $\log \sigma_i = \gamma_0 + \gamma_1 \log LSTAT_i$, a mesma forma escolhida na Parte 9. O teste de **razão de verossimilhança** compara com o Tobit homocedástico ($H_0: \gamma_1 = 0$, 1 GL). Se os $\beta$ mudarem pouco, o viés causado pela heterocedasticidade é pequeno.

In [84]:
p_valores_tobit = pd.DataFrame({
    "MQO (HC3)": mqo_hc3.pvalues,
    "Tobit (hessiana)": modelo_tobit.pvalues[termos],
    "Tobit (sanduíche)": modelo_tobit_robusto.pvalues[termos],
}).drop(index="Intercept")
razao_ep = (modelo_tobit_robusto.bse[termos] / modelo_tobit.bse[termos]).drop(index="Intercept")
print(f"EP sanduíche / EP hessiana: média {razao_ep.mean():.2f}, de {razao_ep.min():.2f} a {razao_ep.max():.2f}")
print(f"Termos com p < 0,05: MQO (HC3) = {(p_valores_tobit['MQO (HC3)'] < 0.05).sum()}, "
      f"Tobit (sanduíche) = {(p_valores_tobit['Tobit (sanduíche)'] < 0.05).sum()} de {len(p_valores_tobit)}")

var_tobit = pd.DataFrame({"log_sigma: Intercept": 1.0, "log_sigma: logLSTAT": df_t["logLSTAT"]}, index=df_t.index)
tobit_hetero = Tobit(y_t["logMEDV"], X_t, limite_log, exog_var=var_tobit).fit()
tobit_hetero_robusto = Tobit(y_t["logMEDV"], X_t, limite_log, exog_var=var_tobit).fit(cov_type="HC0")
assert tobit_hetero.mle_retvals["converged"] and tobit_hetero_robusto.mle_retvals["converged"]
razao_verossimilhanca = 2 * (tobit_hetero.llf - modelo_tobit.llf)
print(f"\nTobit heterocedástico: gamma_logLSTAT = {tobit_hetero.params['log_sigma: logLSTAT']:.3f} "
      f"(EP sanduíche {tobit_hetero_robusto.bse['log_sigma: logLSTAT']:.3f}), "
      f"isto é, sigma² proporcional a LSTAT^{2 * tobit_hetero.params['log_sigma: logLSTAT']:.2f}")
print(f"Razão de verossimilhança: {razao_verossimilhanca:.1f} (1 GL), p = {stats.chi2.sf(razao_verossimilhanca, 1):.2g}")

comparacao_hetero = pd.DataFrame({
    "Tobit": modelo_tobit.params[termos],
    "Tobit heterocedástico": tobit_hetero.params[termos],
    "Diferença / EP sanduíche": (tobit_hetero.params[termos] - modelo_tobit.params[termos]) / modelo_tobit_robusto.bse[termos],
    "p Tobit heterocedástico (sanduíche)": tobit_hetero_robusto.pvalues[termos],
}).drop(index="Intercept")
print(f"Maior |diferença| / EP sanduíche: {comparacao_hetero['Diferença / EP sanduíche'].abs().max():.2f} "
      f"({comparacao_hetero['Diferença / EP sanduíche'].abs().idxmax()})")
pd.concat([p_valores_tobit.map(lambda v: f"{v:.2g}"), comparacao_hetero.round(4)], axis=1)

EP sanduíche / EP hessiana: média 1.17, de 0.76 a 1.89
Termos com p < 0,05: MQO (HC3) = 17, Tobit (sanduíche) = 18 de 18



Tobit heterocedástico: gamma_logLSTAT = 0.343 (EP sanduíche 0.093), isto é, sigma² proporcional a LSTAT^0.69
Razão de verossimilhança: 41.9 (1 GL), p = 9.8e-11
Maior |diferença| / EP sanduíche: 0.87 (logDIS)


,MQO (HC3),Tobit (hessiana),Tobit (sanduíche),Tobit,Tobit heterocedástico,Diferença / EP sanduíche,p Tobit heterocedástico (sanduíche)
C(CHAS)[T.1],0.012,0.00026,0.0067,0.1213,0.0997,-0.4826,0.0262
C(RAD)[T.2],0.025,0.088,0.04,0.0951,0.0629,-0.6941,0.1614
C(RAD)[T.3],6.3e-05,0.0004,5.6e-05,0.1768,0.1651,-0.2675,0.0001
C(RAD)[T.4],0.024,0.029,0.02,0.0976,0.0833,-0.3415,0.0417
C(RAD)[T.5],0.0021,0.004,0.0015,0.1301,0.1120,-0.4407,0.0050
C(RAD)[T.6],0.059,0.082,0.05,0.0946,0.0688,-0.5357,0.1360
C(RAD)[T.7],0.00043,0.0071,0.00041,0.1599,0.1537,-0.1384,0.0007
C(RAD)[T.8],0.0088,0.042,0.018,0.1144,0.0976,-0.3487,0.0649
C(RAD)[T.24],0.00096,0.00015,0.00051,0.1959,0.2201,0.4292,0.0002
CRIM,7.4e-05,2.9e-22,2.6e-07,-0.0122,-0.0123,-0.0416,0.0000


### 10.5 Validação cruzada

A validação usa as **mesmas 10 partes** da Seção 8.6 (`KFold(10, shuffle=True, random_state=42)` em `df_t`). O valor observado é $\min(MEDV, 50)$, então a previsão do Tobit é o **valor esperado observado**. Com $\log MEDV \mid x \sim N(\mu, \sigma^2)$, $\mu = x'\hat\beta$ e $\alpha = (\log 50 - \mu)/\sigma$:

$$
E[\min(MEDV, 50) \mid x] = e^{\mu + \sigma^2/2}\,\Phi(\alpha - \sigma) + 50\,[1 - \Phi(\alpha)].
$$

O primeiro termo é a média lognormal na parte abaixo de 50 e o segundo é o valor 50 vezes a probabilidade de censura. A comparação é com o modelo final (MQO e fator de smearing de Duan, `prever_log`), em todas as linhas e separadamente nas 16 censuradas.

In [85]:
def prever_tobit(m, X_novo):
    # E[min(MEDV, 50) | x] com log(MEDV) ~ N(mu, sigma²) e censura em 50 (variância constante)
    k = m.model.exog.shape[1]
    mu = np.asarray(X_novo) @ m.params.values[:k]
    sigma = np.exp(m.params["log_sigma"])
    alpha = (m.model.limite - mu) / sigma
    return np.exp(mu + sigma ** 2 / 2) * stats.norm.cdf(alpha - sigma) + np.exp(m.model.limite) * stats.norm.sf(alpha)


modelos_tobit_cv = ["Final (MQO + Duan)", "Tobit"]
metricas_tobit_cv = {nome: [] for nome in modelos_tobit_cv}
previsoes_tobit_cv = []
for treino, teste in KFold(n_splits=10, shuffle=True, random_state=42).split(df_t):
    dados_treino, dados_teste = df_t.iloc[treino], df_t.iloc[teste]
    y_teste = dados_teste["MEDV"]
    y_treino, X_treino = patsy.dmatrices(formula_final, dados_treino, return_type="dataframe")
    X_teste = patsy.build_design_matrices([X_treino.design_info], dados_teste, return_type="dataframe")[0]
    m_ols = smf.ols(formula_final, data=dados_treino).fit()
    m_tobit = Tobit(y_treino["logMEDV"], X_treino, limite_log).fit()
    assert m_tobit.mle_retvals["converged"]
    previsoes = {"Final (MQO + Duan)": prever_log(m_ols, dados_teste),
                 "Tobit": pd.Series(prever_tobit(m_tobit, X_teste), index=dados_teste.index)}
    previsoes_tobit_cv.append(pd.DataFrame(previsoes).assign(MEDV=y_teste))
    for nome, previsto in previsoes.items():
        erro = y_teste - previsto
        metricas_tobit_cv[nome].append({
            "RMSE": np.sqrt(np.mean(erro ** 2)),
            "MAE": np.mean(np.abs(erro)),
            "R² fora da amostra": 1 - np.sum(erro ** 2) / np.sum((y_teste - y_teste.mean()) ** 2),
        })

resultado_tobit_cv = pd.DataFrame({
    nome: {**pd.DataFrame(rodadas).mean().to_dict(), "Desvio padrão do RMSE": pd.DataFrame(rodadas)["RMSE"].std()}
    for nome, rodadas in metricas_tobit_cv.items()
}).T
print("Erro de previsão (mil dólares), média das 10 rodadas:")
print(resultado_tobit_cv.round(3).to_string())

previsoes_tobit_cv = pd.concat(previsoes_tobit_cv)
grupos_censura = previsoes_tobit_cv["MEDV"].ge(50).map({True: "MEDV = 50 (16)", False: "MEDV < 50 (485)"}).rename("Grupo")
erros_grupo = previsoes_tobit_cv[modelos_tobit_cv].rsub(previsoes_tobit_cv["MEDV"], axis=0).groupby(grupos_censura)
print("\nPor grupo, juntando as 10 partes (mil dólares):")
pd.concat({"RMSE": erros_grupo.apply(lambda e: np.sqrt((e ** 2).mean())),
           "Erro médio (observado - previsto)": erros_grupo.mean()}, axis=1).round(2)

Erro de previsão (mil dólares), média das 10 rodadas:
                     RMSE    MAE  R² fora da amostra  Desvio padrão do RMSE
Final (MQO + Duan)  3.996  2.753               0.788                  1.193
Tobit               3.827  2.686               0.810                  0.919

Por grupo, juntando as 10 partes (mil dólares):


RMSE        Erro médio (observado - previsto)  \
                Final (MQO + Duan)  Tobit                Final (MQO + Duan)   
Grupo                                                                         
MEDV < 50 (485)               3.52   3.35                             -0.35   
MEDV = 50 (16)               12.78  11.90                              7.21   

                       
                Tobit  
Grupo                  
MEDV < 50 (485) -0.38  
MEDV = 50 (16)   8.81

### Conclusão - Censura em `MEDV = 50` (Tobit)

**Implementação.** O Tobit convergiu em todos os ajustes. Na simulação (200 réplicas, 5,2% de censura), a média das estimativas fica perto do valor verdadeiro: o maior viés dos coeficientes é de 0,24 desvio padrão (`RM_c²`). O viés de 0,53 desvio padrão em `log_sigma` é o viés conhecido da máxima verossimilhança para $\sigma$ em amostra finita. O MQO no mesmo $y$ censurado mostra a **atenuação** prevista: `RM_c` cai de 0,074 para 0,050 (viés de 1,3 desvio padrão) e `RM_c²` de 0,056 para 0,039 (1,6 desvio padrão). Sem censura, o Tobit reproduz o MQO (diferença de 8·10⁻⁸ e $\hat\sigma = \sqrt{SQR/n}$).

**A censura muda as conclusões do modelo final?** Não. Nos dados reais, o Tobit fica a menos de **0,5 erro padrão HC3** do MQO em todos os termos. As maiores mudanças estão onde a simulação indicou, nos termos de `RM`, que explicam as casas caras: `RM_c` sobe 12% (0,074 → 0,082) e `RM_c²` sobe 14% (0,056 → 0,064). `logLSTAT` quase não muda (−0,383 → −0,386; −31,8% → −32,0% por aumento de 1 em log). $\hat\sigma$ também é igual (0,178). Os sinais e a significância a 5% se mantêm: com erros padrão sanduíche, os 18 termos são significativos (no MQO com HC3, 17; `RAD = 6` passa de p = 0,059 para 0,050). Com só 16 linhas censuradas (3%), o viés do MQO é pequeno.

**A remoção muda mais, e na direção errada.** O MQO sem `MEDV = 50` (a verificação da Parte 6) atenua `logLSTAT` (−0,383 → −0,319, 1,9 erro padrão) e `logDIS` (1,8 erro padrão) e reduz `CHAS` pela metade. Remover as casas mais caras trunca a amostra. O Tobit confirma que a remoção não é um bom tratamento da censura.

**Pressupostos.** O Tobit depende de erros normais e de variância constante:

- os erros padrão sanduíche são, em média, 1,17 vez os da hessiana (até 1,9 em `CRIM` e 1,8 em `RM_c²`). O modelo não está correto em toda a distribuição, e a inferência deve usar o sanduíche;
- o Tobit heterocedástico rejeita a variância constante (razão de verossimilhança 41,9, p ≈ 10⁻¹⁰), com $\sigma^2 \propto LSTAT^{0{,}69}$, perto da forma da Parte 9 ($LSTAT^{0{,}82}$). Mesmo assim, os coeficientes mudam no máximo 0,87 erro padrão (`logDIS`), e `RM_c` não muda. O viés do Tobit homocedástico causado pela heterocedasticidade é pequeno. No Tobit heterocedástico, `RAD = 2`, `6` e `8` deixam de ser significativos a 5%.

**Previsão.** Com as mesmas 10 partes, o Tobit prevê melhor que o modelo final: RMSE 3,83 contra 4,00, MAE 2,69 contra 2,75 e R² fora da amostra 0,81 contra 0,79. O ganho aparece nos dois grupos (RMSE 3,35 contra 3,52 nas linhas abaixo de 50 e 11,9 contra 12,8 nas 16 censuradas). As casas censuradas continuam mal previstas: em média, o valor observado (50) fica 8,8 mil dólares acima da previsão.

**Limitações.** O Tobit supõe a normalidade dos erros, e os resíduos têm caudas pesadas (Parte 8); a variância constante é rejeitada, embora o efeito nos coeficientes seja pequeno; o limite de 50 é tratado como conhecido e único; o ajuste usa as 501 linhas de `df_t`, com as linhas influentes (5 das 28 têm `MEDV = 50`), e não o `df_final` do modelo final; e a correlação espacial (Parte 7) continua. O modelo final da Parte 8 é mantido. O Tobit fica como verificação de robustez: a censura não muda as conclusões, e o efeito de `RM` no MQO é um pouco **atenuado** (cerca de 12% a 14%).

## **Parte 11: Correlação espacial**

Os setores censitários do Boston Housing são vizinhos no mapa, e setores próximos tendem a ter preços parecidos. Se o modelo não captura essa semelhança, os resíduos de setores vizinhos ficam correlacionados. Isso viola a independência dos erros: os erros padrão ficam menores do que deveriam e, se a dependência estiver na resposta, os coeficientes podem ter viés.

A Parte 8 (Seção 8.4) mostrou que o Durbin-Watson baixo (1,41) vem da ordem do arquivo, que segue a geografia, e que as conclusões resistem a erros padrão agrupados por uma **cidade aproximada** (`cidade`). Isso trata a correlação **dentro** de cada cidade, mas não **entre** cidades vizinhas. Faltavam as coordenadas.

**Fonte das coordenadas.** A versão corrigida do Boston Housing (Gilley e Pace, 1996; Pace e Gilley, 1997) tem a cidade (`TOWN`), o setor censitário (`TRACT`), a longitude e a latitude do setor (`LON`, `LAT`) e o valor corrigido de `MEDV` (`CMEDV`). O arquivo foi baixado da StatLib (`https://lib.stat.cmu.edu/datasets/boston_corrected.txt`) e salvo em `data/boston_corrected.csv` (as colunas `OBS.` e `TOWN#` viraram `OBS` e `TOWN_ID`).

Etapas:

1. ligar as linhas do projeto às da fonte (11.1);
2. montar as matrizes de vizinhança (11.2);
3. testar a correlação espacial dos resíduos com o I de Moran (11.3);
4. escolher e ajustar um modelo espacial (11.4) e comparar com o modelo final (11.5);
5. agrupar os erros padrão pela cidade verdadeira (`TOWN`) (11.6).

### 11.1 Ligação das linhas e mapa dos resíduos

As linhas 0 a 505 de `housing.csv` seguem a ordem do conjunto original, e `df_t` e `df_final` mantêm o índice do arquivo (célula 86: `df.loc[:505]`). A ligação é feita pelo índice: a linha `i` do projeto corresponde à linha `i` da fonte. A tabela abaixo confere, variável por variável, quantas linhas têm o mesmo valor (tolerância de 0,001). Uma ligação errada apareceria como muitas divergências em todas as variáveis.

O mapa mostra os resíduos do `modelo_final` na posição de cada setor. Se os resíduos forem independentes, as cores ficam misturadas sem padrão. Manchas da mesma cor indicam correlação espacial.

In [86]:
import warnings
from contextlib import redirect_stdout
from io import StringIO

from libpysal.weights import KNN, DistanceBand, min_threshold_distance
from esda.moran import Moran
import spreg
from sklearn.metrics import adjusted_rand_score

boston_c = pd.read_csv("../data/boston_corrected.csv")
variaveis_ligacao = ["CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS", "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"]
ligacao = pd.DataFrame({
    nome: {v: int(np.isclose(dados[v], boston_c.loc[dados.index, v], atol=1e-3).sum()) for v in variaveis_ligacao}
    for nome, dados in [("df_t (n = 501)", df_t), ("df_final (n = 473)", df_final)]
})
print(ligacao.T.to_string())

crim_diferente = df_t.index[~np.isclose(df_t["CRIM"], boston_c.loc[df_t.index, "CRIM"], atol=1e-3)]
print(f"\nCRIM diferente nas linhas {list(crim_diferente)}: housing.csv = {df_t.loc[crim_diferente, 'CRIM'].tolist()} | "
      f"fonte = {boston_c.loc[crim_diferente, 'CRIM'].tolist()}")
medv_corrigido = boston_c.loc[df_t.index].query("MEDV != CMEDV")[["TOWN", "MEDV", "CMEDV"]]
print(f"\nLinhas de df_t com MEDV corrigido (CMEDV != MEDV): {len(medv_corrigido)}, "
      f"{medv_corrigido.index.isin(df_final.index).sum()} delas em df_final")
print(medv_corrigido.to_string())

# coordenadas em km: a longitude é multiplicada pelo cosseno da latitude média
lat_media = np.deg2rad(boston_c["LAT"].mean())


def coordenadas_km(indice):
    b = boston_c.loc[indice]
    return np.column_stack([b["LON"] * 111.32 * np.cos(lat_media), b["LAT"] * 110.57])


xy_final = coordenadas_km(df_final.index)
residuos_mapa = modelo_final.resid.values
limite_cor = np.abs(residuos_mapa).max()
fig = go.Figure(go.Scatter(
    x=boston_c.loc[df_final.index, "LON"], y=boston_c.loc[df_final.index, "LAT"], mode="markers", name="Resíduo",
    marker=dict(color=residuos_mapa, colorscale=ESCALA_DIVERGENTE, cmin=-limite_cor, cmax=limite_cor, size=8,
                line=dict(width=0.5, color="#898781"), colorbar=dict(title="Resíduo<br>(log MEDV)")),
    hovertext=texto_hover(df_final.index, {"Resíduo": residuos_mapa}), hovertemplate="%{hovertext}<extra></extra>",
))
fig.update_xaxes(title_text="Longitude")
fig.update_yaxes(title_text="Latitude", scaleanchor="x", scaleratio=1 / np.cos(lat_media))
mostrar_e_salvar(fig, "residuos_mapa", titulo="Resíduos do modelo final no mapa (n = 473)", altura=800, largura=800)

                    CRIM   ZN  INDUS  CHAS  NOX   RM  AGE  DIS  RAD  TAX  PTRATIO    B  LSTAT  MEDV
df_t (n = 501)       500  501    501   501  501  501  501  501  501  501      501  501    501   501
df_final (n = 473)   472  473    473   473  473  473  473  473  473  473      473  473    473   473

CRIM diferente nas linhas [500]: housing.csv = [0.177783] | fonte = [0.22438]

Linhas de df_t com MEDV corrigido (CMEDV != MEDV): 8, 6 delas em df_final
                  TOWN  MEDV  CMEDV
7                Salem  27.1   22.1
38              Sargus  24.7   24.2
190          Lexington  37.0   33.0
240             Natick  22.0   27.0
437  Boston Savin Hill   8.7    8.2
442  Boston Savin Hill  18.4   14.8
454  Boston Savin Hill  14.9   14.4
505           Winthrop  11.9   19.0


### 11.2 Matrizes de vizinhança

O I de Moran e os modelos espaciais precisam de uma matriz $W$ que diga quem é vizinho de quem. Duas matrizes são usadas, para ver se a conclusão depende da escolha:

- **k vizinhos mais próximos** (`libpysal.weights.KNN.from_array`, $k = 6$): cada setor tem os 6 setores mais próximos como vizinhos. Todos os setores têm o mesmo número de vizinhos, mesmo nas bordas e nas áreas com poucos setores;
- **distância** (`libpysal.weights.DistanceBand`, `binary=True`): são vizinhos os setores a menos de um limiar $d$. O limiar é o menor que dá ao menos um vizinho a cada setor (`min_threshold_distance`). Nas áreas densas (centro de Boston), um setor tem muitos vizinhos; nas áreas afastadas, poucos.

As duas são **padronizadas por linha** (`transform = "r"`): os pesos de cada linha somam 1, e $Wy$ é a média dos vizinhos. As distâncias são euclidianas em km, com a longitude multiplicada por $\cos(\text{latitude média})$. Na escala da região (cerca de 40 km), o erro dessa aproximação é desprezível.

As matrizes são montadas para as linhas de cada modelo: `df_final` (473) e `df_t` (501).

In [87]:
def matrizes_vizinhanca(indice):
    xy = coordenadas_km(indice)
    w_knn = KNN.from_array(xy, k=6)
    w_knn.transform = "r"
    limiar = min_threshold_distance(xy)
    w_dist = DistanceBand(xy, threshold=limiar, binary=True, silence_warnings=True)
    w_dist.transform = "r"
    return {"KNN (k = 6)": w_knn, f"Distância (d = {limiar:.2f} km)": w_dist}


W_final = matrizes_vizinhanca(df_final.index)
W_t = matrizes_vizinhanca(df_t.index)
for nome, w in W_final.items():
    viz = pd.Series(w.cardinalities)
    print(f"{nome}: vizinhos por setor: mín = {viz.min()}, mediana = {viz.median():.0f}, máx = {viz.max()} | "
          f"ilhas = {len(w.islands)}")
W_knn = W_final["KNN (k = 6)"]

KNN (k = 6): vizinhos por setor: mín = 6, mediana = 6, máx = 6 | ilhas = 0
Distância (d = 5.27 km): vizinhos por setor: mín = 1, mediana = 88, máx = 204 | ilhas = 0


### 11.3 I de Moran nos resíduos

O I de Moran (`esda.moran.Moran`) mede a correlação entre o resíduo de cada setor e a média dos resíduos dos vizinhos:

$$
I = \frac{n}{\sum_{ij} w_{ij}} \, \frac{\sum_{ij} w_{ij}\, e_i e_j}{\sum_i e_i^2}.
$$

Sem correlação espacial, $E[I] = -1/(n-1) \approx 0$. Valores positivos indicam que vizinhos têm resíduos parecidos. Dois p-valores:

- **normal** (`p_norm`): supõe que $I$ segue a distribuição normal sob a hipótese de independência;
- **por permutação** (`p_sim`, 999 permutações, semente 42): os resíduos são embaralhados entre os locais 999 vezes. O p-valor é a fração de permutações com $I$ maior ou igual ao observado. Não depende da normalidade. O menor valor possível é $1/1000 = 0{,}001$.

Este teste usa os resíduos do MQO como se fossem os erros. Para resíduos de regressão, a média e a variância exatas de $I$ são um pouco diferentes (os testes LM da Seção 11.4 levam isso em conta), mas a diferença é pequena com $n$ grande.

Os testes são feitos para o `modelo_final` (473 linhas) e para o `modelo_reduzido` (501 linhas, com as observações influentes), nas duas matrizes.

In [88]:
linhas_moran = []
for nome_modelo, m, matrizes in [("modelo_final (n = 473)", modelo_final, W_final),
                                 ("modelo_reduzido (n = 501)", modelo_reduzido, W_t)]:
    for nome_w, w in matrizes.items():
        np.random.seed(42)
        mi = Moran(m.resid.values, w, permutations=999)
        linhas_moran.append({"Modelo": nome_modelo, "Matriz": nome_w, "I": mi.I, "E[I]": mi.EI, "z": mi.z_norm,
                             "p normal": mi.p_norm, "p permutação": mi.p_sim})

# conferência: com os resíduos embaralhados entre os locais, I deve ficar perto de E[I]
rng_moran = np.random.default_rng(42)
np.random.seed(42)
mi_aleatorio = Moran(rng_moran.permutation(modelo_final.resid.values), W_knn, permutations=999)
print(f"Conferência, resíduos em ordem aleatória (KNN): I = {mi_aleatorio.I:.4f} | p permutação = {mi_aleatorio.p_sim:.3f}\n")
tabela_moran = pd.DataFrame(linhas_moran).set_index(["Modelo", "Matriz"])
tabela_moran.style.format({"I": "{:.4f}", "E[I]": "{:.4f}", "z": "{:.2f}", "p normal": "{:.2e}", "p permutação": "{:.3f}"})

Conferência, resíduos em ordem aleatória (KNN): I = -0.0158 | p permutação = 0.312



### 11.4 Escolha do modelo espacial (testes LM)

Há dois modelos espaciais básicos (Anselin, 1988):

- **erro espacial** (SEM): $y = X\beta + u$, $u = \lambda W u + \varepsilon$. A correlação está nos erros: algum fator do lugar que não está no modelo (acesso, vizinhança, qualidade das escolas) é parecido entre setores vizinhos. $\beta$ tem a mesma interpretação do MQO; o MQO não tem viés, mas os erros padrão estão errados e a estimação é ineficiente;
- **defasagem espacial** (SAR): $y = \rho W y + X\beta + \varepsilon$. O preço de um setor depende diretamente do preço dos vizinhos. Aqui o MQO tem viés, e $\beta$ não é o efeito marginal: uma mudança em $x_i$ se propaga pelos vizinhos, e o efeito total é $\beta/(1-\rho)$ em média.

Os **testes de multiplicadores de Lagrange** (`spreg.OLS(..., spat_diag=True)`) usam só o ajuste do MQO. `LM erro` testa $\lambda = 0$ e `LM defasagem` testa $\rho = 0$. As versões **robustas** (Anselin et al., 1996) testam cada um permitindo a presença do outro. A regra usual: se só um dos robustos é significativo, escolhe-se esse modelo; se os dois são, escolhe-se o de maior estatística.

`spreg.OLS` recebe a mesma matriz $X$ do `modelo_final` (`patsy.dmatrices(formula_final, df_final)`, sem a coluna do intercepto, que o spreg acrescenta). A primeira linha da saída confere que os coeficientes são iguais aos do `modelo_final`. Os modelos são ajustados por máxima verossimilhança (`spreg.ML_Error`, `spreg.ML_Lag`, método `full`), que supõe erros $\varepsilon$ normais e homocedásticos. Os testes LM e os ajustes são repetidos com a matriz de distância.

In [89]:
y_sp, X_sp = patsy.dmatrices(formula_final, df_final, return_type="dataframe")
nomes_x = list(X_sp.columns[1:])
with warnings.catch_warnings():
    # aviso do número de condição calculado pelo spreg (autovalores com parte imaginária desprezível)
    warnings.simplefilter("ignore", np.exceptions.ComplexWarning)
    ols_espacial = {nome: spreg.OLS(y_sp.values, X_sp.values[:, 1:], w=w, spat_diag=True, moran=True,
                                    name_y="logMEDV", name_x=nomes_x) for nome, w in W_final.items()}
print(f"spreg.OLS reproduz o modelo_final: {np.allclose(ols_espacial['KNN (k = 6)'].betas.ravel(), modelo_final.params.values)}\n")

testes_lm = pd.DataFrame({
    nome: {"LM erro": o.lm_error, "LM defasagem": o.lm_lag, "LM erro robusto": o.rlm_error,
           "LM defasagem robusto": o.rlm_lag, "LM SARMA": o.lm_sarma}
    for nome, o in ols_espacial.items()
})
tabela_lm = pd.concat({nome: pd.DataFrame(testes_lm[nome].tolist(), index=testes_lm.index, columns=["Estatística", "p"])
                       for nome in testes_lm.columns}, axis=1)
print(tabela_lm.to_string(float_format=lambda v: f"{v:.3g}"))

modelos_ml = {}
for nome, w in W_final.items():
    # o spreg imprime o nome do modelo ao ajustar e avisa sobre a tolerância do otimizador; saída e avisos são descartados
    with redirect_stdout(StringIO()), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        modelos_ml[("Erro (SEM)", nome)] = spreg.ML_Error(y_sp.values, X_sp.values[:, 1:], w=w, name_y="logMEDV", name_x=nomes_x)
        modelos_ml[("Defasagem (SAR)", nome)] = spreg.ML_Lag(y_sp.values, X_sp.values[:, 1:], w=w, name_y="logMEDV", name_x=nomes_x)

linhas_ml = []
for nome, w in W_final.items():
    linhas_ml.append({"Modelo": "MQO", "Matriz": nome, "Parâmetro espacial": np.nan, "z": np.nan,
                      "log-verossimilhança": ols_espacial[nome].logll, "AIC": ols_espacial[nome].aic})
for (tipo, nome), m in modelos_ml.items():
    linhas_ml.append({"Modelo": tipo, "Matriz": nome, "Parâmetro espacial": float(m.betas[-1, 0]),
                      "z": m.z_stat[-1][0], "log-verossimilhança": m.logll, "AIC": m.aic})
print()
pd.DataFrame(linhas_ml).set_index(["Modelo", "Matriz"]).round(3)

spreg.OLS reproduz o modelo_final: True

                     KNN (k = 6)          Distância (d = 5.27 km)         
                     Estatística        p             Estatística        p
LM erro                      165 7.65e-38                      18 2.25e-05
LM defasagem                 112 3.12e-26                    14.1 0.000176
LM erro robusto               81 2.26e-19                    10.8  0.00101
LM defasagem robusto        27.9 1.27e-07                    6.91  0.00858
LM SARMA                     193 1.08e-42                    24.9 3.98e-06


Parâmetro espacial       z  \
Modelo          Matriz                                                
MQO             KNN (k = 6)                             NaN     NaN   
                Distância (d = 5.27 km)                 NaN     NaN   
Erro (SEM)      KNN (k = 6)                           0.770  22.954   
Defasagem (SAR) KNN (k = 6)                           0.357  11.556   
Erro (SEM)      Distância (d = 5.27 km)               0.867  16.805   
Defasagem (SAR) Distância (d = 5.27 km)               0.184   3.573   

                                         log-verossimilhança      AIC  
Modelo          Matriz                                                 
MQO             KNN (k = 6)                          309.014 -580.028  
                Distância (d = 5.27 km)              309.014 -580.028  
Erro (SEM)      KNN (k = 6)                          389.407 -740.814  
Defasagem (SAR) KNN (k = 6)                          363.836 -687.671  
Erro (SEM)      Distância (d = 5.27 km)              324.183 -610.366  
Defasagem (SAR) Distância (d = 5.27 km)              315.214 -590.428

### 11.5 Modelo de erro espacial contra o modelo final

A tabela compara os coeficientes do `modelo_final` (MQO com erros padrão HC3) com os do modelo de erro espacial (SEM) com a matriz KNN, ajustado de duas formas:

- **máxima verossimilhança** (`spreg.ML_Error`), que supõe erros normais e homocedásticos;
- **GMM robusto a heterocedasticidade** (`spreg.GM_Error_Het`, Arraiz et al., 2010; Kelejian e Prucha, 2010). A Parte 9 mostrou que a variância dos erros não é constante, e este estimador não supõe variância constante nem normalidade.

A coluna "(SEM − MQO) / EP HC3" mede a mudança em erros padrão HC3 do MQO, como na Parte 10.

Depois, o I de Moran é recalculado nos resíduos "filtrados" do SEM ($\hat\varepsilon = (I - \hat\lambda W)\hat u$). Se o modelo capturou a dependência, esses resíduos não devem ter correlação espacial. Os resíduos do SAR também são testados.

In [90]:
sem_ml = modelos_ml[("Erro (SEM)", "KNN (k = 6)")]
sar_ml = modelos_ml[("Defasagem (SAR)", "KNN (k = 6)")]
with redirect_stdout(StringIO()):
    sem_het = spreg.GM_Error_Het(y_sp.values, X_sp.values[:, 1:], w=W_knn, name_y="logMEDV", name_x=nomes_x)
print(f"lambda: ML = {sem_ml.betas[-1, 0]:.3f} (EP {sem_ml.std_err[-1]:.3f}) | "
      f"GMM heterocedástico = {sem_het.betas[-1, 0]:.3f} (EP {sem_het.std_err[-1]:.3f})")
print(f"rho (SAR): {sar_ml.rho:.3f} | multiplicador médio 1 / (1 - rho) = {1 / (1 - sar_ml.rho):.2f}")

for nome, residuos in [("SEM (ML), filtrados", sem_ml.e_filtered), ("SEM (GMM het.), filtrados", sem_het.e_filtered),
                       ("SAR (ML)", sar_ml.u)]:
    np.random.seed(42)
    mi = Moran(np.asarray(residuos).ravel(), W_knn, permutations=999)
    print(f"I de Moran nos resíduos {nome}: I = {mi.I:.4f} | p permutação = {mi.p_sim:.3f}")

termos_sp = X_sp.columns
p_het = np.array([z[1] for z in sem_het.z_stat[:-1]])
comparacao_espacial = pd.DataFrame({
    "MQO": modelo_final.params[termos_sp].values,
    "EP HC3": modelo_final.bse[termos_sp].values,
    "p HC3": modelo_final.pvalues[termos_sp].values,
    "SEM (ML)": sem_ml.betas[:-1, 0],
    "EP SEM (ML)": sem_ml.std_err[:-1],
    "SEM (GMM het.)": sem_het.betas[:-1, 0],
    "EP SEM (GMM het.)": sem_het.std_err[:-1],
    "p SEM (GMM het.)": p_het,
}, index=termos_sp)
comparacao_espacial["(SEM - MQO) / EP HC3"] = (comparacao_espacial["SEM (GMM het.)"] - comparacao_espacial["MQO"]) / comparacao_espacial["EP HC3"]
comparacao_espacial["Efeito % MQO"] = 100 * (np.exp(comparacao_espacial["MQO"]) - 1)
comparacao_espacial["Efeito % SEM (GMM het.)"] = 100 * (np.exp(comparacao_espacial["SEM (GMM het.)"]) - 1)
comparacao_espacial.drop(index="Intercept").round(4)

lambda: ML = 0.770 (EP 0.034) | GMM heterocedástico = 0.693 (EP 0.035)
rho (SAR): 0.357 | multiplicador médio 1 / (1 - rho) = 1.55
I de Moran nos resíduos SEM (ML), filtrados: I = -0.0414 | p permutação = 0.064
I de Moran nos resíduos SEM (GMM het.), filtrados: I = -0.0200 | p permutação = 0.251
I de Moran nos resíduos SAR (ML): I = 0.1687 | p permutação = 0.001


,MQO,EP HC3,p HC3,SEM (ML),EP SEM (ML),SEM (GMM het.),EP SEM (GMM het.),p SEM (GMM het.),(SEM - MQO) / EP HC3,Efeito % MQO,Efeito % SEM (GMM het.)
C(CHAS)[T.1],0.0913,0.0227,0.0001,-0.0006,0.0254,0.0217,0.0190,0.2545,-3.0625,9.5558,2.1930
C(RAD)[T.2],0.0728,0.0352,0.0385,0.0480,0.0394,0.0608,0.0368,0.0987,-0.3406,7.5524,6.2712
C(RAD)[T.3],0.1523,0.0333,0.0000,0.1353,0.0382,0.1425,0.0388,0.0002,-0.2943,16.4548,15.3177
C(RAD)[T.4],0.0659,0.0284,0.0204,0.0413,0.0328,0.0514,0.0318,0.1062,-0.5120,6.8144,5.2707
C(RAD)[T.5],0.0955,0.0292,0.0011,0.0811,0.0323,0.0893,0.0354,0.0116,-0.2126,10.0170,9.3374
C(RAD)[T.6],0.0489,0.0351,0.1641,-0.0187,0.0409,0.0070,0.0385,0.8561,-1.1927,5.0102,0.7007
C(RAD)[T.7],0.1254,0.0347,0.0003,0.1152,0.0452,0.1116,0.0376,0.0030,-0.3971,13.3614,11.8118
C(RAD)[T.8],0.0828,0.0345,0.0163,0.0123,0.0491,0.0403,0.0456,0.3763,-1.2322,8.6309,4.1136
C(RAD)[T.24],0.1639,0.0425,0.0001,0.1768,0.0492,0.1483,0.0577,0.0101,-0.3682,17.8106,15.9828
CRIM,-0.0167,0.0032,0.0000,-0.0099,0.0016,-0.0117,0.0023,0.0000,1.5987,-1.6598,-1.1621


### 11.6 Erros padrão agrupados pela cidade verdadeira

A Seção 8.4 agrupou os erros padrão por uma cidade aproximada (`cidade`, combinação de `TAX`, `PTRATIO`, `INDUS` e `ZN`). Com `TOWN`, os grupos passam a ser as cidades verdadeiras. O índice de Rand ajustado (`sklearn.metrics.adjusted_rand_score`) mede a concordância entre os dois agrupamentos: 1 para agrupamentos iguais e perto de 0 para agrupamentos sem relação.

A tabela repete a da célula 122 com a coluna de `TOWN`: p-valores do teste de Wald de cada termo (`wald_test_terms`) e a razão entre os erros padrão agrupados por `TOWN` e os erros padrão HC3.

In [91]:
town_final = boston_c.loc[df_final.index, "TOWN"]
print(f"Cidades (TOWN) em df_final: {town_final.nunique()} | mediana de {town_final.value_counts().median():.0f} linhas por cidade"
      f" | maior: {town_final.value_counts().idxmax()} ({town_final.value_counts().max()} linhas)")
print(f"Grupos aproximados (cidade): {cidade.nunique()}")
print(f"Índice de Rand ajustado entre TOWN e cidade: {adjusted_rand_score(town_final, cidade):.3f}")
pureza = pd.crosstab(cidade, town_final).max(axis=1).sum() / len(cidade)
print(f"Linhas cujo grupo aproximado tem a cidade mais frequente igual à sua: {pureza:.1%}\n")

modelo_final_town = smf.ols(formula_final, data=df_final).fit(cov_type="cluster",
                                                              cov_kwds={"groups": pd.factorize(town_final)[0]})
comparacao_town = pd.DataFrame({
    "p HC3": modelo_final.wald_test_terms(scalar=True).table["pvalue"],
    "p agrupado (cidade)": modelo_final_cluster.wald_test_terms(scalar=True).table["pvalue"],
    "p agrupado (TOWN)": modelo_final_town.wald_test_terms(scalar=True).table["pvalue"],
}).drop(index="Intercept")
comparacao_town["Significativo (TOWN)"] = np.where(comparacao_town["p agrupado (TOWN)"] < 0.05, "sim", "não")
razao_town = (modelo_final_town.bse / modelo_final.bse).rename(index={"C(CHAS)[T.1]": "C(CHAS)"})
comparacao_town["Razão EP (TOWN / HC3)"] = razao_town.reindex(comparacao_town.index)
razao_cidade = (modelo_final_cluster.bse / modelo_final.bse).rename(index={"C(CHAS)[T.1]": "C(CHAS)"})
comparacao_town["Razão EP (cidade / HC3)"] = razao_cidade.reindex(comparacao_town.index)
comparacao_town.round(4)

Cidades (TOWN) em df_final: 91 | mediana de 4 linhas por cidade | maior: Cambridge (28 linhas)
Grupos aproximados (cidade): 78
Índice de Rand ajustado entre TOWN e cidade: 0.417
Linhas cujo grupo aproximado tem a cidade mais frequente igual à sua: 81.2%



,p HC3,p agrupado (cidade),p agrupado (TOWN),Significativo (TOWN),Razão EP (TOWN / HC3),Razão EP (cidade / HC3)
C(CHAS),0.0001,0.0330,0.0194,sim,1.7183,1.8844
C(RAD),0.0000,0.0000,0.0009,sim,NaN,NaN
CRIM,0.0000,0.0000,0.0000,sim,1.1358,0.4356
NOX,0.0000,0.0000,0.0002,sim,1.5469,0.7695
logDIS,0.0000,0.0000,0.0002,sim,1.5228,1.1170
PTRATIO,0.0000,0.0000,0.0000,sim,1.3387,1.2723
B,0.0000,0.0000,0.0039,sim,1.4173,0.5373
logLSTAT,0.0000,0.0000,0.0000,sim,1.2610,2.2549
RM_c,0.0000,0.0000,0.0000,sim,1.0717,1.5781
I(RM_c ** 2),0.0000,0.0000,0.0000,sim,1.1145,1.1520


### Conclusão - Correlação espacial

**Ligação.** As 501 linhas de `df_t` e as 473 de `df_final` coincidem com a fonte em todas as variáveis, exceto `CRIM` na linha 500 (0,178 em `housing.csv`, 0,224 na fonte). É uma divergência do arquivo do projeto, não da ligação, e o efeito numa só linha é desprezível. `CMEDV` corrige `MEDV` em 8 linhas (6 em `df_final`). A análise continua com `MEDV`, para que os números sejam comparáveis com as partes anteriores.

**Há correlação espacial nos resíduos.** No mapa, os resíduos formam manchas. O I de Moran do `modelo_final` é 0,32 com a matriz KNN (z = 13,0; p por permutação = 0,001, o menor possível) e 0,38 no `modelo_reduzido`. Com a matriz de distância, I = 0,046 (p = 0,002): o valor é menor porque cada setor tem em média muito mais vizinhos (mediana de 88), e a média de muitos resíduos dilui a correlação, mas a conclusão é a mesma. Na conferência com os resíduos embaralhados entre os locais, I = −0,016 (p = 0,31), como esperado sem correlação. A correlação **entre** setores vizinhos, que os erros agrupados não tratam, existe.

**Modelo de erro espacial.** Os quatro testes LM são significativos, mas o LM erro robusto é muito maior que o LM defasagem robusto (81 contra 28 com KNN; 10,8 contra 6,9 com distância). O modelo de erro espacial (SEM) também tem o menor AIC (−741 contra −688 do SAR e −580 do MQO), com $\hat\lambda = 0{,}77$ (ML) e 0,69 (GMM heterocedástico). Nos resíduos filtrados do SEM, a correlação some (I = −0,04, p = 0,064 no ML; I = −0,02, p = 0,25 no GMM). Nos resíduos do SAR, ela continua (I = 0,17, p = 0,001). A dependência está num **fator do lugar que não está no modelo**, e não num efeito direto do preço dos vizinhos.

**O que muda nos coeficientes (SEM por GMM heterocedástico, contra o MQO com HC3).**

- `CHAS` perde o efeito: 0,091 → 0,022 (de +9,6% para +2,2%; 3,1 erros padrão HC3; p = 0,25). No ML, o coeficiente fica em −0,001;
- `logDIS` cai à metade: −0,140 → −0,073 (2,8 erros padrão; p = 0,072), e deixa de ser significativo a 5%;
- `PTRATIO`, `logLSTAT` e `logLSTAT²` ficam menores em valor absoluto (de 1,8 a 2,1 erros padrão), mas continuam significativos;
- `RM_c` e `RM_c²` quase não mudam (menos de 0,9 erro padrão). `CRIM`, `NOX` e `B` continuam significativos;
- `RAD = 2`, `4`, `6` e `8` deixam de ser significativos.

A margem do rio Charles e a distância aos centros de emprego são características do **lugar**. No MQO, parte do efeito atribuído a elas vinha de outros fatores do lugar, que são parecidos entre setores vizinhos. Quando o SEM modela essa semelhança, sobra pouco efeito próprio de `CHAS` e de `logDIS`.

**Erros agrupados por `TOWN`.** Há 91 cidades em `df_final` (mediana de 4 setores por cidade). As cidades aproximadas (`cidade`, 78 grupos) concordam só em parte com as verdadeiras (índice de Rand ajustado 0,42). Ainda assim, 81% das linhas estão num grupo cuja cidade mais frequente é a sua. Com `TOWN` como grupo, todos os termos continuam significativos (o maior p-valor é o de `CHAS`, 0,019), com erros padrão de 1,07 a 1,72 vez os HC3. O agrupamento por cidade, verdadeira ou aproximada, **não basta**: ele trata a correlação dentro da cidade, e o SEM mostra que a correlação entre cidades vizinhas muda a significância de `CHAS` e `logDIS`.

**Consequência para o modelo final.** As conclusões sobre `RM`, `LSTAT`, `CRIM`, `NOX`, `PTRATIO` e `B` resistem à correlação espacial. As conclusões sobre `CHAS` e `DIS` não resistem: elas devem ser apresentadas como associações que dependem da localização, e não como efeitos próprios. O `modelo_final` continua sendo o MQO, para manter a comparação com as partes anteriores. Para inferência sobre `CHAS` e `DIS`, a referência é o SEM com GMM heterocedástico.

## **Parte 12: Pontos que pioram o modelo final**

No gráfico de escala-locação do modelo final (Seção 8.5, célula 124), um ponto isolado no canto esquerdo (valor ajustado ≈ 1,7) parece puxar a curva LOWESS para cima. Esta parte identifica as linhas que mais se afastam do modelo final e verifica se tirar mais linhas melhora o modelo. O modelo final das Partes 8 a 15 não muda: esta é uma análise de sensibilidade.

### 12.1 Identificação

Para cada linha do modelo final (473 linhas, ajuste clássico) são calculados o resíduo studentizado externo, a alavanca $h_{ii}$, a distância de Cook e o DFFITS. A tabela mostra as linhas que aparecem por pelo menos um motivo: as 2 com menor valor ajustado (a ponta esquerda do gráfico), as que têm $|t| > 3$, o outlier pelo teste de Bonferroni e as 5 maiores distâncias de Cook. Os mesmos critérios da Parte 8 (`linhas_influentes`, célula 118) também são aplicados uma 2ª vez, no modelo já sem as influentes.

In [119]:
infl_16 = modelo_final_classico.get_influence()
diag_16 = pd.DataFrame({
    "MEDV": df_final["MEDV"],
    "Ajustado (MEDV)": np.exp(modelo_final_classico.fittedvalues),
    "Ajustado (log)": modelo_final_classico.fittedvalues,
    "t studentizado": infl_16.resid_studentized_external,
    "Alavanca h": infl_16.hat_matrix_diag,
    "Cook": infl_16.cooks_distance[0],
    "DFFITS": infl_16.dffits[0],
    **{v: df_final[v] for v in ["CRIM", "LSTAT", "RM", "NOX", "RAD"]},
})

# Motivos: as 2 linhas com menor valor ajustado (ponta esquerda da escala-locação), |t| > 3 e as 5 maiores distâncias de Cook
bonferroni_16 = modelo_final_classico.outlier_test(method="bonf")["bonf(p)"] < 0.05
motivos = {
    "ponta esquerda": diag_16["Ajustado (log)"].nsmallest(2).index,
    "|t| > 3": diag_16.index[diag_16["t studentizado"].abs() > 3],
    "Bonferroni": diag_16.index[bonferroni_16],
    "Cook (top 5)": diag_16["Cook"].nlargest(5).index,
}
linhas_16 = sorted(set().union(*motivos.values()))
diag_16["Motivo"] = [", ".join(nome for nome, idx in motivos.items() if i in idx) for i in diag_16.index]

segunda_rodada = linhas_influentes(modelo_final_classico)
print(f"Alavanca média: {diag_16['Alavanca h'].mean():.3f} | limite 2p/n: {2 * (modelo_final_classico.df_model + 1) / n_final:.3f}")
print(f"Linhas com RAD = 24 no modelo final: {(df_final['RAD'] == 24).sum()} de {n_final}; "
      f"entre as linhas da tabela: {(diag_16.loc[linhas_16, 'RAD'] == 24).sum()} de {len(linhas_16)}")
print(f"2ª rodada de influentes (mesmos critérios no modelo final): {len(segunda_rodada)} linhas, "
      f"{(df_final.loc[segunda_rodada, 'RAD'] == 24).mean():.0%} com RAD = 24")
diag_16.loc[linhas_16].sort_values("Ajustado (log)").round(3)

Alavanca média: 0.040 | limite 2p/n: 0.080
Linhas com RAD = 24 no modelo final: 112 de 473; entre as linhas da tabela: 8 de 9
2ª rodada de influentes (mesmos critérios no modelo final): 23 linhas, 65% com RAD = 24


,MEDV,Ajustado (MEDV),Ajustado (log),t studentizado,Alavanca h,Cook,DFFITS,CRIM,LSTAT,RM,NOX,RAD,Motivo
414,7.0,5.492,1.703,2.133,0.210,0.063,1.101,45.746,36.98,4.519,0.693,24,"ponta esquerda, Cook (top 5)"
404,8.5,7.236,1.979,1.378,0.173,0.021,0.630,41.529,27.38,5.531,0.693,24,ponta esquerda
427,10.9,9.041,2.202,1.604,0.174,0.028,0.737,37.662,14.52,6.202,0.679,24,Cook (top 5)
413,16.3,10.513,2.353,3.591,0.073,0.052,1.007,28.656,20.08,5.155,0.597,24,"|t| > 3, Cook (top 5)"
385,7.2,10.787,2.378,-3.230,0.032,0.018,-0.585,16.812,30.81,5.277,0.700,24,|t| > 3
397,8.5,15.191,2.721,-4.658,0.016,0.018,-0.601,7.672,19.92,5.747,0.693,24,"|t| > 3, Bonferroni"
391,23.2,15.699,2.754,3.094,0.017,0.009,0.407,5.293,18.76,6.051,0.700,24,|t| > 3
407,27.9,17.482,2.861,3.750,0.032,0.024,0.680,11.951,12.13,5.608,0.659,24,"|t| > 3, Cook (top 5)"
161,50.0,40.723,3.707,1.722,0.136,0.025,0.684,1.463,1.73,7.489,0.605,5,Cook (top 5)


In [120]:

linha_extrema = diag_16["Ajustado (log)"].idxmin()
modelo_sem_extremo = smf.ols(formula_final, data=df_final.drop(index=linha_extrema)).fit()


def inclinacao_escala_locacao(m):
    # Inclinação da reta de √|r_i| contra o valor ajustado (0 = variância constante)
    raiz = np.sqrt(np.abs(m.get_influence().resid_studentized_internal))
    return sm.OLS(raiz, sm.add_constant(m.fittedvalues.values)).fit().params[1]


paineis_232 = [(modelo_final_classico, f"Modelo final (n = {int(modelo_final_classico.nobs)})"),
               (modelo_sem_extremo, f"Sem a linha {linha_extrema} (n = {int(modelo_sem_extremo.nobs)})")]
fig = make_subplots(rows=1, cols=2, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Escala-locação: {titulo}" for _, titulo in paineis_232])
for col, (m, titulo) in enumerate(paineis_232, start=1):
    raiz = pd.Series(np.sqrt(np.abs(m.get_influence().resid_studentized_internal)), index=m.fittedvalues.index)
    medidas = {**medidas_diagnostico(m), "√|res. padronizado|": raiz}
    fig.add_trace(pontos(m.fittedvalues, raiz, raiz.index, medidas), row=1, col=col)
    fig.add_trace(curva_lowess(m.fittedvalues, raiz), row=1, col=col)
    if linha_extrema in raiz.index:
        fig.add_trace(go.Scatter(
            x=[m.fittedvalues[linha_extrema]], y=[raiz[linha_extrema]], mode="markers+text", name=f"Linha {linha_extrema}",
            marker=dict(size=16, color="rgba(0, 0, 0, 0)", line=dict(color=COR_DESTAQUE, width=2.5)),
            text=[f"linha {linha_extrema}"], textposition="top right", textfont=dict(color=COR_DESTAQUE),
            hovertext=texto_hover([linha_extrema], {nome: [pd.Series(v, index=raiz.index)[linha_extrema]]
                                                    for nome, v in medidas.items()}),
            hovertemplate="%{hovertext}<extra></extra>",
        ), row=1, col=col)
    fig.update_xaxes(title_text="Valor ajustado (log MEDV)", row=1, col=col)
fig.update_yaxes(title_text="√|resíduo padronizado|", row=1, col=1)
mostrar_e_salvar(fig, "escala_locacao_sem_extremo", altura=520)

for nome, m in [("Modelo final", modelo_final_classico), (f"Sem a linha {linha_extrema}", modelo_sem_extremo)]:
    print(f"{nome}: inclinação de √|r| x ajustado = {inclinacao_escala_locacao(m):+.3f} | "
          f"Breusch-Pagan p = {het_breuschpagan(m.resid, m.model.exog)[1]:.2g}")

Modelo final: inclinação de √|r| x ajustado = -0.242 | Breusch-Pagan p = 5.9e-12
Sem a linha 414: inclinação de √|r| x ajustado = -0.239 | Breusch-Pagan p = 3.5e-12


### 12.2 Reajuste sem essas linhas

O modelo final é reajustado em três cenários: sem a linha da ponta esquerda, sem as linhas com $|t| > 3$ e sem a 2ª rodada de influentes. Para cada um, a tabela mostra os resíduos (`resumir_residuos`), a maior mudança de um coeficiente em erros padrão HC3 e a validação cruzada. A validação cruzada usa as mesmas 10 partes da célula 126. Em cada rodada, as influentes e as linhas de cada cenário saem **só do treino**, e o teste mantém todas as linhas. A referência é a linha "Final (sem influentes no treino)" da célula 126 (RMSE 4,072).

In [121]:
cenarios_16 = {
    "Modelo final": [],
    f"Sem a linha {linha_extrema}": [linha_extrema],
    "Sem |t| > 3": list(motivos["|t| > 3"]),
    "Sem a 2ª rodada de influentes": list(segunda_rodada),
}

ep_hc3_final = modelo_final.bse
resumo_16 = {}
for nome, fora in cenarios_16.items():
    m = smf.ols(formula_final, data=df_final.drop(index=fora)).fit()
    r = resumir_residuos(m)
    resumo_16[nome] = {
        "Linhas retiradas": len(fora), **{k: r[k] for k in ["n", "R² ajustado", "Erro padrão residual", "Correlação QQ",
                                                          "Shapiro-Wilk p", "|t| > 3", "Breusch-Pagan p", "RESET p"]},
        "Maior |Δβ| / EP HC3": ((m.params - modelo_final.params).abs() / ep_hc3_final).max(),
        "Termo com maior |Δβ|": ((m.params - modelo_final.params).abs() / ep_hc3_final).idxmax() if fora else "n/a",
    }

# Validação cruzada: as mesmas 10 partes da célula 126. Tudo é retirado só do treino; o teste mantém todas as linhas.
regras_16 = {
    "1 rodada (célula 126)": lambda m, d: [],
    f"1 rodada + linha {linha_extrema}": lambda m, d: [i for i in [linha_extrema] if i in d.index],
    "1 rodada + |t| > 3": lambda m, d: list(d.index[np.abs(m.get_influence().resid_studentized_external) > 3]),
    "2 rodadas": lambda m, d: list(linhas_influentes(m)),
}
rodadas_16 = {nome: [] for nome in regras_16}
for treino, teste in KFold(n_splits=10, shuffle=True, random_state=42).split(df_t):
    dados_treino, dados_teste = df_t.iloc[treino], df_t.iloc[teste]
    m_todas = smf.ols(formula_final, data=dados_treino).fit()
    treino_1 = dados_treino.drop(index=linhas_influentes(m_todas))
    m_1 = smf.ols(formula_final, data=treino_1).fit()
    for nome, regra in regras_16.items():
        m_cv = smf.ols(formula_final, data=treino_1.drop(index=regra(m_1, treino_1))).fit()
        erro = dados_teste["MEDV"] - prever_log(m_cv, dados_teste)
        rodadas_16[nome].append({"RMSE": np.sqrt(np.mean(erro ** 2)), "MAE": np.mean(np.abs(erro))})

for (nome, linha), regra in zip(resumo_16.items(), regras_16):
    linha["RMSE (VC)"] = pd.DataFrame(rodadas_16[regra])["RMSE"].mean()
    linha["MAE (VC)"] = pd.DataFrame(rodadas_16[regra])["MAE"].mean()

tabela_16 = pd.DataFrame(resumo_16)
tabela_16.to_csv("../results/tables/sensibilidade_pontos_modelo_final.csv")
tabela_16.map(lambda v: f"{v:.4g}" if isinstance(v, float) else v)

,Modelo final,Sem a linha 414,Sem |t| > 3,Sem a 2ª rodada de influentes
Linhas retiradas,0,1,5,23
n,473,472,468,450
R² ajustado,0.8809,0.8798,0.8947,0.9029
Erro padrão residual,0.1285,0.128,0.1195,0.1089
Correlação QQ,0.9933,0.9923,0.9982,0.9982
Shapiro-Wilk p,0.001145,0.0003298,0.2109,0.4757
|t| > 3,5,5,0,3
Breusch-Pagan p,5.861e-12,3.475e-12,4.44e-13,4.363e-10
RESET p,0.9365,0.5884,0.3024,0.9968
Maior |Δβ| / EP HC3,0,0.4915,0.7262,1.749


### Conclusão - Pontos que pioram o modelo final

**O ponto da ponta esquerda é a linha 414.** `MEDV` = 7,0, a 2ª maior taxa de criminalidade da base (`CRIM` = 45,7), 4,5 cômodos e `LSTAT` = 37%. É a linha com a maior alavanca (h = 0,21, cinco vezes a média e acima do limite 2p/n = 0,08) e a maior distância de Cook (0,063, longe de 1). O modelo prevê 5,5 mil dólares e o valor real é 7,0, então o erro não é grande. A linha 414 aparece isolada porque suas variáveis são extremas, não porque o modelo erra muito.

**A linha 414 não causa a subida da curva.** Sem ela, a curva LOWESS da escala-locação continua alta na esquerda (de cerca de 0,7 para 1,3). A inclinação de $\sqrt{|r_i|}$ contra o valor ajustado fica igual (−0,24 nos dois casos), e o Breusch-Pagan continua rejeitando. A subida mostra que **as casas mais baratas têm resíduos maiores**. Essa heterocedasticidade já foi tratada na Parte 9 (MQGF e cobertura dos intervalos de previsão).

**As outras linhas.** 8 das 9 linhas da tabela têm `RAD = 24` (a cidade de Boston, linhas 380 a 415). A única exceção é a 161, com `MEDV = 50` (censura). A 2ª rodada de `linhas_influentes` marcaria mais 23 linhas (65% com `RAD = 24`). Isso é esperado: o corte 4/n fica mais baixo a cada rodada, e sempre aparecem novas linhas acima dele.

**Retirar mais linhas não melhora a previsão.**

- **Sem a linha 414:** quase nada muda. O RMSE da validação cruzada vai de 4,072 para 4,062, e o maior coeficiente muda 0,5 EP (`CRIM`).
- **Sem as 5 linhas com |t| > 3:** os resíduos ficam normais (Shapiro-Wilk p = 0,21, correlação QQ 0,998), mas o RMSE fica um pouco pior (4,090).
- **Sem a 2ª rodada (23 linhas):** o R² ajustado sobe para 0,903 e o Shapiro-Wilk passa, mas o RMSE piora para 4,180, e o coeficiente de `CRIM` muda 1,7 EP.

O ajuste melhora dentro da amostra porque as linhas difíceis saem, mas a previsão de casas novas não melhora.

**Decisão.** O modelo final fica com as 473 linhas. A linha 414 e as outras são casas reais da região central de Boston, e não erros de registro. Tirá-las melhora só o ajuste dentro da amostra. Elas indicam limitações já conhecidas: a heterocedasticidade nas casas baratas (Parte 9) e a dependência espacial dentro de Boston (Parte 11).